In [ ]:
# =============================================================================
# BLOQUE 0: FLAGS MAESTRAS, DISPOSITIVO, HILOS Y SEMILLAS  (PRIMERA CELDA)
# Debe ejecutarse antes de cualquier otro import/uso de PyTorch/NumPy.
# =============================================================================
import os

# ----- 1) FLAGS MAESTRAS Y CONSTANTES DEL EXPERIMENTO (única fuente de verdad) -----
GLOBAL_SEED: int = 42

SMOKE_TEST: bool = True       # False para ejecución real
REAL_MINI:  bool = False       # True recorta el dataset a ~24 meses
USE_GPU:    bool = True        # Intentar usar GPU
OFFLINE:    bool = False       # Permite red (FRED/Finnhub) si False
ALLOW_GKX:  bool = True       # Desactivar NNs si no las usarás

# --- Parámetros de Cross-Validation y Cadencia ---
TARGET_HORIZON = 3                 # Horizonte de predicción en meses (3M)
STEP_M         = 3                 # Frecuencia de rebalanceo en meses (Trimestral)
EMBARGO_M      = TARGET_HORIZON    # Embargo en meses para evitar fugas
HOLDOUT_M      = 6 if REAL_MINI else 24  # Meses para el holdout final
TRAIN_MIN      = 36                # Meses mínimos de entrenamiento en cada fold
GAP_STEPS      = max(1, EMBARGO_M // STEP_M)  # Embargo en PASOS (trimestres)
OUTER_SPLITS_LIMIT = 40 if not SMOKE_TEST else 2  # Límite de folds para debug

# --- Parámetros de Tuning ---
DO_RETUNE          = True
RETUNE_EVERY_FOLDS = 16             # Re-optimizar hiperparámetros cada N folds
N_ITER_RS          = 20            # Iteraciones para RandomizedSearchCV
CV_SPLITS_INNER    = 3             # Folds para la validación cruzada interna (tuning)

# --- Parámetros del Ensemble ---
import numpy as np
DO_MONTHLY_ENSEMBLE = True
ENSEMBLE_LAGS_SAFE  = [3, 6, 9]    # Lags para el ensemble (>= EMBARGO_M)
ENSEMBLE_WEIGHTS    = np.array([0.5, 0.3, 0.2])
MIN_TRAIN_MONTHS_ENSEMBLE = 36

# --- Parámetros de Estrategia L/S ---
PORTFOLIO_K = 20                   # Nº de activos en la pata larga (y en la corta)
TRANSACTION_COSTS_BPS = 5          # Costo de transacción en bps (0.05%) por lado


# ----- 2) Variables de entorno ANTES de importar librerías científicas -----
# 2.1) Control de hilos de BLAS/numexpr
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

# 2.2) Visibilidad de GPU para Torch/TF (decidido por USE_GPU)
if USE_GPU:
    os.environ.pop("CUDA_VISIBLE_DEVICES", None)  # GPU visible
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = ""       # Fuerza CPU

# (Opcional) Inter-op por env si tu build lo respalda
os.environ.setdefault("TORCH_NUM_INTEROP_THREADS", "1")

# Semillas reproducibles a nivel hash de Python
os.environ.setdefault("PYTHONHASHSEED", "42")

# ----- 3) Imports (después de fijar entorno) -----
import random
import numpy as np
import torch

# ----- 4) Dispositivo final -----
HAS_GPU = USE_GPU and torch.cuda.is_available()
DEVICE  = torch.device("cuda" if HAS_GPU else "cpu")

# Flags maestras
ALLOW_GKX = True            # ← habilita NNs
USE_GPU   = globals().get("USE_GPU", True)

# Detección simple de GPU
try:
    import torch
    HAS_GPU = bool(torch.cuda.is_available())
except Exception:
    HAS_GPU = False

globals()["HAS_GPU"] = HAS_GPU  # asegurar visibilidad global


# ----- 5) Semillas reproducibles -----
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if HAS_GPU:
    torch.cuda.manual_seed_all(GLOBAL_SEED)

# (Opcional) determinismo en PyTorch
try:
    torch.use_deterministic_algorithms(False if SMOKE_TEST else True)
except Exception:
    pass

# ----- 6) Límite de hilos de PyTorch (intra-op e inter-op) -----
torch.set_num_threads(1)  # intra-op
try:
    # inter-op debe fijarse antes de que PyTorch inicie trabajo paralelo
    torch.set_num_interop_threads(1)
except RuntimeError as e:
    print(f"[WARN] set_num_interop_threads no aplicado: {e}")

# ----- 7) Toggles para datos externos (FRED / Finnhub) -----
ENABLE_FRED     = (not OFFLINE) and (os.getenv("NO_FRED", "0") != "1")
FRED_API_KEY    = os.getenv("FRED_API_KEY", None)

ENABLE_FINNHUB  = (not OFFLINE) and (os.getenv("NO_FINNHUB", "0") != "1")
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", None)

# ----- 8) Resumen de configuración -----
print("="*72)
print("CONFIGURACION GLOBAL INICIALIZADA")
print("="*72)
print(f"SMOKE_TEST={SMOKE_TEST} | REAL_MINI={REAL_MINI} | ALLOW_GKX={ALLOW_GKX}")
print(f"OFFLINE={OFFLINE} | ENABLE_FRED={ENABLE_FRED} | ENABLE_FINNHUB={ENABLE_FINNHUB}")
print(f"FRED_API_KEY={'SET' if FRED_API_KEY else 'None'} | FINNHUB_API_KEY={'SET' if FINNHUB_API_KEY else 'None'}")
print(f"USE_GPU={USE_GPU} | HAS_GPU={HAS_GPU} | DEVICE={DEVICE}")
print(f"Semilla={GLOBAL_SEED}")
print(f"intra-op={torch.get_num_threads()}")
try:
    print(f"inter-op={torch.get_num_interop_threads()}")
except Exception:
    pass
print("="*72)


CONFIGURACION GLOBAL INICIALIZADA
SMOKE_TEST=True | REAL_MINI=False | ALLOW_GKX=True
OFFLINE=False | ENABLE_FRED=True | ENABLE_FINNHUB=True
FRED_API_KEY=None | FINNHUB_API_KEY=None
USE_GPU=True | HAS_GPU=False | DEVICE=cpu
Semilla=42
intra-op=1
inter-op=1


In [ ]:
# =============================================================================
# BLOQUE 1: BOOTSTRAP DE DEPENDENCIAS (SOLO PARA NOTEBOOKS)
# =============================================================================
try:
    # Esta línea solo funciona en un entorno de iPython/Jupyter
    get_ipython()
    print("\n🚀 Iniciando Bootstrap para el entorno de Cloud/Notebook...")
    %pip install -q --upgrade pip
    %pip install -q "scikit-learn>=1.3.2" lightgbm==4.1.0 xgboost==1.7.6 catboost pandas-datareader shap openpyxl python-dotenv psutil
    %pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    print("✅ Bootstrap completado.")
except NameError:
    print("ℹ️  Entorno de script detectado. Se asume que las dependencias ya están instaladas.")


🚀 Iniciando Bootstrap para el entorno de Cloud/Notebook...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 14.7 MB/s eta 0:00:00
✅ Bootstrap completado.


In [ ]:
# =============================================================================
# BLOQUE 2: IMPORTS PRINCIPALES Y FUNCIONES DE UTILIDAD
# =============================================================================
# --- Librerías Estándar ---
import time
import warnings
from dateutil.relativedelta import relativedelta
import gc
from contextlib import contextmanager
import json
import platform
import sys

# --- Análisis y Datos ---
import numpy as np
import pandas as pd
import pandas_datareader.data as web
import requests
from scipy.stats import spearmanr
from scipy.stats.mstats import winsorize

# --- Visualización ---
import matplotlib.pyplot as plt

# --- Machine Learning ---
from catboost import CatBoostRegressor, CatBoostRanker, Pool
from lightgbm import LGBMRegressor, LGBMRanker
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import ElasticNetCV, LinearRegression
from xgboost import XGBRegressor, XGBRanker
from sklearn.base import clone
from sklearn.base import clone as _sk_clone
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import RandomizedSearchCV, HalvingRandomSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer

# --- Helpers de Entorno ---
try:
    from IPython.display import display as _display
    def safe_display(obj): _display(obj)
except ImportError:
    def safe_display(obj): print(obj.to_string() if hasattr(obj, "to_string") else obj)

try:
    import psutil
    def mem_mb(): return psutil.Process(os.getpid()).memory_info().rss / 1024**2
except ImportError:
    def mem_mb(): return -1.0

# --- Configuración de Pandas y Warnings ---
warnings.filterwarnings('ignore')
pd.options.mode.chained_assignment = None
print("✅ Imports principales y funciones de utilidad cargados.")

✅ Imports principales y funciones de utilidad cargados.


In [ ]:
# ============================================================
# 1) Scorer Spearman robusto (defínelo ANTES de usarlo)
# ============================================================
import numpy as np
from sklearn.metrics import make_scorer

def _spearman_safe(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    # Máscara de finitos
    m = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[m]
    y_pred = y_pred[m]

    # Casos degenerados
    if y_true.size < 2:
        return 0.0
    if np.all(y_true == y_true[0]) or np.all(y_pred == y_pred[0]):
        return 0.0

    # Rankeos (sin depender de scipy); intenta pandas y cae a un fallback numpy
    try:
        import pandas as pd
        r_true = pd.Series(y_true).rank(method="average").to_numpy()
        r_pred = pd.Series(y_pred).rank(method="average").to_numpy()
    except Exception:
        # Fallback simple de ranks 1..n
        order = y_true.argsort()
        r_true = np.empty_like(order, dtype=float); r_true[order] = np.arange(1, y_true.size+1)
        order = y_pred.argsort()
        r_pred = np.empty_like(order, dtype=float); r_pred[order] = np.arange(1, y_pred.size+1)

    # Pearson sobre ranks
    r_true = (r_true - r_true.mean()) / (r_true.std(ddof=0) + 1e-12)
    r_pred = (r_pred - r_pred.mean()) / (r_pred.std(ddof=0) + 1e-12)
    if not np.isfinite(r_true).all() or not np.isfinite(r_pred).all():
        return 0.0

    return float((r_true * r_pred).mean())

SPEARMAN_SAFE = make_scorer(_spearman_safe, greater_is_better=True)


# ============================================================
# 2) build_halving_lgbm SIN referenciar nombres no definidos
#    en la firma. Resolver dentro.
# ============================================================
# Asegúrate de tener estos imports (y el enable):
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingRandomSearchCV

def build_halving_lgbm(
    scorer=None,
    seed: int = 42,
    # puedes agregar otros kwargs si los usas (cv, distribuciones, etc.)
    **kwargs
):
    """
    Devuelve un HalvingRandomSearchCV para LGBM en CPU (o tu wrapper).
    'scorer' se resuelve aquí adentro para evitar NameError en la firma.
    """
    # 2.1) Resolver scorer adentro
    if scorer is None:
        # toma SPEARMAN_SAFE si existe; si no, usa un fallback r2
        scorer = globals().get("SPEARMAN_SAFE", None)
        if scorer is None:
            from sklearn.metrics import r2_score
            scorer = make_scorer(r2_score)

    # 2.2) Asegura que tienes un constructor del estimador base
    # Debe existir en tu entorno algo como make_lgbm_cpu(seed=seed)
    if "make_lgbm_cpu" not in globals():
        # fallback mínimo (ajústalo a tu factory real)
        from lightgbm import LGBMRegressor
        def make_lgbm_cpu(seed=42):
            return LGBMRegressor(
                random_state=seed,
                device_type="cpu",
                n_estimators=200
            )

    # 2.3) Param distributions por defecto si no pasas ninguna
    param_dist = kwargs.pop("param_distributions", {
        "model__learning_rate": [0.01, 0.03, 0.05],
        "model__num_leaves": [31, 63, 127],
        "model__min_child_samples": [20, 35, 50],
        "model__feature_fraction": [0.70, 0.85, 0.95],
        "model__bagging_fraction": [0.70, 0.85, 0.95],
        "model__lambda_l2": [0.0, 1.0, 5.0, 10.0],
    })

    # 2.4) CV interno por defecto si no pasas uno
    inner_cv = kwargs.pop("cv", 3)

    # 2.5) Recursos (halving) por defecto si no pasas otros
    factor = kwargs.pop("factor", 3)
    resource = kwargs.pop("resource", "model__n_estimators")
    max_resources = kwargs.pop("max_resources", 600)  # ajusta a tu tope
    min_resources = kwargs.pop("min_resources", "exhaust")  # o un entero pequeño

    return HalvingRandomSearchCV(
        estimator=make_lgbm_cpu(seed=seed),
        param_distributions=param_dist,
        scoring=scorer,
        cv=inner_cv,
        factor=factor,
        resource=resource,
        max_resources=max_resources,
        min_resources=min_resources,
        random_state=seed,
        n_jobs=1,
        verbose=0,
    )


In [ ]:
# ======================== FACTORIES (modelos) ========================
import numpy as np
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

def make_lgbm_cpu(seed=42):
    from lightgbm import LGBMRegressor
    return LGBMRegressor(
        objective="regression",
        n_estimators=1200,
        learning_rate=0.05,
        num_leaves=63,
        feature_fraction=0.85,
        bagging_fraction=0.85,
        reg_lambda=1.0,
        min_data_in_leaf=20,
        min_gain_to_split=0.0,
        max_bin=511,
        feature_pre_filter=False,
        bagging_freq=1,
        n_jobs=1,
        random_state=seed,
        verbose=-1
    )

def make_xgb_cpu(seed=42):
    return XGBRegressor(
        objective="reg:squarederror",
        tree_method="hist",
        predictor="cpu_predictor",
        max_depth=6,
        min_child_weight=10,
        subsample=0.8,
        colsample_bytree=0.8,
        learning_rate=0.05,
        n_estimators=1200,
        reg_lambda=1.0,
        n_jobs=1,            # estabilidad en search
        random_state=seed,
        eval_metric="rmse"   # ← evita NaN en tuner
    )

def make_rf(seed=42) -> RandomForestRegressor:
    return RandomForestRegressor(
        n_estimators=600,
        max_depth=None,
        min_samples_leaf=5,
        max_features=0.6,
        bootstrap=True,
        n_jobs=1,
        random_state=seed
    )

# Sanitizador universal para evitar NaN/Inf en CV
def sanitize_xy(X, y):
    import pandas as pd
    if hasattr(X, "assign"):  # pandas DataFrame
        df = X.assign(_y_=y)
        df = df.replace([np.inf, -np.inf], np.nan).dropna(axis=0)
        return df.drop(columns="_y_"), df["_y_"].to_numpy()
    X = np.asarray(X); y = np.asarray(y)
    mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
    return X[mask], y[mask]


In [ ]:
# ======================== SCORERS ========================
import numpy as np
from scipy.stats import spearmanr
from sklearn.metrics import make_scorer

def _spearman_flat(y_true, y_pred):
    r, _ = spearmanr(y_true, y_pred)
    return 0.0 if np.isnan(r) else float(r)

def _spearman_grouped(y_true, y_pred, groups):
    out = []
    for g in np.unique(groups):
        idx = (groups == g)
        if idx.sum() < 3:
            continue
        yt, yp = y_true[idx], y_pred[idx]
        if np.all(yt == yt[0]):
            continue
        r, _ = spearmanr(yt, yp)
        if not np.isnan(r):
            out.append(float(r))
    return float(np.mean(out)) if out else 0.0

def make_spearman_scorer(group_col=None):
    def _score(est, X, y):
        y_pred = est.predict(X)
        y = np.asarray(y); y_pred = np.asarray(y_pred)
        if group_col is None:
            return _spearman_flat(y, y_pred)
        groups = X[group_col].to_numpy() if hasattr(X, "__getitem__") else X[:, group_col]
        return _spearman_grouped(y, y_pred, np.asarray(groups))
    return make_scorer(_score, greater_is_better=True)


In [ ]:
# ===============================================================
# PARCHE GLOBAL DE TAGS (punto único, no en los mixins)
# ===============================================================
from types import SimpleNamespace
import sklearn.utils._tags as _tags_mod

print("Aplicando parche global de tags (punto único)...")

_orig_get_tags = _tags_mod.get_tags

def get_tags_safe(est):
    tags = _orig_get_tags(est)

    # ya viene bien
    if hasattr(tags, "requires_fit"):
        return tags

    # viene como dict / _TagsDict
    if hasattr(tags, "items"):
        return SimpleNamespace(**dict(tags))

    # último recurso
    return SimpleNamespace(
        requires_fit=True,
        estimator_type=getattr(est, "_estimator_type", "regressor"),
    )

_tags_mod.get_tags = get_tags_safe
print("✓ get_tags() parcheado.")
# ===============================================================
# Parches específicos (xgboost / lightgbm) si querés mantenerlos
# ===============================================================
try:
    from xgboost.sklearn import XGBModel
    def _xgb_safe_tags(self):
        return SimpleNamespace(
            requires_fit=True,
            estimator_type="regressor",
        )
    XGBModel.__sklearn_tags__ = _xgb_safe_tags
    print("✓ XGBModel parcheado.")
except Exception:
    pass


Aplicando parche global de tags (punto único)...
✓ get_tags() parcheado.
✓ XGBModel parcheado.


In [ ]:
# =============================================================================
# PARCHE OFFLINE / FRED / FINNHUB  (colocar después de imports y flags)
# =============================================================================
import os
import datetime as dt


# 1) API keys desde variables de entorno
FINNHUB_API_KEY = os.environ.get("FINNHUB_API_KEY", "")
FRED_API_KEY    = os.environ.get("FRED_API_KEY", "")

def safe_get_finnhub_symbols(exchange="US", cache_path=None):
    """
    Devuelve DataFrame de símbolos o None. Si OFFLINE, intenta cargar cache_path (.pkl).
    """
    import pandas as pd
    if OFFLINE:
        print("[OFFLINE] Saltando descarga Finnhub")
        if cache_path and os.path.exists(cache_path):
            print(f"[OFFLINE] Cargando cache: {cache_path}")
            return pd.read_pickle(cache_path)
        return None
    try:
        import finnhub
        if not FINNHUB_API_KEY:
            raise RuntimeError("Falta FINNHUB_API_KEY")
        fh = finnhub.Client(api_key=FINNHUB_API_KEY)
        data = fh.stock_symbols(exchange)
        df = pd.DataFrame(data)
        if cache_path:
            os.makedirs(os.path.dirname(cache_path), exist_ok=True)
            df.to_pickle(cache_path)
        return df
    except Exception as e:
        print(f"[WARN] Finnhub falló: {e} → symbols_df=None")
        # intenta devolver cache si existe
        if cache_path and os.path.exists(cache_path):
            return pd.read_pickle(cache_path)
        return None

# (1) leer .env
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

# (2) construir ENABLE_FRED / ENABLE_FINNHUB según OFFLINE y key
FINNHUB_API_KEY = os.getenv("FINNHUB_API_KEY", "").strip()
FRED_API_KEY    = os.getenv("FRED_API_KEY", "").strip()

ENABLE_FINNHUB  = (not OFFLINE) and bool(FINNHUB_API_KEY)
ENABLE_FRED     = (not OFFLINE) and bool(FRED_API_KEY)



In [ ]:
# ====== CONTROL DE RETUNEO POCO FRECUENTE (solo DEBUG si se habilita) ======
USE_DEBUG = False  # <-- pon True SOLO para diagnóstico rápido

if USE_DEBUG:
    # Retuneo agresivo para diagnóstico (rápido, no productivo)
    RETUNE_EVERY_FOLDS = 1
    DO_RETUNE           = True
    N_ITER_RS           = 10
    CV_SPLITS_INNER     = 3
    SEARCH_N_JOBS       = 1
    PRINT_TIMING        = True
    WARN_FOLD_MINUTES   = 20
else:
    # Respeta las definiciones de Bloque 0; solo asigna si no existen:
    DO_RETUNE           = bool(globals().get("DO_RETUNE", True))
    RETUNE_EVERY_FOLDS  = int(globals().get("RETUNE_EVERY_FOLDS", 4))
    N_ITER_RS           = int(globals().get("N_ITER_RS", 60))
    CV_SPLITS_INNER     = int(globals().get("CV_SPLITS_INNER", 5))
    SEARCH_N_JOBS       = int(globals().get("SEARCH_N_JOBS", 1))
    PRINT_TIMING        = bool(globals().get("PRINT_TIMING", True))
    WARN_FOLD_MINUTES   = int(globals().get("WARN_FOLD_MINUTES", 35))

In [ ]:
# Defaults seguros si no existen
FAST_MODE = True if SMOKE_TEST else globals().get("FAST_MODE", False)
GPU       = globals().get("HAS_GPU", False)
N_JOBS    = 1
HOLD      = 1
TRAIN_MIN = globals().get("TRAIN_MIN", 36)  # meses de entrenamiento mínimo
OUTER_SPLITS_LIMIT = 2 if SMOKE_TEST else None


In [ ]:
# ===============================================================================
# BLOQUE 3· Reproducibilidad total (semillas, env, logging, metadatos)
# ===============================================================================

import os, sys, json, random, platform, subprocess, time
import numpy as np

def set_global_determinism(seed: int = 42, enable_tf: bool = True):
    import os, random, numpy as np
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    os.environ["TF_CUDNN_DETERMINISTIC"] = "1"
    os.environ["OMP_NUM_THREADS"] = "1"
    os.environ["OPENBLAS_NUM_THREADS"] = "1"
    os.environ["MKL_NUM_THREADS"] = "1"
    os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
    os.environ["NUMEXPR_NUM_THREADS"] = "1"
    random.seed(seed); np.random.seed(seed)
    if enable_tf and 'tf' in globals() and globals()['tf'] is not None:
        tf.random.set_seed(seed)
        tf.config.threading.set_intra_op_parallelism_threads(1)
        tf.config.threading.set_inter_op_parallelism_threads(1)
    print(f"[OK] Semillas y entorno fijados con seed={seed}")

# 1) fijar env ANTES de importar TF
set_global_determinism(GLOBAL_SEED, enable_tf=False)

# 2) importar TF/Keras de forma segura
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except Exception:
    tf = None; keras = None; layers = None
    print("[WARN] TensorFlow/Keras no disponibles.")

# 3) Completar determinismo TF y GPU
set_global_determinism(GLOBAL_SEED, enable_tf=True)
if tf is not None:
    try:
        gpus = tf.config.list_physical_devices('GPU')
        if gpus:
            for g in gpus: tf.config.experimental.set_memory_growth(g, True)
            print(f"[TF] GPU detectada ({len(gpus)}), memory_growth=True")
        else:
            print("[TF] Sin GPU; correrá en CPU.")
    except Exception as e:
        print(f"[TF] No pude configurar memory_growth: {e}")

def lib_versions():
    """Devuelve versiones útiles para reproducibilidad."""
    vers = {"python": sys.version.split()[0], "platform": platform.platform()}
    def _get_ver(mod, name):
        try:
            import importlib
            m = importlib.import_module(mod)
            vers[name] = getattr(m, "__version__", "unknown")
        except Exception:
            vers[name] = "not_installed"
    _get_ver("numpy","numpy"); _get_ver("pandas","pandas"); _get_ver("scipy","scipy")
    _get_ver("sklearn","scikit_learn"); _get_ver("xgboost","xgboost")
    _get_ver("lightgbm","lightgbm"); _get_ver("catboost","catboost")
    _get_ver("tensorflow","tensorflow"); _get_ver("keras","keras")
    return vers

def save_run_metadata(path_json:str,
                      seed:int,
                      feature_cols:list=None,
                      target_col:str=None,
                      extra:dict=None):
    """Guarda metadatos mínimos de la corrida."""
    meta = {
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "seed": seed,
        "versions": lib_versions(),
        "feature_cols": list(feature_cols) if feature_cols is not None else None,
        "target_col": target_col,
    }
    if extra:
        meta["extra"] = extra
    os.makedirs(os.path.dirname(path_json) or ".", exist_ok=True)
    with open(path_json, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2, ensure_ascii=False)
    print(f"[OK] Metadatos guardados en: {path_json}")

# ======= INVOCACIÓN TEMPRANA (llamar una sola vez, bien arriba del pipeline) =======

set_global_determinism(GLOBAL_SEED, enable_tf=True)

# === Holdout / particiones y Cadencia ===
HOLDOUT_M = 24
assert HOLDOUT_M % 3 == 0, f"HOLDOUT_M ({HOLDOUT_M}) debe ser múltiplo de 3 para el rebalanceo trimestral."

TARGET_HORIZON = 3                 # Horizonte de predicción en meses
STEP_M         = 3                 # Frecuencia de rebalanceo en meses
EMBARGO_M      = TARGET_HORIZON    # Embargo en MESES reales (sugerido = horizonte)
GAP_STEPS      = max(1, EMBARGO_M // STEP_M)   # Embargo en PASOS (trimestres)

# --- Guardas de coherencia de cadencia ---
assert STEP_M == 3, "STEP_M debe ser 3 (rebalanceo trimestral)"
assert HOLDOUT_M % STEP_M == 0, "HOLDOUT_M debe ser múltiplo de STEP_M"
assert EMBARGO_M >= 1, "EMBARGO_M debe ser >= 1 (en meses reales)"
assert GAP_STEPS >= 1, "GAP_STEPS debe ser >= 1"
print(f"[CFG] TRAIN_MIN (pasos) se define más abajo; GAP_STEPS={GAP_STEPS}, STEP_M={STEP_M}, HOLDOUT_M={HOLDOUT_M}")

# --- Guardas de configuración del ensemble ---
assert len(ENSEMBLE_LAGS_SAFE) == len(ENSEMBLE_WEIGHTS), "Lags y Weights deben tener la misma longitud."
assert min(ENSEMBLE_LAGS_SAFE) >= EMBARGO_M, "El lag mínimo del ensemble debe ser >= EMBARGO_M para evitar fugas."
if ENSEMBLE_WEIGHTS.sum() > 0:
    ENSEMBLE_WEIGHTS = ENSEMBLE_WEIGHTS / ENSEMBLE_WEIGHTS.sum() # Normalizar pesos

[OK] Semillas y entorno fijados con seed=42
[OK] Semillas y entorno fijados con seed=42
[TF] Sin GPU; correrá en CPU.
[OK] Semillas y entorno fijados con seed=42
[CFG] TRAIN_MIN (pasos) se define más abajo; GAP_STEPS=1, STEP_M=3, HOLDOUT_M=24


In [ ]:
# =================================================
# CELDA DE DIAGNÓSTICO DEL SISTEMA DE ARCHIVOS
# =================================================
import subprocess, shlex, os, pandas as pd

print("--- 1. Directorio de trabajo actual ---")
try:
    print(os.getcwd())
except Exception as e:
    print(f"(no disponible) {e}")

print("\n--- 2. Contenido del directorio ---")
try:
    out = subprocess.run(shlex.split("ls -l"), capture_output=True, text=True)
    print(out.stdout if out.returncode==0 else "<sin permisos>")
except Exception as e:
    print(f"(no disponible) {e}")

print("\n--- 3. Verificación desde Python ---")
cache_file_name = "finnhub_sector_industry.pkl"
full_path = os.path.abspath(cache_file_name)
file_exists = os.path.exists(cache_file_name)
print(f"Nombre de caché: '{cache_file_name}'")
print(f"Ruta completa:    '{full_path}'")

if file_exists:
    try:
        df_test = pd.read_pickle(cache_file_name)
        print(f"✅ Lectura OK. Tickers: {len(df_test)}")
    except Exception as e:
        print(f"❌ Error de lectura: {e}")
else:
    print("❌ No se encontró el archivo de caché en esta ruta.")


--- 1. Directorio de trabajo actual ---
/content

--- 2. Contenido del directorio ---
total 4740
-rw-r--r-- 1 root root 4838574 Nov  9 19:23 dataset_oct_2014-set_2024.xlsx
-rw-r--r-- 1 root root    5805 Nov  9 19:23 finnhub_sector_industry.pkl
drwxr-xr-x 1 root root    4096 Nov  5 14:33 sample_data


--- 3. Verificación desde Python ---
Nombre de caché: 'finnhub_sector_industry.pkl'
Ruta completa:    '/content/finnhub_sector_industry.pkl'
✅ Lectura OK. Tickers: 179


In [ ]:
# ================================================================================
# BLOQUE 4: CARGA, LIMPIEZA INICIAL Y ENRIQUECIMIENTO CON DATOS DE SECTOR
# ================================================================================

# --- Carga de datos robusta para Colab ---
# En Colab, los archivos subidos van a /content/ por defecto.
# No necesitamos crear una subcarpeta 'data'.
file_path = 'dataset_oct_2014-set_2024.xlsx'

print(f"Buscando datos en: {file_path}")

df = None
if os.path.exists(file_path):
    try:
        df = pd.read_excel(file_path, sheet_name='dataset')
        print("✅ Archivo de datos real cargado exitosamente.")
    except Exception as e:
        print(f"⚠️  Error al leer el archivo Excel existente: {e}")
        if not SMOKE_TEST: raise

# --- Fallback a datos sintéticos si el archivo no existe Y estamos en SMOKE_TEST ---
if df is None and 'SMOKE_TEST' in globals() and SMOKE_TEST:
    print("\n" + "="*80)
    print("⚠️  Archivo no encontrado y SMOKE_TEST=True. Generando datos sintéticos...")
    print("="*80)

    def generate_synthetic_data(n_companies=5, n_months=24):
        end_date = pd.Timestamp.now().normalize() - relativedelta(months=1)
        dates = pd.date_range(end=end_date, periods=n_months, freq='M')
        companies = [f'COMP_{i:02d} US Equity' for i in range(1, n_companies + 1)]

        records = []
        for company in companies:
            for date in dates:
                records.append({'Empresa': company, 'Fecha': date})

        synth_df = pd.DataFrame(records)

        # Features numéricas
        numeric_cols = [
            'P_Share', 'ROCE', 'ROA', 'EBIT', 'Total Activos', 'Deuda a LP',
            'Beneficio neto', 'ROI', 'EV', 'Cap de mercado', 'Deuda a CP',
            'Efectivo y equiv', 'P_E', 'P_B', 'P_S',
            'CPI', 'CPI_Exp_mediana', 'Fed Funds Rate',
            'Non farm payrolls', 'Non farm payrolls_Exp_mediana'
        ]
        for col in numeric_cols:
            synth_df[col] = np.random.randn(len(synth_df)) * 100 + np.random.randint(50, 200)

        # Simular formato % para testear la limpieza
        synth_df['CPI'] = (np.random.rand(len(synth_df)) * 5).round(2)
        synth_df['CPI_Exp_mediana'] = synth_df['CPI'] + np.random.randn(len(synth_df)) * 0.5

        return synth_df

    df = generate_synthetic_data()
    print("   (Info) Datos sintéticos generados para el smoke test.")

elif df is None:
    print("\n" + "="*80)
    print(f"❌ ERROR: Archivo no encontrado en '{file_path}' y SMOKE_TEST=False.")
    print("Asegúrate de que la carpeta 'data' y el archivo .xlsx están en su lugar.")
    print("="*80)
    raise FileNotFoundError(f"No se encontró el archivo de datos en {file_path}")

# --- A partir de aquí, el pipeline continúa igual, con 'df' real o sintético ---

# Fechas
df['Fecha'] = pd.to_datetime(df['Fecha'], format='%Y%m%d')

# ---------- Conversión % robusta (CPI/FFR/Expectativas) ----------
def to_decimal_smart(series: pd.Series, verbose: bool = True) -> pd.Series:
    s = series.astype(str).str.strip()
    s = (s.str.replace('%', '', regex=False)
           .str.replace(',', '', regex=False))
    s = pd.to_numeric(s, errors='coerce')
    if s.dropna().empty: return s
    nn = s.dropna()
    frac_gt1 = (nn > 1.0).mean()
    p95 = float(nn.quantile(0.95))
    is_percent = (frac_gt1 >= 0.20) or (p95 > 1.5)
    if verbose:
        fmt = "porcentaje (÷100)" if is_percent else "decimal (sin cambios)"
        print(f"  · Detección formato → {fmt} | frac>1={frac_gt1:.2%} | p95={p95:.3f}")
    return s / 100.0 if is_percent else s

pct_cols = [
    'CPI','CPI_Exp_mediana','CPI_Exp_promedio',
    'Fed Funds Rate','Fed Funds Rate_Exp_mediana','Fed Funds Rate_Exp_promedio'
]
print("\nNormalizando columnas en %/decimales (detección automática)…")
for col in pct_cols:
    if col in df.columns:
        print(f"  - {col}:")
        df[col] = to_decimal_smart(df[col], verbose=True)

# --- NFP: limpieza más robusta (k/K y comas) ---
for col in ['Non farm payrolls','Non farm payrolls_Exp_mediana','Non farm payrolls_Exp_promedio']:
    if col in df.columns:
        s = df[col].astype(str).str.replace(',', '', regex=False).str.strip()
        s = s.str.replace('K', '', regex=False).str.replace('k', '', regex=False)
        df[col] = pd.to_numeric(s, errors='coerce')

# Ordenar datos, es crucial para cálculos temporales
df = df.sort_values(['Empresa', 'Fecha']).reset_index(drop=True)
print("Datos cargados y limpieza inicial completada.")

# --- 4.2: Descarga y Unión de Datos de Sector desde Finnhub ---
# Fix: Definir CACHE_FILE y rate limits para Finnhub (si no definidos)
CACHE_FILE = "finnhub_sector_industry.pkl"  # Cache local
RATE_LIMIT = 60  # Límite llamadas/min
SLEEP_SECONDS = 61  # Pausa post-rate limit

print("\nIniciando descarga de datos de sector desde Finnhub...")

def bloomberg_to_symbol(tk: str) -> str:
    base = str(tk).split()[0].strip().replace('/', '.')
    return base.upper()

symbols_to_fetch = df['Empresa'].map(bloomberg_to_symbol).unique().tolist()

# Paso 1: Intentar cargar desde la caché local primero
sector_df = pd.DataFrame()
if os.path.exists(CACHE_FILE):
    print(f"✅ Caché local '{CACHE_FILE}' encontrada. Cargando datos...")
    try:
        sector_df = pd.read_pickle(CACHE_FILE)
        cached_symbols = set(sector_df['Symbol'])
        print(f"   -> Caché cargada con {len(cached_symbols)} tickers.")
    except Exception as e:
        print(f"   -> ⚠️ Error al leer el archivo de caché: {e}. Se intentará descargar.")
        cached_symbols = set()
else:
    print(f"ℹ️  No se encontró el archivo de caché '{CACHE_FILE}'.")
    cached_symbols = set()

# Paso 2: Determinar si se necesita descargar algo
tickers_to_process = [s for s in symbols_to_fetch if s not in cached_symbols]

if not tickers_to_process:
    print("✅ Todos los tickers necesarios ya están en la caché. No se requiere descarga.")
else:
    print(f"⚠️ Se necesitan datos para {len(tickers_to_process)} tickers nuevos. Se intentará la descarga.")

    # Paso 3: Intentar descargar SOLO SI es necesario y posible
    if OFFLINE:
        print("   -> ❌ Descarga omitida porque OFFLINE=True.")
    elif not FINNHUB_API_KEY:
        print("   -> ❌ Descarga omitida porque FINNHUB_API_KEY no está definida.")
    else:
        print("   -> 🔐 API Key detectada. Iniciando descarga...")
        records_to_add = []
        for i, symbol in enumerate(tickers_to_process, 1):
            # ... (aquí va el bucle de descarga que ya tenías)
            url = f"https://finnhub.io/api/v1/stock/profile2?symbol={symbol}&token={FINNHUB_API_KEY}"
            try:
                response = requests.get(url, timeout=15)
                response.raise_for_status()
                data = response.json()
                if data and data.get('finnhubIndustry'):
                    records_to_add.append({
                        'Symbol':   symbol,
                        'Sector':   data.get('gicSector') or data.get('finnhubIndustry'),
                        'Industry': data.get('finnhubIndustry'),
                    })
            except requests.exceptions.RequestException as e:
                print(f"    ❌ Error de red para {symbol}: {e}")
            if i % RATE_LIMIT == 0 and i < len(tickers_to_process):
                time.sleep(SLEEP_SECONDS)

        if records_to_add:
            new_data_df = pd.DataFrame(records_to_add)
            sector_df = pd.concat([sector_df, new_data_df], ignore_index=True).drop_duplicates('Symbol')
            sector_df.to_pickle(CACHE_FILE)
            print(f"\n   -> {len(records_to_add)} nuevos símbolos añadidos a la caché.")

# Paso 4: Unir los datos al DataFrame principal
if not sector_df.empty:
    df['Symbol_key'] = df['Empresa'].map(bloomberg_to_symbol)
    df = pd.merge(df, sector_df, left_on='Symbol_key', right_on='Symbol', how='left')
    df.drop(columns=['Symbol_key', 'Symbol'], inplace=True, errors='ignore')
    print("\n✅ Datos de Sector e Industria unidos al DataFrame principal.")
else:
    print("\n⚠️ No se pudieron obtener datos de sector (ni de caché ni por descarga).")
    # Crear columnas vacías para que el resto del script no falle
    df['Sector'] = 'Unknown'
    df['Industry'] = 'Unknown'

Buscando datos en: dataset_oct_2014-set_2024.xlsx
✅ Archivo de datos real cargado exitosamente.

Normalizando columnas en %/decimales (detección automática)…
  - CPI:
  · Detección formato → porcentaje (÷100) | frac>1=81.67% | p95=8.205
  - CPI_Exp_mediana:
  · Detección formato → decimal (sin cambios) | frac>1=0.00% | p95=0.081
  - CPI_Exp_promedio:
  · Detección formato → decimal (sin cambios) | frac>1=0.00% | p95=0.081
  - Fed Funds Rate:
  · Detección formato → porcentaje (÷100) | frac>1=50.83% | p95=5.500
  - Fed Funds Rate_Exp_mediana:
  · Detección formato → decimal (sin cambios) | frac>1=0.00% | p95=0.055
  - Fed Funds Rate_Exp_promedio:
  · Detección formato → decimal (sin cambios) | frac>1=0.00% | p95=0.055
Datos cargados y limpieza inicial completada.

Iniciando descarga de datos de sector desde Finnhub...
✅ Caché local 'finnhub_sector_industry.pkl' encontrada. Cargando datos...
   -> Caché cargada con 179 tickers.
⚠️ Se necesitan datos para 1 tickers nuevos. Se intentará la

In [ ]:
# ===============================================================================
# BLOQUE 5 · Normalización de fechas a FIN DE MES (alineación global)
#   - Asegura que todo el pipeline (riesgo, anclaje Q-2, señales, buckets, splits)
#     trabaje en la misma grilla temporal (MonthEnd).
# ===============================================================================

print("\n🗓️  Normalizando fechas a fin de mes…")

# 1) Forzar Fecha a MonthEnd (fin de mes)
df['Fecha'] = pd.to_datetime(df['Fecha'])
df['Fecha'] = df['Fecha'].dt.to_period('M').dt.to_timestamp('M')

# 2) Resolver posibles duplicados (Empresa, Fecha) manteniendo la última fila del mes
dup_count = df.duplicated(subset=['Empresa', 'Fecha'], keep=False).sum()
if dup_count:
    df = (
        df.sort_values(['Empresa', 'Fecha'])
          .groupby(['Empresa', 'Fecha'], as_index=False)
          .tail(1)
          .reset_index(drop=True)
    )
    print(f"  · Se resolvieron {dup_count} duplicados (Empresa, Fecha).")

# 3) Orden final de seguridad
df = df.sort_values(['Empresa', 'Fecha']).reset_index(drop=True)

# 4) Chequeo rápido
assert not df.duplicated(subset=['Empresa', 'Fecha']).any(), "Quedaron duplicados (Empresa, Fecha)."

print("  · Fechas normalizadas a fin de mes y dataset ordenado.")



🗓️  Normalizando fechas a fin de mes…
  · Fechas normalizadas a fin de mes y dataset ordenado.


In [ ]:
# --- 6.1: Conteo de Empresas por Sector ---
sector_counts = (
    df.drop_duplicates(['Empresa', 'Sector'])  # ➊ deja una sola fila por empresa-sector
      .groupby('Sector')['Empresa']            # ➋ agrupa por sector
      .nunique()                               # ➌ cuenta empresas únicas
      .reset_index(name='num_empresas')        # ➍ DataFrame ordenado
      .sort_values('num_empresas', ascending=False)
)

sector_counts

,Sector,num_empresas
31,Technology,18
28,Retail,17
30,Semiconductors,17
12,Electrical Equipment,12
16,Health Care,10
14,Financial Services,10
26,Professional Services,7
33,Trading Companies & Distributors,6
24,Media,6
23,Machinery,6


In [ ]:
# --- 6.2: Conteo de Empresas por Industria ---
industry_counts = (
    df.drop_duplicates(['Empresa', 'Industry'])  # ➊ deja una sola fila por empresa-industria
      .groupby('Industry')['Empresa']            # ➋ agrupa por industria
      .nunique()                               # ➌ cuenta empresas únicas
      .reset_index(name='num_empresas')        # ➍ DataFrame ordenado
      .sort_values('num_empresas', ascending=False)
)

industry_counts

,Industry,num_empresas
31,Technology,18
28,Retail,17
30,Semiconductors,17
12,Electrical Equipment,12
16,Health Care,10
14,Financial Services,10
26,Professional Services,7
33,Trading Companies & Distributors,6
24,Media,6
23,Machinery,6


In [ ]:
# ========================================================================
# BLOQUE 7 · Forzar Industry  →  11 Sectores GICS
# ========================================================================

# --- 1.  Diccionario completo ------------------------------------------
INDUSTRY_TO_GICS = {
    # Information Technology
    'Technology':                      'Information Technology',
    'Semiconductors':                  'Information Technology',

    # Consumer Discretionary
    'Retail':                          'Consumer Discretionary',
    'Consumer products':               'Consumer Discretionary',
    'Hotels, Restaurants & Leisure':   'Consumer Discretionary',
    'Auto Components':                 'Consumer Discretionary',
    'Textiles, Apparel & Luxury Goods':'Consumer Discretionary',
    'Leisure Products':                'Consumer Discretionary',
    'Distributors':                    'Consumer Discretionary',
    'Diversified Consumer Services':   'Consumer Discretionary',

    # Industrials
    'Electrical Equipment':            'Industrials',
    'Machinery':                       'Industrials',
    'Trading Companies & Distributors':'Industrials',
    'Road & Rail':                     'Industrials',
    'Commercial Services & Supplies':  'Industrials',
    'Industrial Conglomerates':        'Industrials',
    'Building':                        'Industrials',
    'Aerospace & Defense':             'Industrials',
    'Logistics & Transportation':      'Industrials',
    'Professional Services':           'Industrials',

    # Financials
    'Financial Services':              'Financials',
    'Insurance':                       'Financials',

    # Health Care
    'Health Care':                     'Health Care',
    'Biotechnology':                   'Health Care',
    'Life Sciences Tools & Services':  'Health Care',

    # Communication Services
    'Media':                           'Communication Services',
    'Communications':                  'Communication Services',

    # Consumer Staples
    'Food Products':                   'Consumer Staples',
    'Beverages':                       'Consumer Staples',

    # Utilities
    'Utilities':                       'Utilities',

    # Materials
    'Chemicals':                       'Materials',
    'Metals & Mining':                 'Materials',
    'Construction':                    'Materials',

    # Energy
    'Energy':                          'Energy',

    # Real Estate
    'Real Estate':                     'Real Estate',
}

# --- 2.  Asignar SIEMPRE a partir de Industry --------------------------
df['Sector_GICS'] = df['Industry'].map(INDUSTRY_TO_GICS).fillna('Unknown')

# --- 3.  Chequeo rápido -------------------------------------------------
print("\nDistribución después del mapeo (deberían ser ≤ 11):")
safe_display(
    df.drop_duplicates(['Empresa','Sector_GICS'])
      .groupby('Sector_GICS')['Empresa']
      .nunique()
      .sort_values(ascending=False)
      .to_frame('empresas')
)


Distribución después del mapeo (deberían ser ≤ 11):


,empresas
Sector_GICS,
Industrials,49
Consumer Discretionary,39
Information Technology,35
Health Care,13
Financials,11
Consumer Staples,10
Communication Services,8
Materials,5
Utilities,4


In [ ]:
# --- 4.  Inspección de casos 'Unknown' -----------------------------------
unknown_company_df = df[df['Sector_GICS'] == 'Unknown']

print("Detalles de la empresa (o empresas) con sector 'Unknown':")
# Mostramos columnas clave para identificarla y ver pistas
if not unknown_company_df.empty:
    safe_display(unknown_company_df[['Empresa', 'Fecha', 'Industry', 'Sector']].drop_duplicates(subset='Empresa'))
else:
    print("No se encontraron empresas con sector 'Unknown'.")

Detalles de la empresa (o empresas) con sector 'Unknown':


,Empresa,Fecha,Industry,Sector
10080,LANC US Equity,2014-10-31,NaN,NaN


In [ ]:
# ========================================================================
# BLOQUE 7.a · Corrección Manual de Sectores GICS
# ========================================================================
# Basado en investigación externa, se identificó que 'LANC US Equity'
# no tenía datos de sector/industria de Finnhub. Se asigna manualmente
# al sector GICS correcto.

print("\n🔧 Aplicando correcciones manuales de sector GICS...")

# Corrección para Lancaster Colony Corporation (LANC)
df.loc[df['Empresa'] == 'LANC US Equity', 'Sector_GICS'] = 'Consumer Staples'

print("  · 'LANC US Equity' asignado a 'Consumer Staples'.")

# --- Verificación post-corrección ---
# Volvemos a correr el conteo para asegurarnos de que 'Unknown' ha desaparecido
print("\nDistribución después de la corrección manual:")
safe_display(
    df.drop_duplicates(['Empresa','Sector_GICS'])
      .groupby('Sector_GICS')['Empresa']
      .nunique()
      .sort_values(ascending=False)
      .to_frame('empresas')
)


🔧 Aplicando correcciones manuales de sector GICS...
  · 'LANC US Equity' asignado a 'Consumer Staples'.

Distribución después de la corrección manual:


,empresas
Sector_GICS,
Industrials,49
Consumer Discretionary,39
Information Technology,35
Health Care,13
Consumer Staples,11
Financials,11
Communication Services,8
Materials,5
Real Estate,4


In [ ]:
# ==============================================================================
# BLOQUE 7.b · Monitoreo de cobertura del mapeo GICS (Industry → Sector_GICS)
# ==============================================================================

print("\n📊 Cobertura del mapeo a GICS")

# Normalización suave para reporte (no altera el mapeo original)
df['_Industry_raw'] = df['Industry'].astype(str).str.strip()

unk_mask = df['Sector_GICS'].eq('Unknown')
unknown_rows_pct = float(unk_mask.mean())
unknown_emps = df.loc[unk_mask, 'Empresa'].nunique()
total_emps   = df['Empresa'].nunique()
unknown_emps_pct = unknown_emps / max(1, total_emps)

print(f"  · Filas con Sector_GICS='Unknown': {unknown_rows_pct:.1%}")
print(f"  · Empresas afectadas: {unknown_emps} / {total_emps} ({unknown_emps_pct:.1%})")

# Top industrias sin mapear (por cantidad de empresas distintas)
top_unknown_industries = (
    df.loc[unk_mask & df['_Industry_raw'].ne('nan')]
      .groupby('_Industry_raw')['Empresa']
      .nunique()
      .sort_values(ascending=False)
      .head(20)
      .rename('num_empresas')
      .reset_index()
      .rename(columns={'_Industry_raw': 'Industry'})
)

if not top_unknown_industries.empty:
    print("\nTop industrias sin mapear (por # de empresas):")
    safe_display(top_unknown_industries)
else:
    print("  · No hay industrias sin mapear en el top 20.")

# (Opcional) Ver también los sectores originales asociados a esos Unknown (si existe 'Sector'):
if 'Sector' in df.columns and top_unknown_industries.shape[0] > 0:
    cross = (
        df.loc[unk_mask & df['_Industry_raw'].isin(top_unknown_industries['Industry'])]
          .groupby(['_Industry_raw','Sector'])['Empresa'].nunique()
          .sort_values(ascending=False)
          .groupby(level=0).head(3)  # top-3 sectores por Industry sin mapear
          .rename('num_empresas')
          .reset_index()
          .rename(columns={'_Industry_raw':'Industry'})
    )
    print("\nSugerencias de contexto (Industry sin mapear × Sector original):")
    safe_display(cross)

# Limpieza de columnas auxiliares de este bloque
df.drop(columns=['_Industry_raw'], inplace=True, errors='ignore')



📊 Cobertura del mapeo a GICS
  · Filas con Sector_GICS='Unknown': 0.0%
  · Empresas afectadas: 0 / 180 (0.0%)
  · No hay industrias sin mapear en el top 20.


In [ ]:
# ==============================================================================
# BLOQUE 8 — Tasa Libre de Riesgo (FRED) con soporte OFFLINE
# ==============================================================================

import datetime as _dt
import pandas as _pd
import numpy as _np

def safe_get_risk_free(series="TB3MS", start="2014-01-01", end=None):
    """
    Devuelve DataFrame mensual con:
       ['risk_free_monthly_return','rf_m_lag1','rf_3m']
    o None si estamos OFFLINE o si falla la descarga.
    """
    if OFFLINE or not ENABLE_FRED:
        print("[OFFLINE] FRED deshabilitado → df_risk_free=None")
        return None

    end = end or _dt.date.today().isoformat()
    try:
        # 1) Intento con pandas_datareader (evita dependencia de fredapi)
        import pandas_datareader.data as pdr
        raw = pdr.DataReader(series, "fred", start, end)  # TB3MS en %, anualizado
        m = raw.resample("M").last().ffill()
        annual = _pd.to_numeric(m[series], errors="coerce") / 100.0
        annual = annual.clip(lower=-0.99, upper=1.00)
        out = _pd.DataFrame(index=m.index)
        out["risk_free_monthly_return"] = (1.0 + annual).pow(1.0/12.0) - 1.0
        out["rf_m_lag1"] = out["risk_free_monthly_return"].shift(1)
        out["rf_3m"] = (1 + out["risk_free_monthly_return"]).rolling(3).apply(_np.prod, raw=True) - 1
        out.index.name = "Fecha"
        return out
    except Exception as e:
        print(f"[WARN] FRED falló: {e} → df_risk_free=None")
        return None

# --- Obtener RF real o construir una neutra (ceros) si no hay red ---
df_risk_free = safe_get_risk_free(series="TB3MS", start="2014-01-01", end=None)

if df_risk_free is None:
    # Serie neutra 0% mensual alineada al rango de tu dataset
    m0 = df["Fecha"].min().to_period("M").to_timestamp("M")
    m1 = df["Fecha"].max().to_period("M").to_timestamp("M")
    idx = _pd.date_range(m0, m1, freq="M")
    df_risk_free = _pd.DataFrame(index=idx)
    df_risk_free["risk_free_monthly_return"] = 0.0
    df_risk_free["rf_m_lag1"] = 0.0
    df_risk_free["rf_3m"] = 0.0
    df_risk_free.index.name = "Fecha"

print("RF listo (head):")
print(df_risk_free.head(3))


[OFFLINE] FRED deshabilitado → df_risk_free=None
RF listo (head):
            risk_free_monthly_return  rf_m_lag1  rf_3m
Fecha                                                 
2014-10-31                       0.0        0.0    0.0
2014-11-30                       0.0        0.0    0.0
2014-12-31                       0.0        0.0    0.0


In [ ]:
# ==============================================================================
# BLOQUE 9 · FEATURE ENGINEERING AVANZADO
#   – Momentum (1-3-6-12 m)
#   – Volatilidad (3-6-12 m)
#   – Sorpresas macro
#   – Momentum 18 m + Vol-de-Vol 6 m
#   – PE-zscore 6 m
#   – Momentum sectorial (seguro, usando GICS/fallback)
#   – Dummies sector GICS  / Dummy pandemia
# ==============================================================================

print("\nIniciando Feature Engineering Avanzado…")

# ────────────────────────────────────────────────────────────────────────────────
# 0)  Asegurar listas y parámetros base
# ────────────────────────────────────────────────────────────────────────────────
if 'features_mercado' not in locals() or not isinstance(features_mercado, list):
    features_mercado = []

if 'MIN_PERIODS_WINDOW' not in locals():
    MIN_PERIODS_WINDOW = 3

if 'VALUATION_MOMENTUM_WINDOW' not in locals():
    VALUATION_MOMENTUM_WINDOW = 6

# ────────────────────────────────────────────────────────────────────────────────
# 9.1  Momentum y volatilidad de precios + sorpresas macro
# ────────────────────────────────────────────────────────────────────────────────

# --- Momentum de precios -------------------------------------------------------
print("  · Calculando momentum de precios...")
for p in (1, 3, 6, 12):
    col = f'ret_P_Share_{p}m_base'
    df[col] = df.groupby('Empresa')['P_Share'].pct_change(p)
    features_mercado.append(col)

# --- Volatilidad rolling -------------------------------------------------------
print("  · Calculando volatilidad de precios...")
for w in (3, 6, 12):
    col = f'vol_P_Share_{w}m_base'
    # rolling std por empresa manteniendo alineación
    df[col] = (
        df.groupby('Empresa', group_keys=False)['P_Share']
          .apply(lambda s: s.rolling(window=w, min_periods=MIN_PERIODS_WINDOW).std())
    )
    features_mercado.append(col)

# --- Sorpresas macro -----------------------------------------------------------
print("  · Calculando sorpresas macro...")
df['dif_CPI_mediana'] = (df['CPI'] - df['CPI_Exp_mediana']) * 10_000
df['dif_FFR_mediana'] = (df['Fed Funds Rate'] - df['Fed Funds Rate_Exp_mediana']) * 10_000
df['dif_NFP_mediana'] =  df['Non farm payrolls'] - df['Non farm payrolls_Exp_mediana']

print("  · Features base creadas (sin imputar aquí; el _lag1 y dropna limpian luego).")

# --- Momentum "skip-1" (12–1 y 6–1): mejor alineado al horizonte 3m ---
print("  · Calculando momentum 'skip-1' (12-1 y 6-1)...")
df['mom_12_1_tmp12'] = df.groupby('Empresa')['P_Share'].pct_change(12)
df['mom_12_1_tmp1']  = df.groupby('Empresa')['P_Share'].pct_change(1)
df['mom_12_1']       = df['mom_12_1_tmp12'] - df['mom_12_1_tmp1']

df['mom_6_1_tmp6']   = df.groupby('Empresa')['P_Share'].pct_change(6)
df['mom_6_1']        = df['mom_6_1_tmp6'] - df['mom_12_1_tmp1']

df.drop(columns=['mom_12_1_tmp12','mom_12_1_tmp1','mom_6_1_tmp6'], inplace=True)
features_mercado += ['mom_12_1','mom_6_1']

# ────────────────────────────────────────────────────────────────────────────────
# 9-plus  · Momentum 18 m (ret_18m_lag1)  +  Vol-de-Vol 6 m (vov_6m_lag1)
# ────────────────────────────────────────────────────────────────────────────────
print("  · Añadiendo momentum 18 m y vol-de-vol 6 m…")

# Momentum 18m con un lag adicional (info conocida al cierre t-1)
df['ret_18m_tmp']  = df.groupby('Empresa')['P_Share'].pct_change(18)
df['ret_18m_lag1'] = df.groupby('Empresa')['ret_18m_tmp'].shift(1).fillna(0.0)
df.drop(columns='ret_18m_tmp', inplace=True)
features_mercado.append('ret_18m_lag1')

# Vol-de-Vol 6m (std de la std rolling 6m), winsorizado 1-99%, luego lag(1)
vol6 = (
    df.groupby('Empresa', group_keys=False)['P_Share']
      .apply(lambda s: s.rolling(window=6, min_periods=3).std())
)
vov6 = (
    vol6.groupby(df['Empresa'], group_keys=False)
        .apply(lambda s: s.rolling(window=6, min_periods=3).std())
)

# winsorizar por percentiles globales (robusto)
p1, p99 = vov6.quantile([.01, .99])
vov6_clip = vov6.clip(lower=p1 if pd.notna(p1) else None,
                      upper=p99 if pd.notna(p99) else None)

df['vov_6m_lag1'] = vov6_clip.groupby(df['Empresa']).shift(1).fillna(0.0)
features_mercado.append('vov_6m_lag1')
print("    · ret_18m_lag1 y vov_6m_lag1 añadidas.")

# ────────────────────────────────────────────────────────────────────────────────
# 9.2  Z-score de valoración (P/E winsorizado, ventana 6 m)
# ────────────────────────────────────────────────────────────────────────────────
print("  · Calculando Z-score de valoración (PE_zscore_6m)...")

def winsor_1pct(series: pd.Series) -> pd.Series:
    """Winsoriza la serie al 1 % por cola, conservando índice."""
    if series.dropna().shape[0] < 2:
        return series
    arr = winsorize(series.dropna().astype(float), limits=[.01, .01])
    return pd.Series(arr, index=series.dropna().index, dtype='float64').reindex(series.index)

# 1) P/E winsorizado empresa-a-empresa
df['P_E_wins'] = (
    df.groupby('Empresa', group_keys=False)['P_E']
      .apply(winsor_1pct)
)

# 2) Mediana y MAD móvil (6m) por empresa
eps = 1e-6
def rolling_mad(x):
    med = np.median(x)
    return np.median(np.abs(x - med)) + eps

df['median_6m'] = (
    df.groupby('Empresa', group_keys=False)['P_E_wins']
      .apply(lambda s: s.rolling(window=VALUATION_MOMENTUM_WINDOW,
                                 min_periods=MIN_PERIODS_WINDOW).median())
)
df['mad_6m'] = (
    df.groupby('Empresa', group_keys=False)['P_E_wins']
      .apply(lambda s: s.rolling(window=VALUATION_MOMENTUM_WINDOW,
                                 min_periods=MIN_PERIODS_WINDOW).apply(rolling_mad, raw=True))
)

# 3) Z-score robusto
df['PE_zscore_6m'] = (df['P_E_wins'] - df['median_6m']) / (df['mad_6m'] * 1.4826)
df['PE_zscore_6m'].replace([np.inf, -np.inf], 0.0, inplace=True)
df['PE_zscore_6m'].fillna(0.0, inplace=True)

# 4) Dummy historial corto
df['is_short_hist_zscore'] = (df.groupby('Empresa').cumcount() < VALUATION_MOMENTUM_WINDOW).astype(int)

# Limpieza temporales
df.drop(columns=['P_E_wins', 'median_6m', 'mad_6m'], inplace=True)
features_mercado += ['PE_zscore_6m', 'is_short_hist_zscore']
print("  · Z-score de valoración (PE_zscore_6m) calculado.")

# ────────────────────────────────────────────────────────────────────────────────
# 9.2-bis  Momentum sectorial (seguro) — usa Sector_GICS con fallback
# ────────────────────────────────────────────────────────────────────────────────
print("  · Calculando señal de momentum sectorial (versión segura, GICS/fallback)…")

# Columna de sector de referencia: GICS si existe; si no, Sector; si no, 'Unknown'
sec1 = df['Sector_GICS'] if 'Sector_GICS' in df.columns else pd.Series(index=df.index, dtype='object')
sec2 = df['Sector']      if 'Sector'      in df.columns else pd.Series(index=df.index, dtype='object')
df['Sector_ref'] = sec1.where(sec1.notna(), sec2).fillna('Unknown')


# Usar precios lag-1 (info disponible al cierre t-1)
df['P_Share_l1'] = df.groupby('Empresa')['P_Share'].shift(1)

tmp = (
    df[['Fecha', 'Empresa', 'Sector_ref', 'P_Share_l1']]
      .dropna(subset=['Sector_ref', 'P_Share_l1'])
      .sort_values(['Empresa', 'Fecha'])
)

# Retornos sobre P_Share_l1 (→ info hasta t-1)
tmp['ret1']  = tmp.groupby('Empresa')['P_Share_l1'].pct_change(1)
tmp['ret12'] = tmp.groupby('Empresa')['P_Share_l1'].pct_change(12)

# Promedio por sector y score
sector_mom = (
    tmp.groupby(['Fecha', 'Sector_ref'])[['ret12', 'ret1']].mean().reset_index()
)
sector_mom['sec_score'] = 0.5 * (sector_mom['ret12'] - sector_mom['ret1'])

# Z-score cross-sectional por fecha
def _z(s):
    mu, sd = s.mean(), s.std(ddof=0)
    return (s - mu) / (sd + 1e-9) if sd > 0 else 0.0

sector_mom['sec_score_z'] = sector_mom.groupby('Fecha')['sec_score'].transform(_z)

# Unir al DF principal y dejar la columna YA laggeada
df = df.merge(sector_mom[['Fecha', 'Sector_ref', 'sec_score_z']],
              on=['Fecha', 'Sector_ref'], how='left')

df['sec_score_z_safe'] = df['sec_score_z'].fillna(0.0)
df.drop(columns=['sec_score_z'], inplace=True)

# registra la feature con el nuevo nombre
features_mercado.append('sec_score_z_safe')
print("  · Señal sectorial segura incorporada: sec_score_z_safe")

# ────────────────────────────────────────────────────────────────────────────────
# 9.3  Dummies de sector (GICS)
# ────────────────────────────────────────────────────────────────────────────────
print("  · Creando dummies sector GICS…")
d_sec = pd.get_dummies(df['Sector_GICS'], prefix='sector', drop_first=True)
df = pd.concat([df, d_sec], axis=1)

# añadir las dummies a la lista de mercado
features_mercado += [c for c in d_sec.columns]
print(f"  · Dummies sector GICS creadas: {len(d_sec.columns)} columnas")

# ────────────────────────────────────────────────────────────────────────────────
# 9.4  Dummy pandemia (mar-20 → jun-21)
# ────────────────────────────────────────────────────────────────────────────────
print("  · Creando dummy de pandemia...")
df['pandemic_dummy'] = ((df['Fecha'] >= '2020-03-01') & (df['Fecha'] <= '2021-06-30')).astype(int)
features_mercado.append('pandemic_dummy')
print("  · Dummy pandemia creada y añadida a las features.")

# ────────────────────────────────────────────────────────────────────────────────
# 9.5  Consolidar lista de features de mercado (sin duplicados)
# ────────────────────────────────────────────────────────────────────────────────
features_mercado = list(dict.fromkeys(features_mercado))  # dedup preservando orden

print("\n— Fin Bloque 4 · DataFrame listo para los lags contables —")
try:
    print("Ejemplo de filas:")
    print(df.head(5))
except Exception:
    pass



Iniciando Feature Engineering Avanzado…
  · Calculando momentum de precios...
  · Calculando volatilidad de precios...
  · Calculando sorpresas macro...
  · Features base creadas (sin imputar aquí; el _lag1 y dropna limpian luego).
  · Calculando momentum 'skip-1' (12-1 y 6-1)...
  · Añadiendo momentum 18 m y vol-de-vol 6 m…
    · ret_18m_lag1 y vov_6m_lag1 añadidas.
  · Calculando Z-score de valoración (PE_zscore_6m)...
  · Z-score de valoración (PE_zscore_6m) calculado.
  · Calculando señal de momentum sectorial (versión segura, GICS/fallback)…
  · Señal sectorial segura incorporada: sec_score_z_safe
  · Creando dummies sector GICS…
  · Dummies sector GICS creadas: 10 columnas
  · Creando dummy de pandemia...
  · Dummy pandemia creada y añadida a las features.

— Fin Bloque 4 · DataFrame listo para los lags contables —
Ejemplo de filas:
          Empresa      Fecha      P_E     P_B     P_S  P_Share       ROCE  \
0  AAPL US Equity 2014-10-31  16.8500  5.6796  3.6047   27.000  32.2468

In [ ]:
df.head(20)

,Empresa,Fecha,P_E,P_B,P_S,P_Share,ROCE,EBIT,Total Activos,Deuda a LP,...,sector_Consumer Staples,sector_Energy,sector_Financials,sector_Health Care,sector_Industrials,sector_Information Technology,sector_Materials,sector_Real Estate,sector_Utilities,pandemic_dummy
0,AAPL US Equity,2014-10-31,16.8500,5.6796,3.6047,27.000,32.246867,10576.333333,225626.333333,29015.666667,...,False,False,False,False,False,True,False,False,False,0
1,AAPL US Equity,2014-11-30,18.5553,6.2544,3.9695,29.733,32.929333,10870.666667,228732.666667,29001.333333,...,False,False,False,False,False,True,False,False,False,0
2,AAPL US Equity,2014-12-31,14.9215,5.2147,3.2904,27.595,33.611800,11165.000000,231839.000000,28987.000000,...,False,False,False,False,False,True,False,False,False,0
3,AAPL US Equity,2015-01-31,15.8381,5.5350,3.4925,29.290,34.123267,15525.333333,241857.333333,30159.333333,...,False,False,False,False,False,True,False,False,False,0
4,AAPL US Equity,2015-02-28,17.3656,6.0689,3.8294,32.115,34.634733,19885.666667,251875.666667,31331.666667,...,False,False,False,False,False,True,False,False,False,0
5,AAPL US Equity,2015-03-31,15.3999,5.5579,3.4464,31.108,35.146200,24246.000000,261894.000000,32504.000000,...,False,False,False,False,False,True,False,False,False,0
6,AAPL US Equity,2015-04-30,15.4890,5.5900,3.4664,31.288,36.221300,22256.666667,261660.666667,35026.666667,...,False,False,False,False,False,True,False,False,False,0
7,AAPL US Equity,2015-05-31,16.1239,5.8192,3.6085,32.570,37.296400,20267.333333,261427.333333,37549.333333,...,False,False,False,False,False,True,False,False,False,0
8,AAPL US Equity,2015-06-30,14.4905,5.6940,3.2548,31.356,38.371500,18278.000000,261194.000000,40072.000000,...,False,False,False,False,False,True,False,False,False,0
9,AAPL US Equity,2015-07-31,14.0140,5.5067,3.1477,30.325,39.296467,16879.666667,265179.666667,42521.000000,...,False,False,False,False,False,True,False,False,False,0


In [ ]:
# ===============================================================================
# BLOQUE 10 · ANCLAJE CONTABLE (Q-2) + TARGET + LISTAS DE FEATURES (SIMPLIFICADO)
# ===============================================================================

print("\n⏩  BLOQUE 5  –  Anclaje contable y construcción del dataset final…")

# ───────────────────────────────────────────────────────────────────────────────
# 10.0 · DEFINICIÓN DE LAS LISTAS DE FEATURES
# ───────────────────────────────────────────────────────────────────────────────

# 1) CONTABLES (nombres limpios desde el Excel)
features_contables = [
    'ROCE', 'ROA', 'EBIT', 'Total Activos', 'Deuda a LP',
    'Beneficio neto', 'ROI', 'EV', 'Cap de mercado',
    'Deuda a CP', 'Efectivo y equiv'
]
# Usar solo las columnas que realmente existen
features_contables = [col for col in features_contables if col in df.columns]

# 2) MERCADO / MOMENTUM (consolidar sin duplicar, preservando orden)
base_market = [
    'P_E','P_B','P_S',
    'ret_P_Share_1m_base','ret_P_Share_3m_base','ret_P_Share_6m_base','ret_P_Share_12m_base',
    'vol_P_Share_3m_base','vol_P_Share_6m_base','vol_P_Share_12m_base',
    'PE_zscore_6m','is_short_hist_zscore',
    'ret_18m_lag1','vov_6m_lag1','sec_score_z_safe'
]

if 'features_mercado' in locals() and isinstance(features_mercado, list):
    # dedup preservando el orden
    seen = set()
    features_mercado = [x for x in (features_mercado + base_market) if not (x in seen or seen.add(x))]
else:
    features_mercado = base_market[:]

# Añadir dummies sectoriales creadas en Bloque 4.3 (si existen)
features_mercado += [c for c in df.columns if c.startswith('sector_')]

# 3) MACRO
features_macro = ['dif_CPI_mediana', 'dif_NFP_mediana']

print(f"    · {len(features_contables):2d} contables, "
      f"{len(features_mercado):2d} mercado, "
      f"{len(features_macro):2d} macro definidos.")

# ───────────────────────────────────────────────────────────────────────────────
# 10.1 · ANCLAJE CONTABLE (Q-2)
# ───────────────────────────────────────────────────────────────────────────────
# Mapea cada mes (t) con el último dato contable disponible de t-2 trimestres
df['periodo_trimestre'] = df['Fecha'].dt.to_period('Q')
QUARTERLY_OFFSET = 2
df['periodo_informe'] = df['periodo_trimestre'] - QUARTERLY_OFFSET

# Tomar el último registro de cada trimestre por empresa (fin de trimestre)
df_cont_q = (
    df.sort_values('Fecha')
      .groupby(['Empresa', 'periodo_trimestre'])
      .tail(1)
      .copy()
)

# Eliminar columnas contables actuales antes del merge (evita sombras)
df.drop(columns=features_contables, errors='ignore', inplace=True)

# Merge: trae las contables del trimestre (t-2) a cada fila mensual en t
df = pd.merge(
    df,
    df_cont_q[['Empresa', 'periodo_trimestre'] + features_contables],
    left_on=['Empresa', 'periodo_informe'],
    right_on=['Empresa', 'periodo_trimestre'],
    how='left'
)

# Limpieza de columnas auxiliares
df.drop(columns=[c for c in df.columns if c.startswith('periodo_')], inplace=True, errors='ignore')
print("    · Contables anclados correctamente (Q-2).")

# ───────────────────────────────────────────────────────────────────────────────
# 10.2 · VARIABLE OBJETIVO (retorno futuro a 3 meses)
# ───────────────────────────────────────────────────────────────────────────────
# TARGET_HORIZON se define globalmente en el Bloque 3
TARGET_COL = f"retorno_futuro_{TARGET_HORIZON}m"

df[TARGET_COL] = df.groupby('Empresa')['P_Share'].shift(-TARGET_HORIZON) / df['P_Share'] - 1

rows_before = len(df)
df.dropna(subset=[TARGET_COL], inplace=True)
print(f"    · Target creado. Filas eliminadas por NaN en target: {rows_before - len(df):,}")

print("\n✅  Fin Bloque 5 – DataFrame listo para el Bloque 6 (lags finales).")

# ───────────────────────────────────────────────────────────────────────────────
# 10.3 · GUARDADO DE METADATOS DE LA CORRIDA (reproducibilidad)
# ───────────────────────────────────────────────────────────────────────────────
# Si ya existe predictor_cols_final (p.ej., tras Bloque 6), lo guardamos.
# Si no, guardamos un snapshot de features crudas (sin lag) para trazabilidad.
if 'predictor_cols_final' in locals():
    feature_cols_snapshot = predictor_cols_final
else:
    feature_cols_snapshot = list(dict.fromkeys(features_mercado + features_macro
     + features_contables))

# Asegurar carpeta de artefactos
os.makedirs("artifacts", exist_ok=True)

# Guardar registro de columnas/target y versiones
save_run_metadata(
    path_json="artifacts/run_metadata.json",
    seed=GLOBAL_SEED if 'GLOBAL_SEED' in globals() else 42,
    feature_cols=feature_cols_snapshot,
    target_col=TARGET_COL,
    extra={
    "cv": {"outer": "WalkForwardPurged", "gap_steps": int(GAP_STEPS), "hold_steps": 1},
    "scorer": "IC_neutralizado_fast",
    "notes": "Rebalance 3M (cohortes), target 3M residual, embargo 3M."
    }
)



⏩  BLOQUE 5  –  Anclaje contable y construcción del dataset final…
    · 11 contables, 38 mercado,  2 macro definidos.
    · Contables anclados correctamente (Q-2).
    · Target creado. Filas eliminadas por NaN en target: 540

✅  Fin Bloque 5 – DataFrame listo para el Bloque 6 (lags finales).
[OK] Metadatos guardados en: artifacts/run_metadata.json


In [ ]:
features_contables

['ROCE',
 'ROA',
 'EBIT',
 'Total Activos',
 'Deuda a LP',
 'Beneficio neto',
 'ROI',
 'EV',
 'Cap de mercado',
 'Deuda a CP',
 'Efectivo y equiv']

In [ ]:
df.head(20)

,Empresa,Fecha,P_E,P_B,P_S,P_Share,CPI,CPI_Exp_mediana,CPI_Exp_promedio,Fed Funds Rate,...,EBIT,Total Activos,Deuda a LP,Beneficio neto,ROI,EV,Cap de mercado,Deuda a CP,Efectivo y equiv,retorno_futuro_3m
0,AAPL US Equity,2014-10-31,16.8500,5.6796,3.6047,27.000,0.017,1.600000e-02,0.0162,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.084815
1,AAPL US Equity,2014-11-30,18.5553,6.2544,3.9695,29.733,0.013,1.600000e-02,0.0157,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.080113
2,AAPL US Equity,2014-12-31,14.9215,5.2147,3.2904,27.595,0.008,1.400000e-02,0.0142,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.127306
3,AAPL US Equity,2015-01-31,15.8381,5.5350,3.4925,29.290,-0.001,7.000000e-03,0.0069,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.068214
4,AAPL US Equity,2015-02-28,17.3656,6.0689,3.8294,32.115,0.000,-1.000000e-03,-0.0013,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.014168
5,AAPL US Equity,2015-03-31,15.3999,5.5579,3.4464,31.108,-0.001,-1.000000e-03,-0.0008,0.0025,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.007972
6,AAPL US Equity,2015-04-30,15.4890,5.5900,3.4664,31.288,-0.002,1.000000e-10,0.0002,0.0025,...,11165.0,231839.0,28987.0,8467.0,29.6832,471071.7208,603277.6191,6308.0,155239.0,-0.030779
7,AAPL US Equity,2015-05-31,16.1239,5.8192,3.6085,32.570,0.000,-2.000000e-03,-0.0016,0.0025,...,11165.0,231839.0,28987.0,8467.0,29.6832,471071.7208,603277.6191,6308.0,155239.0,-0.134480
8,AAPL US Equity,2015-06-30,14.4905,5.6940,3.2548,31.356,0.001,1.000000e-03,0.0006,0.0025,...,11165.0,231839.0,28987.0,8467.0,29.6832,471071.7208,603277.6191,6308.0,155239.0,-0.120583
9,AAPL US Equity,2015-07-31,14.0140,5.5067,3.1477,30.325,0.002,1.000000e-03,0.0015,0.0025,...,24246.0,261894.0,32504.0,18024.0,28.3788,522601.5018,668533.0938,3899.0,177955.0,-0.014839


In [ ]:
# ===============================================================================
# BLOQUE DIAGNÓSTICO · Validaciones de anclaje y target
# ===============================================================================

print("\n🕵️  Iniciando validaciones…")

# --- ASUNCIONES ---
# df: DataFrame final (contables ya anclados Q-2)
# df_cont_q: DataFrame creado en el Bloque 5 que contiene los datos originales
# features_contables: Lista de features contables
# TARGET_COL: Nombre de la columna objetivo
# TARGET_HORIZON: Horizonte de la variable objetivo

# --------------------------------------------------------------------------- #
# 1) Chequeo de anclaje contable (Q-2) sin tautologías
#    - Debe ser CONSTANTE dentro de cada trimestre
#    - Los CAMBIOS solo deben ocurrir cuando cambia el trimestre
# --------------------------------------------------------------------------- #
print("\n🔎 Chequeo 1 — Constancia intra-trimestre y cambios solo en cambio de trimestre")

# Asegurar lista de columnas contables
if 'features_contables' not in globals() or not features_contables:
    features_contables = []

viol_intra_all = []
viol_boundary_all = []

# Precalcular periodos trimestrales actuales y previos por fila/empresa
Q_now  = df['Fecha'].dt.to_period('Q')
Q_prev = df.groupby('Empresa')['Fecha'].shift(1).dt.to_period('Q')

for col in features_contables:
    if col not in df.columns:
        continue

    # (A) Constancia dentro de cada trimestre
    nunq = (df.groupby(['Empresa', Q_now])[col]
              .nunique(dropna=False)
              .rename('n_unique')
              .reset_index())
    bad_intra = nunq.loc[nunq['n_unique'] > 1].copy()
    if not bad_intra.empty:
        bad_intra['col'] = col
        viol_intra_all.append(bad_intra)

    # (B) Cambios solo cuando cambia el trimestre
    chg = df.groupby('Empresa')[col].diff().abs() > 1e-12  # cambio de valor
    same_q = (Q_now == Q_prev)
    bad_boundary = df.loc[chg & same_q, ['Empresa', 'Fecha', col]].copy()
    if not bad_boundary.empty:
        bad_boundary['col'] = col
        bad_boundary['detalle'] = 'Cambio dentro del mismo trimestre'
        viol_boundary_all.append(bad_boundary)

# Reporte
if not viol_intra_all and not viol_boundary_all:
    print("✅ Chequeo 1 – Contables constantes por trimestre y cambios solo en límites de trimestre.")
else:
    if viol_intra_all:
        print("⚠️ Chequeo 1A – Variaciones INTRA-trimestre detectadas (revisar anclaje Q-2):")
        safe_display(pd.concat(viol_intra_all, ignore_index=True).head(20))
    if viol_boundary_all:
        print("⚠️ Chequeo 1B – Cambios detectados dentro del mismo trimestre (deberían ocurrir solo en el cambio de Q):")
        safe_display(pd.concat(viol_boundary_all, ignore_index=True).head(20))

# --------------------------------------------------------------------------- #
# 2) Vista rápida de 3 empresas al azar
# --------------------------------------------------------------------------- #
np.random.seed(42)
sel_empresas = np.random.choice(df['Empresa'].unique(), size=3, replace=False)

# --- CORRECCIÓN: Usar solo las columnas '_l' que ahora existen en df ---
cont_cols_demo = ['ROA', 'Total Activos']
# Asegurarse de que las columnas de demostración realmente existan en el DataFrame
cont_cols_demo_exist = [col for col in cont_cols_demo if col in df.columns]
cols_demo = ['Empresa', 'Fecha', 'P_Share'] + cont_cols_demo_exist

muestra = (df.loc[df['Empresa'].isin(sel_empresas), cols_demo]
             .sort_values(['Empresa', 'Fecha'])
             .groupby('Empresa', group_keys=False)
             .head(15)
             .reset_index(drop=True))

print("\n🔍 Chequeo 2 – 3 empresas al azar (primeras 15 fechas c/u):")
safe_display(muestra)

# --------------------------------------------------------------------------- #
# 3) ¿Los contables cambian solo en el mes en que llega el nuevo informe?
# --------------------------------------------------------------------------- #
violaciones = []
# El bucle ahora itera sobre 'features_contables', que solo contiene las '_l'
for col in features_contables:
    if col in df.columns:
        cambia = df.groupby('Empresa')[col].diff().abs() > 1e-12
        bad = (cambia.groupby([df['Empresa'], df['Fecha'].dt.to_period('Q')])
                     .sum()
                     .reset_index(name='n_cambios')
                     .query('n_cambios > 1'))
        if not bad.empty:
            bad['col'] = col
            violaciones.append(bad)

if not violaciones:
    print("✅ Chequeo 3 – Sin look-ahead: los contables solo cambian cuando toca.")
else:
    print("⚠️ Chequeo 3 – Algunos contables cambian antes de tiempo:")
    safe_display(pd.concat(violaciones).head())

# --------------------------------------------------------------------------- #
# 4) Consistencia del target
# --------------------------------------------------------------------------- #
chk = df[['Empresa', 'Fecha', 'P_Share', TARGET_COL]].copy()
chk['P_Share_futuro'] = chk.groupby('Empresa')['P_Share'].shift(-TARGET_HORIZON)
chk['reconstruido'] = (1 + chk[TARGET_COL]) * chk['P_Share']

err = (chk['reconstruido'] - chk['P_Share_futuro']).abs()
if err.dropna().max() < 1e-6:
    print(f"✅ Chequeo 4 – El target ({TARGET_HORIZON}m) se reconstruye sin error numérico.")
else:
    print("⚠️ Chequeo 4 – Discrepancias en la reconstrucción del target:")
    safe_display(chk.loc[err > 1e-6, ['Empresa', 'Fecha', 'P_Share',
                                 'P_Share_futuro', TARGET_COL]].head())

print("\n🟢 Validaciones finalizadas.\n")


🕵️  Iniciando validaciones…

🔎 Chequeo 1 — Constancia intra-trimestre y cambios solo en cambio de trimestre
✅ Chequeo 1 – Contables constantes por trimestre y cambios solo en límites de trimestre.

🔍 Chequeo 2 – 3 empresas al azar (primeras 15 fechas c/u):


,Empresa,Fecha,P_Share,ROA,Total Activos
0,CENT US Equity,2014-10-31,6.2457,NaN,NaN
1,CENT US Equity,2014-11-30,5.7291,NaN,NaN
2,CENT US Equity,2014-12-31,6.8718,NaN,NaN
3,CENT US Equity,2015-01-31,6.6135,NaN,NaN
4,CENT US Equity,2015-02-28,7.0910,NaN,NaN
5,CENT US Equity,2015-03-31,7.7249,NaN,NaN
6,CENT US Equity,2015-04-30,7.2436,0.7623,1148.727
7,CENT US Equity,2015-05-31,7.4353,0.7623,1148.727
8,CENT US Equity,2015-06-30,8.2650,0.7623,1148.727
9,CENT US Equity,2015-07-31,7.1927,1.3489,1188.963


✅ Chequeo 3 – Sin look-ahead: los contables solo cambian cuando toca.
✅ Chequeo 4 – El target (3m) se reconstruye sin error numérico.

🟢 Validaciones finalizadas.



In [ ]:
# ===============================================================================
# Tabla de Diagnóstico de Anclaje
#   - Une por TRIMESTRE (Q), no por Fecha, para evitar NaNs en meses intra-Q
#   - (Opcional) Trae también el valor ORIGINAL del trimestre t-2 (Qm2)
# ===============================================================================

print("\n🕵️  Generando Tabla de Diagnóstico de Anclaje (Versión Final, Q-merge)...")

# 1) Elegí empresa y features a verificar
empresa_a_verificar = sel_empresas[0]
features_a_verificar = ['ROA', 'EBIT']
features_a_verificar_exist = [c for c in features_a_verificar if c in df.columns]

print(f"\nGenerando tabla para Empresa: {empresa_a_verificar}")

# 2) Subsets: anclado (mensual) y crudo trimestral
df_empresa_anclado   = df.loc[df['Empresa'] == empresa_a_verificar].copy()
df_empresa_original_q = df_cont_q.loc[df_cont_q['Empresa'] == empresa_a_verificar].copy()

# 3) Periodos trimestrales
df_empresa_anclado['Q']    = pd.PeriodIndex(df_empresa_anclado['Fecha'], freq='Q')
df_empresa_original_q['Q'] = pd.PeriodIndex(df_empresa_original_q['Fecha'], freq='Q')

# 4) Base de anclado (mensual) + renombrar columnas ancladas
tabla_final = df_empresa_anclado[['Fecha', 'Q'] + features_a_verificar_exist].copy()
tabla_final.rename(
    columns={col: f'{col}_Anclado' for col in features_a_verificar_exist},
    inplace=True
)

# 5) Traer ORIGINAL del mismo trimestre (Q)
df_orig_Q = df_empresa_original_q[['Q'] + features_a_verificar_exist].rename(
    columns={col: f'{col}_Original_Q' for col in features_a_verificar_exist}
)
tabla_final = tabla_final.merge(df_orig_Q, on='Q', how='left')

# 6) (Opcional) Traer ORIGINAL del trimestre t-2 (Qm2) para comparar 1:1 con Anclado
tabla_final['Qm2'] = tabla_final['Q'] - 2
df_orig_Qm2 = df_empresa_original_q[['Q'] + features_a_verificar_exist].rename(
    columns={'Q': 'Qm2', **{col: f'{col}_Original_Qm2' for col in features_a_verificar_exist}}
)
tabla_final = tabla_final.merge(df_orig_Qm2, on='Qm2', how='left')

# 7) Armar columnas para mostrar
cols_display = ['Fecha']
for f in features_a_verificar_exist:
    # Orden sugerido: Original del Q actual, Original Q-2 (alineable con anclado), y Anclado(Q-2)
    cols_display.extend([f'{f}_Original_Q', f'{f}_Original_Qm2', f'{f}_Anclado'])

tabla_final_display = (
    tabla_final[cols_display]
      .sort_values('Fecha')
      .reset_index(drop=True)
)

# 8) Mostrar una vista corta
print("\nTabla de Diagnóstico (Original_Q, Original_Qm2, Anclado[Q-2]):")
safe_display(tabla_final_display.head(15))
print("\n🟢 Tabla de diagnóstico generada (merge por trimestre).")



🕵️  Generando Tabla de Diagnóstico de Anclaje (Versión Final, Q-merge)...

Generando tabla para Empresa: CENT US Equity

Tabla de Diagnóstico (Original_Q, Original_Qm2, Anclado[Q-2]):


,Fecha,ROA_Original_Q,ROA_Original_Qm2,ROA_Anclado,EBIT_Original_Q,EBIT_Original_Qm2,EBIT_Anclado
0,2014-10-31,0.7623,NaN,NaN,1.373,NaN,NaN
1,2014-11-30,0.7623,NaN,NaN,1.373,NaN,NaN
2,2014-12-31,0.7623,NaN,NaN,1.373,NaN,NaN
3,2015-01-31,1.3489,NaN,NaN,1.138,NaN,NaN
4,2015-02-28,1.3489,NaN,NaN,1.138,NaN,NaN
5,2015-03-31,1.3489,NaN,NaN,1.138,NaN,NaN
6,2015-04-30,1.4166,0.7623,0.7623,49.971,1.373,1.373
7,2015-05-31,1.4166,0.7623,0.7623,49.971,1.373,1.373
8,2015-06-30,1.4166,0.7623,0.7623,49.971,1.373,1.373
9,2015-07-31,2.7799,1.3489,1.3489,38.993,1.138,1.138



🟢 Tabla de diagnóstico generada (merge por trimestre).


In [ ]:
# ===============================================================================
# BLOQUE 11  –  Dataset final (lags solo mercado & macro)  [versión robusta]
# ===============================================================================

print("\n⏩  BLOQUE 11  –  Generando dataset definitivo para el modelo…")

# 0) Orden correcto para que el shift funcione bien
df = df.sort_values(['Empresa','Fecha']).reset_index(drop=True)

# 1) Compilar listas finales
#    - Mercado + Macro → lag(1), filtrando estáticas
#    - Contables → directos (ya anclados Q-2)
STATIC_PREFIXES = ('sector_',)
STATIC_EXACT    = {'is_short_hist_zscore', 'sec_score_z_safe', 'pandemic_dummy'}

need_lag1_raw = sorted(list({*(features_mercado or []), *(features_macro or [])}))
need_lag1 = [c for c in need_lag1_raw
             if not c.startswith(STATIC_PREFIXES) and c not in STATIC_EXACT]
use_as_is = sorted(features_contables or [])

print(f"    · Mercado+Macro (lag-1): {len(need_lag1)}  ·  Contables directos: {len(use_as_is)}")

# 2) Crear/poblar columnas lag-1 para mercado+macro (evita doblar lag)
for col in need_lag1:
    if col in df.columns and not col.endswith('_lag1'):
        df[f"{col}_lag1"] = df.groupby('Empresa', sort=False)[col].shift(1)

# 2bis) Detectar columnas estáticas que deben ir 'tal cual' (sin lag)
sector_dummy_cols = [c for c in df.columns if c.startswith('sector_')]
static_passthrough = [c for c in ['sec_score_z_safe', 'is_short_hist_zscore', 'pandemic_dummy'] if c in df.columns]

# 2ter) Predictores definitivos:
#   - mercado/macro con _lag1,
#   - contables tal cual,
#   - estáticas tal cual (dummies y señales ya normalizadas)
predictor_cols_final = [
    (c if c.endswith('_lag1') else f"{c}_lag1") for c in need_lag1 if c in df.columns
] + use_as_is + sector_dummy_cols + static_passthrough

# dedup preservando orden + filtrar por existencia real
predictor_cols_final = list(dict.fromkeys([c for c in predictor_cols_final if c in df.columns]))

# 3) Construir df_model y castear a numérico ANTES de limpiar NaNs
model_cols = ['Empresa', 'Fecha', TARGET_COL] + predictor_cols_final
df_model   = df[model_cols].copy()

for c in predictor_cols_final:
    # robustez: coerciona a numérico; NaNs se limpian en el paso siguiente
    df_model[c] = pd.to_numeric(df_model[c], errors='coerce').astype('float32')

# 4) Diagnóstico de NaNs en predictores (mostrar solo los que tienen huecos)
nan_cnt = df_model[predictor_cols_final].isna().sum()
nan_cnt = nan_cnt[nan_cnt > 0].sort_values(ascending=False)
if not nan_cnt.empty:
    print("\n[NaNs por columna en predictores] (top):")
    print(nan_cnt.head(30))

# 4bis) Imputación neutral por mes y limpieza final de NaNs
before = len(df_model)

num_cols = df_model[predictor_cols_final].select_dtypes(include=np.number).columns.tolist()
cols_a_imputar = [
    c for c in num_cols
    if not c.startswith('sector_') and c not in {'is_short_hist_zscore', 'pandemic_dummy', 'sec_score_z_safe'}
]
print(f"    · Realizando imputación neutral por mes en {len(cols_a_imputar)} columnas...")

# Imputar con mediana del mes
df_model[cols_a_imputar] = (
    df_model.groupby(df_model['Fecha'].dt.to_period('M'))[cols_a_imputar]
            .transform(lambda g: g.fillna(g.median()))
)

# Fallback con ffill por empresa y luego 0.0
df_model[cols_a_imputar] = df_model.groupby('Empresa')[cols_a_imputar].ffill()
df_model.fillna(0.0, inplace=True)

print(f"    · Imputación completada. NaNs restantes: {df_model[predictor_cols_final].isna().sum().sum()}")

# 5) Salidas finales
X = df_model[predictor_cols_final]
y = df_model[TARGET_COL]
groups = df_model['Empresa']

# ✅ Sanity checks
print("En predictor_cols_final → sector_*:", sum(c.startswith('sector_') for c in predictor_cols_final))
print("En predictor_cols_final → sec_score_z_safe?:", 'sec_score_z_safe' in predictor_cols_final)
print("En predictor_cols_final → is_short_hist_zscore?:", 'is_short_hist_zscore' in predictor_cols_final)

print("En X → incluye sector_*?:", any(c.startswith('sector_') for c in X.columns))
print("En X → incluye sec_score_z_safe?:", 'sec_score_z_safe' in X.columns)
print("En X → incluye is_short_hist_zscore?:", 'is_short_hist_zscore' in X.columns)

# (opcional) corta la corrida si faltan
assert any(c.startswith('sector_') for c in X.columns), "Faltan dummies sector_* en X"
assert 'sec_score_z_safe' in X.columns, "Falta sec_score_z_safe en X"

print(f"    · X listo: {X.shape[0]:,} filas × {X.shape[1]} cols | empresas={groups.nunique()}")

# 6) Checks rápidos de cobertura de features estáticas
print("✔ Incluye sec_score_z_safe?:", 'sec_score_z_safe' in predictor_cols_final)
print("✔ Incluye is_short_hist_zscore?:", 'is_short_hist_zscore' in predictor_cols_final)
print("✔ Incluye dummies sector_*?:", any(c.startswith('sector_') for c in predictor_cols_final))

# 7) (Opcional) Exportar set de columnas estáticas para wrappers (BLOQUE 11.y)
#    Construido en base a lo que realmente quedó en predictor_cols_final.
try:
    macro_bases = list(features_macro)
except NameError:
    macro_bases = []

macro_lag_cols_in_X = [f"{c}_lag1" for c in macro_bases if f"{c}_lag1" in predictor_cols_final]
sector_cols_in_X    = [c for c in predictor_cols_final if c.startswith('sector_')]
static_cols_for_wrappers = sorted(set(
    macro_lag_cols_in_X + sector_cols_in_X +
    [c for c in ['sec_score_z_safe','is_short_hist_zscore'] if c in predictor_cols_final]
))
print(f"    · Passthrough estático (p/ wrappers): {len(static_cols_for_wrappers)} cols")



⏩  BLOQUE 11  –  Generando dataset definitivo para el modelo…
    · Mercado+Macro (lag-1): 17  ·  Contables directos: 11

[NaNs por columna en predictores] (top):
mom_12_1_lag1                2340
ret_P_Share_12m_base_lag1    2340
mom_6_1_lag1                 1260
ret_P_Share_6m_base_lag1     1260
EV                           1080
EBIT                         1080
Beneficio neto               1080
Cap de mercado               1080
Deuda a CP                   1080
Efectivo y equiv             1080
ROA                          1080
ROCE                         1080
ROI                          1080
Total Activos                1080
Deuda a LP                   1080
ret_P_Share_3m_base_lag1      720
vol_P_Share_12m_base_lag1     540
vol_P_Share_6m_base_lag1      540
vol_P_Share_3m_base_lag1      540
ret_P_Share_1m_base_lag1      360
P_E_lag1                      180
P_S_lag1                      180
PE_zscore_6m_lag1             180
P_B_lag1                      180
dif_CPI_mediana_lag1

In [ ]:
# ======================================================================
# BLOQUE: Auxiliares para Pipelines (Ensure2D y NanFix)
# ======================================================================
from sklearn.preprocessing import FunctionTransformer
import numpy as np
import pandas as pd

def _ensure_2d(A):
    if isinstance(A, pd.DataFrame):
        A = A.to_numpy()
    A = np.asarray(A, dtype=np.float32)
    if A.ndim == 1:
        A = A.reshape(-1, 1)
    return A

ensure2d = FunctionTransformer(_ensure_2d, validate=False)

def _ensure_df(X):
    import pandas as pd
    return X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)

ensure_df = FunctionTransformer(_ensure_df, validate=False)

def _nan_fix(a):
    return np.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)

nan_fix = FunctionTransformer(_nan_fix, validate=False)


In [ ]:
# ===============================================================================
# BLOQUE 11.x · Dataset Final: Filtros, Cohortes y Target Residual
# ===============================================================================

# --- 1. Filtro de universo mínimo (adaptativo) ---
print("\n🧹  Filtrando meses con universo insuficiente…")
K = 20 # Tamaño del portafolio long/short
n_by_month0 = df_model.groupby('Fecha')['Empresa'].nunique()
u_p25 = float(np.percentile(n_by_month0, 25)) if not n_by_month0.empty else 3 * K
UNIVERSE_MIN = int(max(3 * K, u_p25))
print(f"[INFO ADAPTATIVO] UNIVERSE_MIN se ha fijado en {UNIVERSE_MIN} (basado en K={K} y p25={u_p25:.0f})")

mask_month_size_ok = (n_by_month0 >= UNIVERSE_MIN)
good_months = n_by_month0.index[mask_month_size_ok]
df_model = df_model[df_model['Fecha'].isin(good_months)].copy()

# --- 2. Etiquetado de Cohortes Trimestrales (SIN FILTRAR MESES) ---
print("\n" + "="*50)
print("ETIQUETANDO COHORTES TRIMESTRALES (0,1,2) - Rebalanceo 3m, evaluación mensual")
if 'cohorte3' not in df_model.columns:
    df_model['cohorte3'] = (df_model['Fecha'].dt.month - 1) % 3
df_model = df_model.sort_values(['Fecha','Empresa']).reset_index(drop=True)
print(f"  · Se ha añadido la columna 'cohorte3'. El número de filas no cambia: {len(df_model):,}")
print("="*50 + "\n")

# --- 3. Alineación de DataFrames Auxiliares ---
df_model_sorted = df_model.copy()
df_sorted = (
    df.set_index(['Empresa','Fecha'])
      .loc[pd.MultiIndex.from_frame(df_model[['Empresa','Fecha']])]
      .reset_index()
)

# --- 4. Saneamiento de Nombres de Columnas ---
def _sanitize(idx):
    return (pd.Index(idx).str.replace(r"\s+", "_", regex=True).str.replace("/", "_"))
df_model = df_model.rename(columns=dict(zip(df_model.columns, _sanitize(df_model.columns))))
predictor_cols_final = list(_sanitize(predictor_cols_final))
TARGET_COL = str(_sanitize([TARGET_COL])[0])

# --- 5. Reconstrucción de X, y, groups, DATES_SERIES ---
X = df_model[predictor_cols_final].copy().astype('float32')

# --- 6. TARGET RESIDUAL PARA ENTRENAMIENTO ---
print("\n--- Modificando target a retorno residual (vs. sector) para entrenamiento ---")
m   = pd.to_datetime(df_model['Fecha']).dt.to_period('M')
sec = df_sorted.loc[df_model.index, 'Sector_GICS'] if 'Sector_GICS' in df_sorted.columns else pd.Series('UNK', index=df_model.index)
y_raw = df_model[TARGET_COL].astype(float)
sec_ret = df_model.assign(_m=m, _s=sec).groupby(['_m','_s'])[TARGET_COL].transform('mean')
y = (y_raw - sec_ret).astype('float32')
print("   ✅ Target 'y' ahora es el retorno idiosincrático.\n")

groups = df_model['Empresa'].copy()
DATES_SERIES = pd.to_datetime(df_model['Fecha']).reset_index(drop=True)

# --- 7. Limpieza Final y Alineación de y_raw ---
X.columns = (pd.Index(X.columns).str.replace(r"\s+", "_", regex=True).str.replace("/", "_"))
predictor_cols_final = list(X.columns)
X.replace([np.inf, -np.inf], np.nan, inplace=True)
X.fillna(0.0, inplace=True)
mask_ok = y.notna().to_numpy() & np.isfinite(y.to_numpy())

X = X.loc[mask_ok].reset_index(drop=True)
y = y.loc[mask_ok].reset_index(drop=True)
y_raw = y_raw.loc[mask_ok].reset_index(drop=True) # <-- ALINEACIÓN CRÍTICA DE Y_RAW
groups = groups.loc[mask_ok].reset_index(drop=True)
DATES_SERIES = DATES_SERIES.loc[mask_ok].reset_index(drop=True)
df_model_sorted = df_model_sorted.loc[mask_ok].reset_index(drop=True)
df_sorted       = df_sorted.loc[mask_ok].reset_index(drop=True)

print(f"  · Dataset final tras filtro: X={X.shape}, y={y.shape}, empresas={groups.nunique()}")

# ================================================================================
# === CREACIÓN DE LA SERIE DE MESES ALINEADA PARA EL ENSEMBLE ===
# ================================================================================
# Esta Serie es crucial: tiene el mismo índice que X e y, garantizando una alineación perfecta.
# Se define aquí para que esté disponible globalmente para las funciones helper que vienen después.
MONTHS_ALL = pd.Series(
    pd.to_datetime(DATES_SERIES).dt.to_period('M').values,
    index=X.index,
    name="Mes"
)
print(f"  · Serie MONTHS_ALL creada y alineada. Longitud: {len(MONTHS_ALL)}")


🧹  Filtrando meses con universo insuficiente…
[INFO ADAPTATIVO] UNIVERSE_MIN se ha fijado en 180 (basado en K=20 y p25=180)

ETIQUETANDO COHORTES TRIMESTRALES (0,1,2) - Rebalanceo 3m, evaluación mensual
  · Se ha añadido la columna 'cohorte3'. El número de filas no cambia: 21,060


--- Modificando target a retorno residual (vs. sector) para entrenamiento ---
   ✅ Target 'y' ahora es el retorno idiosincrático.

  · Dataset final tras filtro: X=(21060, 41), y=(21060,), empresas=180
  · Serie MONTHS_ALL creada y alineada. Longitud: 21060


In [ ]:
# ============================================================
# Alineación robusta para "Baselines de señal — IC mensual"
# Colocar justo antes de calcular el IC mensual.
# Requisitos en memoria: df_model, y, df_sorted
#   - df_model: columnas ['Fecha','Empresa', ...]
#   - y: target (Series o 1-col DataFrame) ya filtrado (len == n obs. válidas)
#   - df_sorted: columnas ['Fecha','Empresa','P_Share'] de la señal
# ============================================================
import pandas as pd
import numpy as np

def _norm_emp(s):
    return (s.astype(str)
              .str.strip()
              .str.upper()
              .str.replace(r"\s+", " ", regex=True))

def _norm_fecha_monthend(s):
    # Normaliza: parsea a datetime con tz unificado, elimina tz, lleva a fin de mes
    dt = pd.to_datetime(s, errors='coerce', utc=True)
    dt = dt.dt.tz_convert(None)  # vuelve naive
    return dt.dt.to_period('M').dt.to_timestamp('M')  # fin de mes

# --- 1) Copias y normalización de tipos
df_model = df_model.copy()
df_sorted = df_sorted.copy()

# --- 2) Subconjunto de df_model consistente con y
if isinstance(y, pd.DataFrame):
    y_series = y.iloc[:, 0].copy()
elif isinstance(y, pd.Series):
    y_series = y.copy()
else:
    y_series = pd.Series(np.asarray(y), name='target')

# Si y indexa a df_model, respétalo; si no, usa corte posicional
if hasattr(y_series, "index") and y_series.index.dtype == df_model.index.dtype and len(y_series.index) > 0:
    df_model_filtered = df_model.loc[y_series.index].copy()
else:
    df_model_filtered = df_model.iloc[:len(y_series)].copy()
    y_series.index = df_model_filtered.index  # fuerza mismo índice

# --- 3) Claves normalizadas
df_model_filtered["_FechaN"]   = _norm_fecha_monthend(df_model_filtered["Fecha"])
df_model_filtered["_EmpresaN"] = _norm_emp(df_model_filtered["Empresa"])

df_sorted["_FechaN"]   = _norm_fecha_monthend(df_sorted["Fecha"])
df_sorted["_EmpresaN"] = _norm_emp(df_sorted["Empresa"])

# --- 4) Deduplicación en señal por clave (si hay múltiples por (Fecha,Empresa))
df_sorted_key = (df_sorted
                 .dropna(subset=["_FechaN","_EmpresaN"])
                 .drop_duplicates(subset=["_FechaN","_EmpresaN"], keep="last")
                 .set_index(["_FechaN","_EmpresaN"]))

# --- 5) Construye base con clave canónica desde df_model_filtered + target
base = df_model_filtered[["_FechaN","_EmpresaN"]].copy()
base["target"] = y_series.values
base = base.dropna(subset=["_FechaN","_EmpresaN"])
base = base.set_index(["_FechaN","_EmpresaN"])

# --- 6) Join interno por clave; si queda vacío, diagnosticamos y hacemos fallback ordenado
aux = base.join(df_sorted_key[["P_Share"]], how="inner").reset_index()

print("[DEBUG] Alineación para IC mensual")
print("   len(y)                 :", len(y_series))
print("   len(df_sorted P_Share) :", len(df_sorted_key))
print("   len(aux tras inner)    :", len(aux))

fallback_used = False
if len(aux) == 0:
    # ---------- Diagnóstico rápido ----------
    # Muestra algunas claves únicas de cada lado para detectar patrón de mismatch
    left_keys  = base.reset_index()[["_FechaN","_EmpresaN"]].drop_duplicates().head(5)
    right_keys = df_sorted_key.reset_index()[["_FechaN","_EmpresaN"]].drop_duplicates().head(5)
    print("[DIAG] Ejemplos claves LEFT (base):")
    print(left_keys)
    print("[DIAG] Ejemplos claves RIGHT (señal):")
    print(right_keys)

    # ---------- Fallback por orden ----------
    # Ordena por claves normalizadas y alinea por posición
    base_ord = (base
                .reset_index()
                .sort_values(by=["_FechaN","_EmpresaN"], kind="mergesort")
                .reset_index(drop=True))
    sign_ord = (df_sorted_key
                .reset_index()
                .sort_values(by=["_FechaN","_EmpresaN"], kind="mergesort")
                .reset_index(drop=True))

    n = min(len(base_ord), len(sign_ord))
    aux = pd.DataFrame({
        "Fecha"   : base_ord["_FechaN"].iloc[:n].values,
        "Empresa" : base_ord["_EmpresaN"].iloc[:n].values,
        "target"  : base_ord["target"].iloc[:n].values,
        "P_Share" : sign_ord["P_Share"].iloc[:n].values
    })
    fallback_used = True
    print(f"[WARN] Join por claves vacío → se aplicó FALLBACK por orden (n={n}). "
          "Verifica que Empresa y Fecha estén normalizadas a las mismas convenciones.")

else:
    # Renombra columnas normalizadas a salida estándar
    aux = aux.rename(columns={"_FechaN":"Fecha","_EmpresaN":"Empresa"})

# --- 7) Chequeos finales
aux = aux.dropna(subset=["target","P_Share"]).copy()
print("   len(aux final)         :", len(aux))
if fallback_used:
    # Recomendación explícita si se usó fallback
    print("[SUGERENCIA] Alinea la generación de df_sorted para que use EXACTAMENTE "
          "las mismas claves (Empresa y Fecha fin de mes) que df_model/target.")

# --- 8) Cálculo de IC mensual (ya con aux alineado)
aux["Mes"] = pd.to_datetime(aux["Fecha"]).dt.to_period("M").dt.to_timestamp("M")

def _spearman_ic(g):
    g2 = g[["P_Share","target"]].dropna()
    if len(g2) < 3:
        return np.nan
    return g2.corr(method="spearman").iloc[0,1]

ic_mensual = aux.groupby("Mes", sort=True).apply(_spearman_ic).rename("IC")
ic_prom = ic_mensual.mean(skipna=True)
pct_pos = (ic_mensual > 0).mean() * 100

print(ic_mensual.describe())
print(f"IC medio: {ic_prom:.4f} | % meses IC>0: {pct_pos:.1f}%")


[DEBUG] Alineación para IC mensual
   len(y)                 : 21060
   len(df_sorted P_Share) : 21060
   len(aux tras inner)    : 21060
   len(aux final)         : 21060
count    117.000000
mean      -0.045781
std        0.129049
min       -0.409846
25%       -0.127216
50%       -0.043374
75%        0.039171
max        0.200045
Name: IC, dtype: float64
IC medio: -0.0458 | % meses IC>0: 37.6%


In [ ]:
# === Wrapper para que el tuner respete los grupos/meses en rankers ===
from sklearn.pipeline import Pipeline

class RankPipeline(Pipeline):
    """
    Pipeline que agrega soporte de 'ranking' (grupos por mes) sin caer en recursión.
    Se apoya en fit_with_rank_support, pero usa una bandera interna para no reentrar.
    """
    def fit(self, X, y=None, **fit_params):
        # Si ya estamos dentro de un fit "con ranking", actuamos como un pipeline normal
        if getattr(self, "_in_rank_fit", False):
            # Purga TOTAL de params no estándar para Pipeline.fit
            keys = list(fit_params.keys())
            drop = [k for k in keys
                    if k in ("months", "group")
                    or k.endswith("__months")
                    or k.endswith("__group")]
            for k in drop:
                fit_params.pop(k, None)
            if drop:
                print("[RANKPIPELINE FIX] Params cleaned for base Pipeline fit:", drop)
            return super().fit(X, y, **fit_params)

        # Marcamos que estamos entrando en modo "rank"
        self._in_rank_fit = True
        try:
            # Delegamos toda la lógica de meses/grupos a la función helper
            # Le pasamos "self" para que lo entrene in-place
            months = fit_params.get("months")
            if months is None:
                try:
                    months = pd.to_datetime(DATES_SERIES.loc[X.index]).dt.to_period("M")
                except Exception:
                    months = MONTHS_ALL.loc[X.index] if 'MONTHS_ALL' in globals() else None
            fit_with_rank_support(self, X, y, months, clone_first=False)
        finally:
            # Desmarcamos la bandera al salir
            self._in_rank_fit = False
        return self

print("✅ Clase RankPipeline actualizada con protección anti-recursión.")
# ===================================================================


✅ Clase RankPipeline actualizada con protección anti-recursión.


In [ ]:
# ================= TorchMLPRegressorSk =================
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin
import torch
import torch.nn as nn
import torch.optim as optim

class _MLP(nn.Module):
    def __init__(self, in_dim, hidden=128, depth=2, dropout=0.0):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, hidden), nn.ReLU(), nn.Dropout(dropout)]
            d = hidden
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

class TorchMLPRegressorSk(BaseEstimator, RegressorMixin):
    def __init__(self,
                 hidden=128, depth=2, dropout=0.0,
                 lr=1e-3, weight_decay=1e-4,
                 epochs=25, batch_size=1024,
                 patience=5, clip_grad=1.0,
                 device=None, random_state=42, verbose=0):
        self.hidden=hidden; self.depth=depth; self.dropout=dropout
        self.lr=lr; self.weight_decay=weight_decay
        self.epochs=epochs; self.batch_size=batch_size
        self.patience=patience; self.clip_grad=clip_grad
        self.device=device; self.random_state=random_state
        self.verbose=verbose
        self._fitted=False

    def _seed(self):
        torch.manual_seed(self.random_state)
        np.random.seed(self.random_state)

    def fit(self, X, y, months=None):
        self._seed()
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        in_dim = X.shape[1]

        dev = self.device or ("cuda" if torch.cuda.is_available() else "cpu")
        model = _MLP(in_dim, self.hidden, self.depth, self.dropout).to(dev)
        opt = optim.Adam(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        loss_fn = nn.MSELoss()

        X_t = torch.from_numpy(X)
        y_t = torch.from_numpy(y)
        ds = torch.utils.data.TensorDataset(X_t, y_t)
        dl = torch.utils.data.DataLoader(ds, batch_size=self.batch_size, shuffle=True)

        best_loss = float("inf"); no_improve = 0
        model.train()
        for epoch in range(self.epochs):
            run_loss = 0.0
            for xb, yb in dl:
                xb = xb.to(dev); yb = yb.to(dev)
                opt.zero_grad()
                pred = model(xb)
                loss = loss_fn(pred, yb)
                loss.backward()
                if self.clip_grad is not None:
                    nn.utils.clip_grad_norm_(model.parameters(), self.clip_grad)
                opt.step()
                run_loss += loss.item() * xb.size(0)
            epoch_loss = run_loss / len(ds)
            if self.verbose:
                print(f"[TorchMLP] epoch {epoch+1}/{self.epochs} loss={epoch_loss:.6f}")
            if epoch_loss + 1e-8 < best_loss:
                best_loss = epoch_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                no_improve = 0
            else:
                no_improve += 1
                if no_improve >= self.patience:
                    if self.verbose:
                        print("[TorchMLP] early stopping")
                    break
        # load best
        model.load_state_dict(best_state)
        model.eval()
        self.model_ = model.to(dev)
        self.dev_ = dev
        self._fitted = True
        return self

    def predict(self, X):
        assert self._fitted
        X = np.asarray(X, dtype=np.float32)
        with torch.no_grad():
            xb = torch.from_numpy(X).to(self.dev_)
            pred = self.model_(xb).cpu().numpy()
        return pred


In [ ]:
# --- Helpers para CatBoost wrapper tolerante a 'random_state' / 'random_seed' ---
def _cb_kwargs_base(FAST_MODE: bool):
    return dict(
        iterations=(50 if FAST_MODE else 600),
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        loss_function="YetiRank",  # estable en CPU
        task_type="CPU",
        verbose=False,
    )

def _mk_cb_ranker(seed: int, base_kwargs: dict):
    # 1) Intentar con random_state (estilo sklearn)
    try:
        return CatBoostRankerSk(**base_kwargs, random_state=seed)
    except TypeError:
        pass
    # 2) Intentar con random_seed (estilo CatBoost nativo)
    try:
        return CatBoostRankerSk(**base_kwargs, random_seed=seed)
    except TypeError:
        pass
    # 3) Sin semilla (último recurso)
    return CatBoostRankerSk(**base_kwargs)


In [ ]:
def _months_to_period_m(months):
    """Normaliza cualquier months (Period, datetime, str, ndarray) a Period[M]."""
    if pd.api.types.is_period_dtype(months):
        return pd.Series(months).dt.asfreq("M")
    m = pd.to_datetime(months, errors="coerce")
    return pd.Series(m).dt.to_period("M")


def _build_lgbm_groups_from_months(months) -> np.ndarray:
    """
    LightGBM Ranker requiere un vector 1D de tamaños de grupo (query group).
    Aquí lo derivamos de los meses: cada mes = un grupo.
    Salida: np.ndarray[int32] ya copiado (no Index, no Series).
    """
    m = _months_to_period_m(months)
    # cuenta por mes, ordenado cronológicamente
    counts = m.value_counts().sort_index()
    # MUY IMPORTANTE: forzar a ndarray con copia y dtype correcto
    group_arr = np.asarray(counts.values, dtype=np.int32).copy()
    return group_arr


In [ ]:
# ================================================================================
# BLOQUE COMPLETO DE CONSTRUCCIÓN DE MODELOS (GPU-aware, smoke-aware)
# ================================================================================

from sklearn.pipeline import Pipeline as _SkPipe
from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor, LGBMRanker

# CatBoost puede ser tu wrapper; si no está, lo marcamos
CatBoostRankerSk = globals().get("CatBoostRankerSk", None)
GKXEnsemble      = globals().get("GKXEnsemble", None)

# --- Parámetros ultraligeros para RF en smoke ---
RF_SMOKE_PARAMS = {
    "n_estimators": 10,
    "max_depth": 5,
    "n_jobs": 1,
    "random_state": 42,
}

# ===== 0) helpers CatBoost tolerantes a la firma =================================
def _cb_kwargs_base(FAST_MODE: bool):
    return dict(
        iterations=(50 if FAST_MODE else 600),
        learning_rate=0.05,
        depth=6,
        l2_leaf_reg=3.0,
        loss_function="YetiRank",  # estable en CPU
        task_type="CPU",
        verbose=False,
    )

def _mk_cb_ranker(seed: int, base_kwargs: dict):
    """Instancia el wrapper CatBoostRankerSk probando random_state y random_seed."""
    # 1) Estilo sklearn
    try:
        return CatBoostRankerSk(**base_kwargs, random_state=seed)
    except TypeError:
        pass
    # 2) Estilo CatBoost nativo
    try:
        return CatBoostRankerSk(**base_kwargs, random_seed=seed)
    except TypeError:
        pass
    # 3) Sin semilla si el wrapper no admite ninguno
    return CatBoostRankerSk(**base_kwargs)

# ===== 1) helpers varios =========================================================
def mkpipe(steps):
    return _SkPipe(steps)

def _lgbm_kwargs(GPU=False, FAST_MODE=False, N_JOBS=4):
    kwargs = {
        "random_state": 42,
        "n_estimators": 20 if FAST_MODE else 400,
        "n_jobs": N_JOBS,
        "feature_pre_filter": False,
        "min_split_gain": 0.0,
    }
    if GPU:
        kwargs.update({
            "device_type": "gpu",
            "gpu_platform_id": 0,
            "gpu_device_id": 0,
        })
    else:
        kwargs["device_type"] = "cpu"
    return kwargs

def _lgbm_ranker_kwargs(GPU=False, FAST_MODE=False, N_JOBS=4):
    kwargs = {
        "objective": "rank_xendcg",
        "random_state": 42,
        "n_estimators": 20 if FAST_MODE else 400,
        "n_jobs": N_JOBS,
        "feature_pre_filter": False,
        "min_split_gain": 0.0,
    }
    if GPU:
        kwargs.update({
            "device_type": "gpu",
            "gpu_platform_id": 0,
            "gpu_device_id": 0,
        })
    else:
        kwargs["device_type"] = "cpu"
    return kwargs

def _drop_none_models(models_all: dict):
    """Elimina entradas None en cada familia; borra familias vacías."""
    families = list(models_all.keys())
    for fam in families:
        models_all[fam] = {k: v for k, v in models_all[fam].items() if v is not None}
        if not models_all[fam]:
            del models_all[fam]
    return models_all

# ===== 2) función principal ======================================================
def build_models_and_search(
    FAST_MODE: bool = False,
    GPU_LGBM: bool = True,
    N_JOBS: int = 1,
    ALLOW_GKX: bool = True,
):
    """
    Construye MODELS_ALL y SEARCH_CFG usando la info global de la celda 0.
    - Respeta SMOKE_TEST.
    - Usa HAS_GPU si está disponible.
    - Fuerza GPU en XGBoost / LGBM / CatBoost cuando existe.
    """
    # -------- flags globales ----------
    SMOKE_TEST    = bool(globals().get("SMOKE_TEST", False))
    HAS_GPU       = bool(globals().get("HAS_GPU", False))
    USE_GPU       = bool(globals().get("USE_GPU", False))
    GLOBAL_SEED   = int(globals().get("GLOBAL_SEED", 42))

    ensure_df     = globals().get("ensure_df")
    sector_demean = globals().get("sector_demean")
    winsor_cs     = globals().get("winsor_cs")
    zscore_cs     = globals().get("zscore_cs")
    nan_fix       = globals().get("nan_fix")
    RFCAsRanker   = globals().get("RFCAsRanker")
    RankPipeline  = globals().get("RankPipeline")  # asumimos que existe

    # si estamos en smoke, hacemos todo más chico
    EFFECTIVE_FAST = FAST_MODE or SMOKE_TEST

    # XGBoost: detectar GPU desde la celda 0
    use_xgb_gpu = bool(HAS_GPU and USE_GPU)
    xgb_kwargs = dict(
        random_state=GLOBAL_SEED,
        n_estimators=(20 if EFFECTIVE_FAST else 400),
        verbosity=0,
        n_jobs=4,
        tree_method=('gpu_hist' if use_xgb_gpu else 'hist'),
        predictor=('auto' if use_xgb_gpu else 'cpu_predictor'),
        gpu_id=0,
        subsample=0.8,
        colsample_bytree=0.8,
    )
    print(f"[Config] XGBoost → tree_method={xgb_kwargs['tree_method']} | "
          f"predictor={xgb_kwargs['predictor']} | GPU={use_xgb_gpu}")

    # LightGBM
    use_lgbm_gpu = bool(HAS_GPU and GPU_LGBM and USE_GPU)
    lgbm_kwargs = _lgbm_kwargs(GPU=use_lgbm_gpu, FAST_MODE=EFFECTIVE_FAST, N_JOBS=4)
    lgbm_ranker_kwargs = _lgbm_ranker_kwargs(GPU=use_lgbm_gpu, FAST_MODE=EFFECTIVE_FAST, N_JOBS=4)
    print(f"[Config] LightGBM → device_type={lgbm_kwargs.get('device_type', 'cpu')}")

    # CatBoost (base kwargs y fábrica tolerante a firma)
    cb_base = _cb_kwargs_base(FAST_MODE=EFFECTIVE_FAST)
    print(f"[Config] CatBoost → task_type={cb_base['task_type']}")

    # --- Construcción de MODELS_ALL ---
    MODELS_ALL = {
        "Lineales": {
            "OLS (Linear Regression)": mkpipe([
                ("ensure_df", ensure_df),
                ("sectordemean", sector_demean),
                ("winsor_cs", winsor_cs),
                ("cs_z", zscore_cs),
                ("nanfix", nan_fix),
                ("model", LinearRegression()),
            ]),
            "ElasticNet": mkpipe([
                ("ensure_df", ensure_df),
                ("sectordemean", sector_demean),
                ("winsor_cs", winsor_cs),
                ("cs_z", zscore_cs),
                ("nanfix", nan_fix),
                ("model", ElasticNetCV(
                    l1_ratio=[.1, .5, .9, .99],
                    cv=5,
                    fit_intercept=False,
                    max_iter=1000 if EFFECTIVE_FAST else 5000,
                    n_jobs=N_JOBS,
                    random_state=GLOBAL_SEED,
                )),
            ]),
        },
        "Boosting": {
            "Random Forest": mkpipe([
                ("ensure_df", ensure_df),
                ("sectordemean", sector_demean),
                ("winsor_cs", winsor_cs),
                ("cs_z", zscore_cs),
                ("nanfix", nan_fix),
                ("model",
                 RandomForestRegressor(**RF_SMOKE_PARAMS) if EFFECTIVE_FAST else
                 RandomForestRegressor(
                     random_state=GLOBAL_SEED,
                     n_jobs=N_JOBS,
                     n_estimators=200,
                     max_depth=None,
                     min_samples_split=5,
                     min_samples_leaf=2,
                     max_features="sqrt",
                 )),
            ]),
            "Random Forest (Ranker-bins)": (
                RankPipeline([
                    ("ensure_df", ensure_df),
                    ("sectordemean", sector_demean),
                    ("winsor_cs", winsor_cs),
                    ("cs_z", zscore_cs),
                    ("nanfix", nan_fix),
                    ("model", RFCAsRanker(
                        n_estimators=(20 if EFFECTIVE_FAST else 500),
                        n_jobs=N_JOBS,
                    )),
                ]) if (RankPipeline is not None and RFCAsRanker is not None) else None
            ),
            "XGBoost": mkpipe([
                ("ensure_df", ensure_df),
                ("sectordemean", sector_demean),
                ("winsor_cs", winsor_cs),
                ("cs_z", zscore_cs),
                ("nanfix", nan_fix),
                ("model", XGBRegressor(**xgb_kwargs)),
            ]),
            "LightGBM": mkpipe([
                ("ensure_df", ensure_df),
                ("sectordemean", sector_demean),
                ("winsor_cs", winsor_cs),
                ("cs_z", zscore_cs),
                ("nanfix", nan_fix),
                ("model", LGBMRegressor(**lgbm_kwargs)),
            ]),
            "LightGBM (Ranker)": (
                RankPipeline([
                    ("ensure_df", ensure_df),
                    ("sectordemean", sector_demean),
                    ("winsor_cs", winsor_cs),
                    ("cs_z", zscore_cs),
                    ("nanfix", nan_fix),
                    ("model", LGBMRanker(**lgbm_ranker_kwargs)),
                ]) if RankPipeline is not None else None
            ),
            "CatBoost (Ranker)": (
                RankPipeline([
                    ("ensure_df", ensure_df),
                    ("sectordemean", sector_demean),
                    ("winsor_cs", winsor_cs),
                    ("cs_z", zscore_cs),
                    ("nanfix", nan_fix),
                    ("model", _mk_cb_ranker(GLOBAL_SEED, cb_base)),
                ]) if (RankPipeline is not None and CatBoostRankerSk is not None) else None
            ),
        },
    }

    # --- GKX opcional, con fallback a CPU ---
    if ALLOW_GKX and GKXEnsemble is not None:
        use_cuda = bool(HAS_GPU and USE_GPU)
        device = "cuda" if use_cuda else "cpu"
        n_ens = (3 if EFFECTIVE_FAST else 5)
        try:
            gkx_model = GKXEnsemble(
                n_ens=n_ens,
                hidden=(64, 32, 16),
                dropout=0.2,
                l2=1e-4,
                lr=1e-3,
                batch=256,
                epochs=(10 if EFFECTIVE_FAST else 60),
                patience=10,
                verbose=0,
                random_state=GLOBAL_SEED,
                device=device,  # puede no ser soportado
            )
        except TypeError:
            gkx_model = GKXEnsemble(
                n_ens=n_ens,
                hidden=(64, 32, 16),
                dropout=0.2,
                l2=1e-4,
                lr=1e-3,
                batch=256,
                epochs=(10 if EFFECTIVE_FAST else 60),
                patience=10,
                verbose=0,
                random_state=GLOBAL_SEED,
            )
        print(f"[Config] GKX → device={device} | n_ens={n_ens}")
        MODELS_ALL["Redes neuronales"] = {
            "GKX NN (64-32-16)": mkpipe([
                ("ensure_df", ensure_df),
                ("sectordemean", sector_demean),
                ("winsor_cs", winsor_cs),
                ("cs_z", zscore_cs),
                ("nanfix", nan_fix),
                ("model", gkx_model),
            ])
        }

    # --- Limpieza de entradas None (p. ej., si falta algún wrapper) ---
    MODELS_ALL = _drop_none_models(MODELS_ALL)

    # --- grids ---
    SEARCH_CFG = {
        "Random Forest": {
            "model__n_estimators": [200, 300, 400],
            "model__max_depth": [10, 12, 14, 16],
            "model__min_samples_leaf": [5, 10, 20],
            "model__max_features": ["sqrt", 0.4, 0.5, 0.6],
            "model__max_samples": [0.7, 0.8, 0.9],
            "model__n_jobs": [1],
        },
        "Random Forest (Ranker-bins)": {
            "model__max_depth": [10, 12, 14],
            "model__max_samples": [0.7, 0.8],
            "model__n_jobs": [1],
        },
        "XGBoost": {
            "model__n_estimators": [600, 800, 1000, 1200],
            "model__max_depth": [4, 5, 6, 7],
            "model__learning_rate": [0.01, 0.03, 0.05],
            "model__subsample": [0.7, 0.85, 1.0],
            "model__colsample_bytree": [0.7, 0.85, 1.0],
        },
        "LightGBM": {
            "model__learning_rate": [0.01, 0.03, 0.05, 0.10],
            "model__num_leaves": [31, 63, 127],
            "model__min_child_samples": [20, 35, 50, 60],
            "model__feature_fraction": [0.70, 0.85, 0.95],
            "model__bagging_fraction": [0.70, 0.85, 0.95],
            "model__lambda_l2": [0.0, 1.0, 5.0, 10.0],
        },
        "LightGBM (Ranker)": {
            "model__learning_rate": [0.01, 0.03, 0.05],
            "model__num_leaves": [31, 63, 127],
        },
        "CatBoost (Ranker)": {
            "model__iterations": [800, 1000, 1200],
            "model__depth": [5, 6, 7, 8],
            "model__learning_rate": [0.03, 0.05, 0.06],
            "model__l2_leaf_reg": [1.0, 3.0, 5.0],
        },
    }

    return MODELS_ALL, SEARCH_CFG


In [ ]:
# ================================================================================
# CONTEXT MANAGER PARA CONTROL DE HILOS
# ================================================================================
from contextlib import contextmanager
import os

@contextmanager
def no_oversubscription():
    """Context manager para limitar temporalmente los hilos de CPU a 1."""
    old_omp = os.environ.get("OMP_NUM_THREADS", None)
    os.environ["OMP_NUM_THREADS"] = "1"
    try:
        yield
    finally:
        # Restaura el valor original solo si existía
        if old_omp is None:
            os.environ.pop("OMP_NUM_THREADS", None)
        else:
            os.environ["OMP_NUM_THREADS"] = old_omp

In [ ]:
# === Helpers LightGBM: AUTO + fallback seguro ===
import os

ACCEL_PREF = os.getenv("LGBM_ACCEL", "auto")  # "auto" | "gpu" | "cpu"

CPU_ENFORCE = {
    "model__device_type": "cpu",
    "model__gpu_platform_id": -1,
    "model__gpu_device_id": -1,
    "model__n_jobs":  max(1, (os.cpu_count() or 2)//2),
    "model__num_threads": max(1, (os.cpu_count() or 2)//2),
}
GPU_ENFORCE = {
    "model__device_type": "gpu",
    "model__gpu_platform_id": int(os.getenv("LGBM_GPU_PLATFORM_ID", 0)),
    "model__gpu_device_id":   int(os.getenv("LGBM_GPU_DEVICE_ID", 0)),
}

def _strip_accel_keys(d: dict) -> dict:
    d = dict(d or {})
    for k in ["model__device","model__device_type",
              "model__gpu_platform_id","model__gpu_device_id",
              "model__n_jobs","model__num_threads"]:
        d.pop(k, None)
    return d

def prepare_lgbm_params(base: dict, accel: str) -> dict:
    p = _strip_accel_keys(base)
    p.update(GPU_ENFORCE if accel=="gpu" else CPU_ENFORCE)
    return p

def lgbm_gpu_is_usable() -> bool:
    import lightgbm as lgb, numpy as np
    X = np.random.rand(64, 10); y = (np.random.rand(64) > 0.5).astype(int)
    d = lgb.Dataset(X, label=y, free_raw_data=False)
    try:
        lgb.train({"objective":"binary","num_leaves":2,"min_data_in_bin":1,"device_type":"gpu"}, d, num_boost_round=1)
        return True
    except Exception:
        return False

def choose_accel_for_lgbm(n_rows: int, n_cols: int, pref: str="auto") -> str:
    if pref in ("cpu","gpu"):
        return "gpu" if (pref=="gpu" and lgbm_gpu_is_usable()) else "cpu"
    if (n_rows >= 200_000) or (n_rows >= 100_000 and n_cols >= 100):
        return "gpu" if lgbm_gpu_is_usable() else "cpu"
    return "cpu"

In [ ]:
# ============================================================
# BLOQUE 11.z · Size buckets por mes (sin fuga de información)
#   Usa sólo la info disponible en cada MES.
#   Trabaja sobre df_sorted, que está 1:1 con df_model/X/y
# ============================================================

# 1) Mes como Period[M]
df_sorted['Mes'] = pd.to_datetime(df_sorted['Fecha']).dt.to_period('M')

# 2) Función de bucket por mes (Small/Mid/Large)
def _bucket_size_mensual(g, col='Cap de mercado', q=3, labels=('Small','Mid','Large')):
    s = g[col].astype(float)
    # si hay muy pocos nombres, devuelve NA (evita cortes inestables)
    if s.notna().sum() < q:
        return pd.Series(pd.NA, index=g.index, dtype='object')
    r = s.rank(method='first')  # rompe empates antes de qcut
    try:
        return pd.qcut(r, q=q, labels=labels)
    except ValueError:
        return pd.Series(pd.NA, index=g.index, dtype='object')

# 3) Asignar bucket dentro de cada mes
df_sorted['size_bucket'] = (
    df_sorted.groupby('Mes', group_keys=False)
             .apply(_bucket_size_mensual)
             .astype('category')
)

In [ ]:
# =================================================================================
# === MÉTRICA DE TUNING: IC NEUTRALIZADO (VERSIÓN RÁPIDA) ===
# =================================================================================
# Precomputar códigos para acelerar el scorer (se hace DESPUÉS de crear size_bucket)
print("\nPre-computando códigos para el scorer rápido...")
MONTH_CODES  = pd.Categorical(pd.to_datetime(DATES_SERIES).dt.to_period('M')).codes.astype('int32')
SECTOR_CODES = (pd.Categorical(df_sorted['Sector_GICS']).codes.astype('int32')
                if 'Sector_GICS' in df_sorted.columns else np.full(len(MONTH_CODES), -1, np.int32))
SIZE_CODES   = (pd.Categorical(df_sorted['size_bucket'].astype(str)).codes.astype('int32')
                if 'size_bucket' in df_sorted.columns else np.full(len(MONTH_CODES), -1, np.int32))
print("Códigos listos.")

from scipy.stats import rankdata

def ic_scorer_neut_fast(estimator, X_val, y_val):
    import numpy as np
    idx = X_val.index.to_numpy()
    p   = estimator.predict(X_val).astype('float32')
    yv  = y_val.to_numpy(dtype='float32')

    m = MONTH_CODES[idx]; s = SECTOR_CODES[idx]; z = SIZE_CODES[idx]

    # Neutralización (z-score) por grupo (mes, sector, size)
    keys = (m.astype(np.int64)*10000 + (s.astype(np.int64)+1)*100 + (z.astype(np.int64)+1))
    for k in np.unique(keys):
        g = (keys == k)
        if g.sum() >= 3:
            mu = p[g].mean(); sd = p[g].std(ddof=0)
            p[g] = (p[g] - mu) / (sd + 1e-9)

    # IC por mes (usando rankdata para un Spearman más clásico)
    ic_vals = []
    for mm in np.unique(m):
        g = (m == mm)
        if g.sum() >= 5 and np.unique(p[g]).size > 1:
            rp = rankdata(p[g], method='average').astype('float32')
            ry = rankdata(yv[g], method='average').astype('float32')
            corr = np.corrcoef(rp, ry)[0, 1]
            if np.isfinite(corr): ic_vals.append(float(corr))
    return float(np.mean(ic_vals)) if ic_vals else 0.0

spearman_scorer = ic_scorer_neut_fast
print("✅ Scorer de tuning: IC neutralizado (versión rápida con rankdata).")

# Ajustar LABEL_GAIN para dar más peso a los extremos
if 'N_BINS' not in globals(): N_BINS = 10
LABEL_GAIN = [int(round((i/(N_BINS-1))**1.8 * 10)) for i in range(N_BINS)]
# =================================================================================


Pre-computando códigos para el scorer rápido...
Códigos listos.
✅ Scorer de tuning: IC neutralizado (versión rápida con rankdata).


In [ ]:
# ===============================================================================
# Baselines de señal — IC mensual (compat sec_score_z_safe / sec_score_z_lag1)
# ===============================================================================

from scipy.stats import spearmanr
import pandas as pd

print("\n### Baselines de señal — IC mensual ###")

# Frame auxiliar perfectamente alineado con df_model/df_sorted
#aux = pd.DataFrame({
#    'Fecha'   : pd.to_datetime(df_model['Fecha']).values,
#    'Empresa' : df_model['Empresa'].values,
#    'target'  : y.values,
#    'P_Share' : df_sorted['P_Share'].values
#})

# ---- Baseline A: momentum 12m lag1 (stock-level) ----
# Calcula en la serie mensual completa (df_sorted) y luego alinea a aux
mom12_full = df_sorted.groupby('Empresa', group_keys=False)['P_Share'].pct_change(12)
mom12_lag1 = mom12_full.groupby(df_sorted['Empresa']).shift(1)

# df_sorted y aux están 1:1 por construcción, así que podemos asignar los valores directamente
aux['mom12_lag1'] = mom12_lag1.values

ic_mom = (
    aux.dropna(subset=['mom12_lag1'])
       .groupby(aux['Fecha'].dt.to_period('M'), sort=True)
       .apply(lambda g: spearmanr(g['target'], g['mom12_lag1'])[0])
       .dropna()
)

print(f"Baseline (mom12_lag1) — IC medio: {float(ic_mom.mean()):.4f} | n_meses={len(ic_mom)}")

# ---- Baseline B: señal sectorial segura (ya en t-1 por construcción) ----
# Soporta ambos nombres para atrás/adelante
sec_col = ('sec_score_z_safe' if 'sec_score_z_safe' in df_sorted.columns else
           ('sec_score_z_lag1' if 'sec_score_z_lag1' in df_sorted.columns else None))

if sec_col:
    aux[sec_col] = df_sorted[sec_col].values
    ic_sec = (
        aux.dropna(subset=[sec_col])
           .groupby(aux['Fecha'].dt.to_period('M'), sort=True)
           .apply(lambda g: spearmanr(g['target'], g[sec_col])[0])
           .dropna()
    )
    print(f"Baseline (sector-mom: {sec_col}) — IC medio: {float(ic_sec.mean()):.4f} | n_meses={len(ic_sec)}")
else:
    print("Baseline sectorial: señal no disponible en df_sorted (ni 'sec_score_z_safe' ni 'sec_score_z_lag1').")



### Baselines de señal — IC mensual ###
Baseline (mom12_lag1) — IC medio: -0.0177 | n_meses=104
Baseline (sector-mom: sec_score_z_safe) — IC medio: -0.0018 | n_meses=104


In [ ]:
# ======================================================================
# GKXEnsemble — Ensamble simple de MLPs con interfaz scikit-learn
# ======================================================================
from sklearn.base import BaseEstimator, RegressorMixin
import numpy as np

# Tomar los símbolos ya importados en el Bloque 3
tf     = globals().get('tf', None)
keras  = globals().get('keras', None)
layers = keras.layers if keras is not None else None

class GKXEnsemble(BaseEstimator):
    def __init__(self,
                 n_ens=5,
                 hidden=(64, 32, 16),
                 dropout=0.2,
                 l2=1e-4,
                 lr=1e-3,
                 batch=256,
                 epochs=60,
                 patience=10,
                 val_frac=0.2,
                 verbose=0,
                 random_state=42):
        self.n_ens=n_ens; self.hidden=hidden; self.dropout=dropout
        self.l2=l2; self.lr=lr; self.batch=batch; self.epochs=epochs
        self.patience=patience; self.val_frac=val_frac; self.verbose=verbose
        self.random_state=random_state
        self._models=[]

    def _build_one(self, n_features, seed):
        if keras is None or layers is None:
            raise RuntimeError("GKXEnsemble: Keras no está disponible (revisá Bloque 3).")
        keras.utils.set_random_seed(seed)
        reg = keras.regularizers.l2(self.l2) if self.l2 else None
        x = inputs = keras.Input(shape=(n_features,))
        for h in self.hidden:
            x = layers.Dense(h, activation="relu", kernel_regularizer=reg)(x)
            x = layers.BatchNormalization()(x)
            if self.dropout and self.dropout>0:
                x = layers.Dropout(self.dropout)(x)
        outputs = layers.Dense(1)(x)
        m = keras.Model(inputs, outputs)
        m.compile(optimizer=keras.optimizers.Adam(self.lr), loss="mse")
        return m

    def fit(self, X, y):
        if tf is None:
            raise RuntimeError("TensorFlow/Keras no disponible.")
        X = np.asarray(X, np.float32)
        y = np.asarray(y, np.float32).reshape(-1,1)
        n = X.shape[0]; n_val = int(max(1, np.floor(self.val_frac*n)))
        X_tr, X_val = (X[:-n_val], X[-n_val:]) if n_val>0 else (X, X[:0])
        y_tr, y_val = (y[:-n_val], y[-n_val:]) if n_val>0 else (y, y[:0])

        self._models=[]; base_seed=int(self.random_state)
        cbs=[]
        if n_val>0:
            cbs=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=self.patience,
                                               restore_best_weights=True, verbose=0),
                 keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=max(3,self.patience//2),
                                                   factor=0.5, min_lr=1e-5, verbose=0)]
        for i in range(self.n_ens):
            m = self._build_one(X.shape[1], seed=base_seed+i)
            m.fit(X_tr, y_tr, validation_data=(X_val, y_val) if n_val>0 else None,
                  batch_size=self.batch, epochs=self.epochs, verbose=self.verbose, callbacks=cbs)
            self._models.append(m)
        return self

    def predict(self, X):
        X = np.asarray(X, np.float32)
        if not self._models:
            return np.zeros(X.shape[0], dtype=np.float32)
        preds = [m.predict(X, verbose=0).reshape(-1) for m in self._models]
        return np.mean(preds, axis=0).astype(np.float32)

In [ ]:
# ======================================================================
# Wrapper scikit para CatBoostRanker
# ======================================================================
class CatBoostRankerSk(BaseEstimator):
    def __init__(self,
                 iterations=600, learning_rate=0.05, depth=6,
                 l2_leaf_reg=3.0, random_state=42,
                 loss_function='YetiRank',  # o 'PairLogit'
                 task_type='CPU', verbose=False):
        self.iterations=iterations; self.learning_rate=learning_rate
        self.depth=depth; self.l2_leaf_reg=l2_leaf_reg
        self.random_state=random_state; self.loss_function=loss_function
        self.task_type=task_type; self.verbose=verbose
        self._mdl=None

    def fit(self, X, y, group=None, eval_set=None):
        self._mdl = CatBoostRanker(
            iterations=self.iterations, learning_rate=self.learning_rate,
            depth=self.depth, l2_leaf_reg=self.l2_leaf_reg,
            random_state=self.random_state, loss_function=self.loss_function,
            task_type=self.task_type, verbose=self.verbose
        )
        train_pool = Pool(X, y, group_id=group)
        eval_pool  = None
        if isinstance(eval_set, tuple) and len(eval_set)==3:
            Xev, yev, gev = eval_set
            eval_pool = Pool(Xev, yev, group_id=gev)
        self._mdl.fit(train_pool, eval_set=eval_pool, verbose=self.verbose)
        return self

    def predict(self, X):
        return self._mdl.predict(X)

# ======================================================================
# Wrapper para usar RandomForestClassifier como Ranker por bins
# ======================================================================

def make_rel_labels(y_cont: pd.Series, months, n_bins: int = 10) -> np.ndarray:
    """
    Etiquetas 0..n_bins-1 por mes a partir del target continuo.
    FIXED: acepta Period[M], Timestamp, string, ndarray y Series.
    """
    y = pd.to_numeric(y_cont, errors='coerce')

    # 1) Normalizar meses
    if "_as_period_m" in globals():
        m = _as_period_m(months)
    else:
        if pd.api.types.is_period_dtype(months):
            m = pd.Series(months).dt.asfreq("M")
        else:
            m = pd.to_datetime(months, errors="coerce")
            m = pd.Series(m).dt.to_period("M")

    labels = np.full(len(y), np.nan, dtype="float32")

    # 2) Rank por mes
    for mm, gidx in m.groupby(m).groups.items():
        gidx = np.asarray(gidx)
        yy = y.iloc[gidx]
        # masa crítica
        if yy.notna().sum() < max(5, n_bins):
            continue

        r = yy.rank(method="first")

        try:
            bins = pd.qcut(r, q=n_bins, labels=False, duplicates="drop")
            labels[gidx] = bins.astype("float32")
        except ValueError:
            # fallback 5 bins
            try:
                bins = pd.qcut(r, q=5, labels=False, duplicates="drop")
                labels[gidx] = (bins * (n_bins / 5.0)).astype("float32")
            except ValueError:
                # sin etiquetar, se rellenan después
                pass

    # 3) Relleno seguro + entero
    nan_mask = ~np.isfinite(labels)
    if nan_mask.any():
        if np.isfinite(labels).any():
            fill_val = float(np.nanmedian(labels[np.isfinite(labels)]))
        else:
            fill_val = 0.0
        labels[nan_mask] = fill_val

    labels = np.rint(labels)
    return labels.astype("int32")

class RFCAsRanker(BaseEstimator):
    def __init__(self, n_estimators=400, max_depth=None, random_state=42, n_jobs=4, max_samples=None):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.random_state = random_state
        self.n_jobs = n_jobs
        self.max_samples = max_samples  # <-- Nuevo parámetro
        self._clf = None
        self._labels_ = None

    def fit(self, X, y, months=None):
        assert months is not None, "RFCAsRanker requiere 'months' en fit(...)"
        y_lab = make_rel_labels(pd.Series(y).reset_index(drop=True),
                                pd.Series(months).reset_index(drop=True),
                                n_bins=N_BINS)
        self._labels_ = np.sort(np.unique(y_lab))

        self._clf = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            random_state=self.random_state,
            n_jobs=self.n_jobs,
            class_weight='balanced_subsample',
            bootstrap=True,
            max_samples=self.max_samples  # <-- Pasar el parámetro
        )
        self._clf.fit(X, y_lab)
        return self

    def predict(self, X):
        proba = self._clf.predict_proba(X)
        exp_rank = (proba * self._labels_[None, :]).sum(axis=1)
        return exp_rank.astype(np.float32)

In [ ]:
# ===============================================================================
# BLOQUE · Transformadores Cross-Sectional (versión corregida INDEX-SAFE)
#   - No resetean índices en transform()
#   - Alinean por etiquetas del subset (train/test) para evitar misalign
#   - Devuelven np.ndarray float32 (compat pipelines)
# ===============================================================================

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import FunctionTransformer

class CrossSectionalWinsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, dates_full, freq='M', lower=0.01, upper=0.01, min_group=5):
        self.dates_full = dates_full
        self.freq = freq
        self.lower = float(lower); self.upper = float(upper); self.min_group = int(min_group)

    def fit(self, X, y=None):
        if not hasattr(self, 'per_full_'):
            s = pd.to_datetime(pd.Series(self.dates_full), errors='coerce')
            self.per_full_ = s.dt.to_period(self.freq)
        return self

    def transform(self, X, y=None):
        Xdf = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        if Xdf.empty: return Xdf
        idx = Xdf.index
        try:
            per_sub = self.per_full_.loc[idx]
        except KeyError:
            pos = np.clip(idx.to_numpy(), 0, len(self.per_full_)-1)
            per_sub = self.per_full_.iloc[pos]; per_sub.index = idx
        valid = per_sub.notna()
        if not valid.any(): return Xdf.fillna(0.0)

        X_val = Xdf.to_numpy(dtype='float32'); X_out = X_val.copy()
        per_arr = per_sub.to_numpy()
        for p in pd.unique(per_sub[valid]):
            m = (per_arr == p)
            if m.sum() < self.min_group: continue
            dm = X_out[m, :]
            ql = np.nanquantile(dm, self.lower, axis=0)
            qh = np.nanquantile(dm, 1.0 - self.upper, axis=0)
            X_out[m, :] = np.clip(dm, ql, qh)
        return (pd.DataFrame(X_out, index=idx, columns=Xdf.columns)
                  .replace([np.inf, -np.inf], np.nan).fillna(0.0))

# === Z-score cross-sectional por período (index-safe) ===
from sklearn.base import BaseEstimator, TransformerMixin

class CrossSectionalZ(BaseEstimator, TransformerMixin):
    def __init__(self, dates_full, freq='M', ddof=0, min_group=5, fill_invalid='zero'):
        self.dates_full = dates_full
        self.freq = freq
        self.ddof = int(ddof)
        self.min_group = int(min_group)
        self.fill_invalid = str(fill_invalid)  # 'zero' o 'nan'

    def fit(self, X, y=None):
        s = pd.to_datetime(pd.Series(self.dates_full), errors='coerce')
        self.per_full_ = s.dt.to_period(self.freq)
        return self

    def transform(self, X, y=None):
        Xdf = X if isinstance(X, pd.DataFrame) else pd.DataFrame(X)
        if Xdf.empty:
            return Xdf
        idx = Xdf.index
        # Mapear períodos del subset por etiqueta de índice (index-safe)
        try:
            per_sub = self.per_full_.loc[idx]
        except KeyError:
            pos = np.clip(np.asarray(idx), 0, len(self.per_full_) - 1)
            per_sub = self.per_full_.iloc[pos]
            per_sub.index = idx

        valid = per_sub.notna()
        if not valid.any():
            return Xdf.fillna(0.0)

        X_val = Xdf.to_numpy(dtype='float32')
        X_out = np.empty_like(X_val, dtype='float32')
        X_out[:] = np.nan

        per_arr = per_sub.to_numpy()
        for p in pd.unique(per_sub[valid]):
            m = (per_arr == p)
            if m.sum() < self.min_group:
                continue
            dm = X_val[m, :]
            mu = np.nanmean(dm, axis=0)
            sd = np.nanstd(dm, axis=0, ddof=self.ddof)
            sd = np.where(sd <= 0.0, np.nan, sd)
            X_out[m, :] = (dm - mu) / sd

        Xout = pd.DataFrame(X_out, index=idx, columns=Xdf.columns)
        if self.fill_invalid == 'zero':
            Xout = Xout.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        else:
            Xout = Xout.replace([np.inf, -np.inf], np.nan)
        return Xout.astype('float32')


In [ ]:
# ===============================================================================
# BLOQUE 11.y · Re-instanciar transformadores
# ===============================================================================
print("\n⏩  BLOQUE 11.y – Re-instanciando transformadores con orden corregido...")
from sklearn.base import BaseEstimator, TransformerMixin

# --- PASO 1: DEFINICIÓN DE CLASES ---
class SafeSkipColsWrapper(BaseEstimator, TransformerMixin):
    def __init__(self, base_transformer, skip_cols=None):
        self.base_transformer = base_transformer
        self.skip_cols = skip_cols
    def fit(self, X, y=None):
        Xdf = pd.DataFrame(X)
        self.cols_ = list(Xdf.columns)
        skip = set(self.skip_cols) if self.skip_cols is not None else set()
        self.keep_ = [c for c in self.cols_ if c not in skip]
        if self.keep_: self.base_transformer.fit(Xdf[self.keep_], y)
        return self
    def transform(self, X):
        Xdf = pd.DataFrame(X, columns=self.cols_)
        if self.keep_:
            Xt_keep = self.base_transformer.transform(Xdf[self.keep_])
            Xt = pd.DataFrame(Xt_keep, columns=self.keep_, index=Xdf.index)
        else:
            Xt = pd.DataFrame(index=Xdf.index)
        for c in self.cols_:
            if c not in (self.keep_ or []):
                Xt[c] = pd.to_numeric(Xdf[c], errors='coerce').astype('float32')
        Xt = Xt[self.cols_].replace([np.inf, -np.inf], 0.0).fillna(0.0).astype('float32')
        return Xt # Devuelve DataFrame con índice

# ===============================================================================
# BLOQUE · Transformadores Cross-Sectional (versión corregida CLONE-SAFE)
# ===============================================================================
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd
import numpy as np

# --- CLASE INTERNA "PURA" ---
# No se usa directamente en el pipeline, pero es llamada por el Wrapper.
# Sigue las reglas de scikit-learn: no modifica parámetros y no acepta datos en __init__.
class _SectorDemeanCS_core(BaseEstimator, TransformerMixin):
    def __init__(self, skip_cols=None):
        self.skip_cols = skip_cols

    def fit(self, X, y=None):
        # Guarda los nombres de las columnas que ve durante el fit
        self.feature_names_in_ = list(pd.DataFrame(X).columns)
        return self

    def transform(self, X, sector_series, dates_series):
        Xdf = pd.DataFrame(X, columns=self.feature_names_in_).astype('float32')
        if Xdf.empty:
            return Xdf

        _skip_cols = set(self.skip_cols) if self.skip_cols is not None else set()

        # Alineación por índice garantizada
        s = sector_series.loc[Xdf.index]
        m = pd.to_datetime(dates_series.loc[Xdf.index]).dt.to_period('M')

        keep = [c for c in Xdf.columns if c not in _skip_cols]
        Xout = Xdf.copy()

        if keep:
            keys = pd.MultiIndex.from_arrays([m, s], names=['Mes','Sector'])
            grp_mean = Xdf[keep].groupby(keys, group_keys=False).transform('mean')
            Xout[keep] = Xdf[keep] - grp_mean

        return Xout.replace([np.inf, -np.inf], 0.0).fillna(0.0)

# --- CLASE WRAPPER "PÚBLICA" ---
# Esta es la clase que SÍ usarás al construir tu pipeline.
# Su trabajo es guardar los datos (Series) e "inyectarlos" en la clase pura.
class SectorDemeanCS(BaseEstimator, TransformerMixin):
    def __init__(self, skip_cols=None, sector_series=None, dates_series=None):
        self.skip_cols = skip_cols
        self.sector_series = sector_series
        self.dates_series = dates_series

    def fit(self, X, y=None):
        # Crea e "instruye" al transformador puro
        self.transformer_ = _SectorDemeanCS_core(skip_cols=self.skip_cols)
        self.transformer_.fit(X, y)
        return self

    def transform(self, X):
        # Llama al transform del objeto puro, inyectando los datos necesarios
        return self.transformer_.transform(
            X,
            sector_series=self.sector_series,
            dates_series=self.dates_series
        )

# ===============================================================================
# --- FUNCIÓN PARA FORZAR DISPOSITIVOS (CPU/GPU) EN MODELOS ---
# ===============================================================================
import lightgbm as lgb
import xgboost as xgb
from sklearn.pipeline import Pipeline

def force_devices(models_dict, use_gpu=True):
    """
    Itera sobre un diccionario de modelos y fuerza a LightGBM a usar CPU
    y a XGBoost/CatBoost a usar GPU (si use_gpu=True y está disponible).
    """
    print(f"\n⚡ Forzando configuración de dispositivos (GPU: {use_gpu})...")
    fixed_models = {}
    for fam, models in models_dict.items():
        fixed_models[fam] = {}
        for name, pipe in models.items():
            p = _sk_clone(pipe) # Trabajar sobre una copia para evitar efectos secundarios

            # Determinar si el paso 'model' existe en el pipeline
            is_pipeline = isinstance(p, Pipeline)
            step_name = 'model' if is_pipeline and 'model' in p.named_steps else None

            if step_name:
                est = p.named_steps[step_name]

                # --- LightGBM siempre a CPU ---
                if isinstance(est, (lgb.LGBMRegressor, lgb.LGBMRanker)):
                    print(f"   -> Configurando '{name}' para CPU.")
                    p.set_params(**{
                        f"{step_name}__device_type": "cpu",
                        f"{step_name}__device": "cpu", # Alias para compatibilidad
                        f"{step_name}__n_jobs": 1
                    })

                # --- XGBoost y CatBoost a GPU si es posible ---
                if use_gpu:
                    if isinstance(est, (xgb.XGBRegressor, xgb.XGBRanker)):
                        print(f"   -> Configurando '{name}' para GPU.")
                        p.set_params(**{
                            f"{step_name}__tree_method": "gpu_hist",
                            f"{step_name}__predictor": "gpu_predictor"
                        })
                    if 'CatBoost' in est.__class__.__name__:
                         print(f"   -> Configurando '{name}' para GPU.")
                         p.set_params(**{f"{step_name}__task_type": "GPU"})
                else: # Forzar CPU si no se usa GPU
                     if isinstance(est, (xgb.XGBRegressor, xgb.XGBRanker)):
                        print(f"   -> Configurando '{name}' para CPU.")
                        p.set_params(**{
                            f"{step_name}__tree_method": "hist",
                            f"{step_name}__predictor": "cpu_predictor"
                        })
                     if 'CatBoost' in est.__class__.__name__:
                         print(f"   -> Configurando '{name}' para CPU.")
                         p.set_params(**{f"{step_name}__task_type": "CPU"})

            fixed_models[fam][name] = p

    return fixed_models

# --- PASO 2: CONSTRUIR `skip_cols` ---
try: macro_bases = list(features_macro)
except NameError: macro_bases = [c.replace('_lag1', '') for c in X.columns if c.startswith('dif_') and c.endswith('_lag1')]
macro_lag_cols    = [f"{c}_lag1" for c in macro_bases if f"{c}_lag1" in X.columns]
sector_dummy_cols = [c for c in X.columns if c.startswith('sector_')]
extra_static      = [c for c in ['sec_score_z_safe','is_short_hist_zscore', 'pandemic_dummy'] if c in X.columns]
skip_cols = sorted(set(macro_lag_cols + sector_dummy_cols + extra_static))
print(f"  · Passthrough estático (skip_cols): {len(skip_cols)} cols")
skip_cols = [c for c in skip_cols if c in X.columns]

# --- PASO 3: INSTANCIAR TRANSFORMADORES ---

# Verificación crucial de alineación de índices
assert X.index.equals(df_sorted.index), "FATAL: El índice de X y df_sorted debe ser idéntico."
assert X.index.equals(df_model_sorted.index), "FATAL: El índice de X y df_model_sorted debe ser idéntico."

_z_base = CrossSectionalZ(dates_full=DATES_SERIES, freq='M', fill_invalid='zero')
_w_base = CrossSectionalWinsorizer(dates_full=DATES_SERIES, freq='M', lower=0.01, upper=0.01, min_group=5)
zscore_cs = SafeSkipColsWrapper(_z_base, skip_cols=skip_cols)
winsor_cs = SafeSkipColsWrapper(_w_base, skip_cols=skip_cols)

# Instanciación del nuevo SectorDemeanCS (el wrapper)
sector_demean = SectorDemeanCS(
    skip_cols=skip_cols,  # Pasa la lista o set directamente
    sector_series=df_sorted['Sector_GICS'],
    dates_series=pd.to_datetime(df_model_sorted['Fecha'])
)

# --- PASO 4: RECONSTRUIR PIPELINES ---
MODELS_ALL, SEARCH_CFG = build_models_and_search(
    FAST_MODE=FAST_MODE,
    GPU_LGBM=(False if 'GPU_LGBM' not in globals() else GPU_LGBM),
    N_JOBS=N_JOBS,
    ALLOW_GKX=ALLOW_GKX
)

# Forzar LGBM=CPU y XGB/CatBoost=GPU si hay
MODELS_ALL = force_devices(MODELS_ALL, use_gpu=HAS_GPU)

print("  ✅ Transformadores y pipelines refrescados y listos para usar.")



⏩  BLOQUE 11.y – Re-instanciando transformadores con orden corregido...
  · Passthrough estático (skip_cols): 15 cols
[Config] XGBoost → tree_method=hist | predictor=cpu_predictor | GPU=False
[Config] LightGBM → device_type=cpu
[Config] CatBoost → task_type=CPU
[Config] GKX → device=cpu | n_ens=3

⚡ Forzando configuración de dispositivos (GPU: False)...
   -> Configurando 'XGBoost' para CPU.
   -> Configurando 'LightGBM' para CPU.
   -> Configurando 'LightGBM (Ranker)' para CPU.
   -> Configurando 'CatBoost (Ranker)' para CPU.
  ✅ Transformadores y pipelines refrescados y listos para usar.


In [ ]:
from pandas.api.types import is_period_dtype

def _as_period_m(x):
    s = pd.Series(x)
    if is_period_dtype(s.dtype):       # ya es Period
        return s.dt.asfreq('M')
    else:                              # timestamps/strings/etc.
        return pd.to_datetime(s, errors='coerce').dt.to_period('M')


In [ ]:
# ===================== HELPERS PARA TUNING / MÉTRICAS =====================

# Elige la cohorte/ancla del trimestre (0: ene/abr/jul/oct; 1: feb/may/ago/nov; 2: mar/jun/sep/dic)
ANCHOR_Q = 2

def _iter_time_blocks(months, train_min=None, gap=None, hold=1, expanding=True):
    """
    Walk-forward con embargo en MESES reales y test en fronteras trimestrales (ANCHOR_Q).
    """
    import numpy as np, pandas as pd
    if train_min is None: train_min = TRAIN_MIN
    if gap is None:       gap       = EMBARGO_M

    m = _as_period_m(months)                         # <-- robusto a Period/Datetime
    uniq = np.array(sorted(pd.unique(m)))

    boundaries   = np.array([u for u in uniq if ((u.month - 1) % 3) == ANCHOR_Q])
    min_quarters = max(1, int(np.ceil(train_min / 3)))

    for b in range(min_quarters - 1, len(boundaries) - hold):
        test_bounds = boundaries[b+1 : b+1+hold]
        cutoff      = test_bounds.min() - gap        # primer mes de test menos embargo

        if expanding:
            tr_mask = (m < cutoff).to_numpy()        # *** estricto: < cutoff ***
        else:
            start   = cutoff - (train_min - 1)
            tr_mask = ((m >= start) & (m < cutoff)).to_numpy()

        te_mask = m.isin(test_bounds).to_numpy()

        if np.any(tr_mask) and np.any(te_mask):
            yield np.where(tr_mask)[0], np.where(te_mask)[0]


# === CV interna con embargo (versión única y estable) ===
from scipy.stats import spearmanr

def make_inner_cv_with_embargo(months_tr, n_splits=CV_SPLITS_INNER, gap=EMBARGO_M, hold=1, min_train_inner=6):
    months_tr = _as_period_m(months_tr)
    # Usa la misma lógica que el outer, pero con train_min interno
    cv_all = list(_iter_time_blocks(months_tr,
                                train_min=min_train_inner,
                                gap=gap,
                                hold=hold,
                                expanding=True))

    if not cv_all:
        # Fallback mínimo (último mes como validación)
        uniq = np.array(sorted(pd.unique(months_tr)))
        if len(uniq) >= 2:
            tr_idx = np.where(months_tr < uniq[-1])[0]
            te_idx = np.where(months_tr == uniq[-1])[0]
            if len(tr_idx) and len(te_idx):
                cv_all = [(tr_idx, te_idx)]
    # Toma los últimos n_splits (más recientes)
    return cv_all[-n_splits:] if len(cv_all) > n_splits else cv_all


# === Helper para crear el tuner (Halving) y elegir “resource”

def _infer_halving_resource(model_name: str, base_pipe, fast_mode=False):
    """
    Devuelve (resource, min_resources, max_resources) o (None, None, None)
    para usar con HalvingRandomSearchCV.
    Heurística simple por tipo de modelo:
      - LightGBM / XGBoost / RandomForest → 'model__n_estimators'
      - CatBoost (Ranker)                 → 'model__iterations'
      - GKX (NN)                          → 'model__epochs' (si existiera)
      - Otros                             → None (cae a RandomizedSearchCV)
    """
    name = (model_name or "").lower()
    if "lightgbm" in name or "xgboost" in name or "random forest" in name:
        res = "n_estimators"
        # límites prudentes (más chicos si fast_mode)
        return (f"model__{res}", 50 if fast_mode else 200, 150 if fast_mode else 800)
    if "catboost" in name:
        res = "iterations"
        return (f"model__{res}", 300 if fast_mode else 600, 600 if fast_mode else 1200)
    if "gkx" in name or "nn" in name:
        # solo si tu wrapper expone epochs como hiperparámetro
        return ("model__epochs", 5 if fast_mode else 20, 20 if fast_mode else 80)
    return (None, None, None)


def make_tuner(model_name, base_pipe, param_dist, inner_cv,
               n_candidates='auto', factor=3, random_state=42, n_iter=N_ITER_RS):
    """
    Devuelve un tuner listo:
      - HalvingRandomSearchCV si hay 'resource' idóneo,
      - Si no, cae a RandomizedSearchCV (más chico).
    """
    search_estimator = RankPipeline(_sk_clone(base_pipe).steps)

    # --- nº de candidatos ---
    if n_candidates == 'auto':
        n_candidates_final = 4 if FAST_MODE else max(24, 3 * n_iter)
    else:
        n_candidates_final = n_candidates

    # --- paralelismo del tuner (CatBoost en 1 hilo) ---
    model_l = model_name.lower()
    n_jobs_tuner = 1 if "catboost" in model_l else SEARCH_N_JOBS

    # --- halving si hay 'resource' ---
    resource, min_res_full, max_res_full = _infer_halving_resource(model_name, base_pipe)
    if resource is not None:
        # quitar 'resource' de la grilla si está
        param_dist_copy = {k: v for k, v in param_dist.items() if k != resource}

        # budgets por modo
        min_res_final = 10 if FAST_MODE else min_res_full
        max_res_final = 20 if FAST_MODE else max_res_full

        # ajuste agresivo para RandomForest (antes del return)
        if "random forest" in model_l:
            min_res_final = 100 if FAST_MODE else max(min_res_full, 200)
            max_res_final = 300 if FAST_MODE else max(max_res_full, 500)

        try:
            # scikit-learn >= 1.4
            return HalvingRandomSearchCV(
                estimator=search_estimator,
                param_distributions=param_dist_copy,
                n_candidates=n_candidates_final,
                factor=factor,
                resource=resource,
                min_resources=min_res_final,
                max_resources=min_res_final if max_res_final is None else max_res_final,
                aggressive_elimination=True,
                cv=inner_cv,
                scoring=spearman_scorer,
                n_jobs=n_jobs_tuner,           # ← aquí controlas hilos del tuner
                random_state=random_state,
                verbose=1,
                refit=False,
                return_train_score=False,
                error_score="raise",
            )
        except TypeError:
            # versiones con 'reduction_factor'
            return HalvingRandomSearchCV(
                estimator=search_estimator,
                param_distributions=param_dist_copy,
                n_candidates=n_candidates_final,
                reduction_factor=factor,
                resource=resource,
                min_resources=min_res_final,
                max_resources=min_res_final if max_res_final is None else max_res_final,
                aggressive_elimination=True,
                cv=inner_cv,
                scoring=spearman_scorer,
                n_jobs=n_jobs_tuner,           # ← idem
                random_state=random_state,
                verbose=1,
                refit=False,
                return_train_score=False,
                error_score="raise",
            )

    # --- fallback: RandomizedSearchCV “chico” ---
    return RandomizedSearchCV(
        estimator=search_estimator,
        param_distributions=param_dist,
        n_iter=(4 if FAST_MODE else n_iter),
        cv=inner_cv,
        scoring=spearman_scorer,
        n_jobs=n_jobs_tuner,                   # ← idem
        random_state=random_state,
        verbose=0,
        refit=False,
        return_train_score=False,
        error_score="raise",
    )

def _ic_monthly_from_oof(dates_series, y, oof):
    """Devuelve (serie IC por mes, IC medio) desde predicciones OOF."""
    months = pd.to_datetime(pd.Series(dates_series)).dt.to_period('M')
    df = pd.DataFrame({'m': months.values, 'y': pd.Series(y).values, 'p': np.asarray(oof)})
    ic_m = (df.groupby('m', sort=True)
              .apply(lambda g: spearmanr(g['y'], g['p'])[0] if g['p'].nunique()>1 else np.nan)
              .dropna())
    ic_mean = float(ic_m.mean()) if len(ic_m) else np.nan
    return ic_m, ic_mean

def _spread_monthly_from_oof(dates_series, y, oof):
    """Devuelve (serie spread por mes, spread medio). Spread = D10–D1 (fallback a Q5–Q1)."""
    months = pd.to_datetime(pd.Series(dates_series)).dt.to_period('M')
    df = pd.DataFrame({'m': months.values, 'y': pd.Series(y).values, 'p': np.asarray(oof)})

    def _spr(g):
        r = g['p'].rank(method='first')
        try:
            if len(g) >= 10 and r.nunique() >= 10:
                d = pd.qcut(r, 10, labels=np.arange(1,11), duplicates='drop')
                m = g.assign(dec=d.astype(int)).groupby('dec')['y'].mean()
                return float(m.get(10, np.nan) - m.get(1, np.nan))
            elif len(g) >= 5 and r.nunique() >= 5:
                q = pd.qcut(r, 5,  labels=np.arange(1,6),  duplicates='drop')
                m = g.assign(q=q.astype(int)).groupby('q')['y'].mean()
                return float(m.get(5, np.nan) - m.get(1, np.nan))
            else:
                return np.nan
        except ValueError:
            return np.nan

    spr_m = df.groupby('m', sort=True).apply(_spr).dropna()
    spr_mean = float(spr_m.mean()) if len(spr_m) else np.nan
    return spr_m, spr_mean


In [ ]:
def _make_inner_cv_dyn(months_tr, gap, hold, splits_desired=None, min_train_inner=6):
    if splits_desired is None:
        # usa lo que tengas en globals, con fallback sensato
        splits_desired = globals().get('CV_SPLITS_INNER', 3)

    try:
        months_tr = pd.to_datetime(months_tr).dt.to_period('M')
    except Exception:
        pass
    uniq = np.array(sorted(pd.unique(months_tr)))
    L = len(uniq)
    if L < 2:
        return []

    # train_min interno: lo más grande posible que deje ≥1 split con embargo
    train_min_inner = max(min_train_inner, min(L - (gap + hold), L - 1))

    cv_all = list(_iter_time_blocks(
        months_tr,
        train_min=train_min_inner,
        gap=gap,
        hold=hold
    ))

    # Fallback: 1 split (último mes como validación)
    if not cv_all:
        last = uniq[-1]
        tr_idx = np.where(months_tr < last)[0]
        te_idx = np.where(months_tr == last)[0]
        if len(tr_idx) and len(te_idx):
            cv_all = [(tr_idx, te_idx)]

    # Nos quedamos con los últimos splits deseados (más recientes)
    if len(cv_all) > splits_desired:
        cv_all = cv_all[-splits_desired:]

    return cv_all


In [ ]:
# === Guardrails de adopción de hiperparámetros ===
ADOPT_MIN_DELTA_IC   = 0.005   # mejora mínima de IC (0.005 = +0.5pp)
ADOPT_MIN_ABS_IC     = -1e9    # opcional: exige un IC mínimo absoluto
ADOPT_COOLDOWN       = 0       # esperar N retuneos antes de volver a cambiar (0 = sin cooldown)

In [ ]:
def ensemble_predict_on_indices(
    base_estimator,
    te_idx,
    tr_idx=None,
    months_series=MONTHS_ALL,
    lags=ENSEMBLE_LAGS_SAFE,
    weights=ENSEMBLE_WEIGHTS,
    min_months=MIN_TRAIN_MONTHS_ENSEMBLE
):
    """
    Genera predicciones para te_idx usando un ensemble de vistas temporales.
    Respeta el embargo y la ventana de entrenamiento del fold si se proporciona tr_idx.
    """
    import numpy as np
    import pandas as pd
    from sklearn.base import clone

    # --- 0) Normaliza tr_idx ---
    if tr_idx is not None and not isinstance(tr_idx, pd.Index):
        tr_idx = pd.Index(tr_idx)

    # --- 1) Ventana test y colección de submodelos por lags ---
    test_start_month = months_series.iloc[te_idx].min()
    sub_predictions = []

    for lag in lags:
        cutoff_month = test_start_month - lag
        sub_tr_idx_pool = months_series.index[months_series < cutoff_month]
        sub_tr_idx_final = sub_tr_idx_pool.intersection(tr_idx) if tr_idx is not None else sub_tr_idx_pool

        # Suficiencia de historia para este submodelo
        if months_series.loc[sub_tr_idx_final].nunique() < min_months:
            continue

        # Anti-leakage explícito
        last_train_month = months_series.loc[sub_tr_idx_final].max()
        embargo_limit = test_start_month - EMBARGO_M
        assert last_train_month < embargo_limit, (
            f"Embargo roto: último mes de train={last_train_month} "
            f"debe ser < {embargo_limit} (embargo={EMBARGO_M} meses)."
        )

        estimator_sub = clone(base_estimator)
        X_train_sub, y_train_sub = X.loc[sub_tr_idx_final], y.loc[sub_tr_idx_final]
        months_train_sub = months_series.loc[sub_tr_idx_final]
        model_sub_fitted = fit_with_rank_support(estimator_sub, X_train_sub, y_train_sub, months_train_sub)
        sub_predictions.append(model_sub_fitted.predict(X.iloc[te_idx]))

    # --- 2) Ensemble normal si hay submodelos válidos ---
    if sub_predictions:
        valid_weights = np.asarray(weights[:len(sub_predictions)], dtype=float).copy()
        valid_weights[~np.isfinite(valid_weights)] = 0.0
        if valid_weights.sum() == 0:
            valid_weights[:] = 1.0
        valid_weights /= valid_weights.sum()
        return np.average(np.vstack(sub_predictions), axis=0, weights=valid_weights)

    # --- 3) Fallback INFORMADO (evita “constante” en smoke)
    # Si no hubo historia suficiente para los lags “formales”, armamos snapshots relajados
    # con lags pequeños sin exigir min_months, siempre respetando el embargo.
    relaxed_lags = (1, 2, 3, 6) if globals().get("SMOKE_TEST", False) else (1, 3, 6, 9, 12)
    relaxed_preds = []
    for lag in relaxed_lags:
        cutoff_month = test_start_month - lag
        idx_relaxed = months_series.index[months_series < cutoff_month]
        if tr_idx is not None:
            idx_relaxed = idx_relaxed.intersection(tr_idx)

        # Piso mínimo de observaciones para evitar ajustes degenerados
        if idx_relaxed.size < 25:
            continue

        last_train_month = months_series.loc[idx_relaxed].max()
        embargo_limit = test_start_month - EMBARGO_M
        assert last_train_month < embargo_limit, (
            f"Embargo roto (relaxed): último mes de train={last_train_month} "
            f"debe ser < {embargo_limit}"
        )

        estimator_relaxed = clone(base_estimator)
        model_relaxed = fit_with_rank_support(
            estimator_relaxed, X.loc[idx_relaxed], y.loc[idx_relaxed], months_series.loc[idx_relaxed]
        )
        relaxed_preds.append(model_relaxed.predict(X.iloc[te_idx]))

    if relaxed_preds:
        # Pesos iguales (o usa 1/log(1+lag) si prefieres ponderar)
        r_weights = np.ones(len(relaxed_preds), dtype=float) / len(relaxed_preds)
        return np.average(np.vstack(relaxed_preds), axis=0, weights=r_weights)

    # --- 4) Último recurso: snapshot clásico (igual que tu versión original) ---
    print(f"      ⚠️ Fallback a snapshot único para {test_start_month} (poca historia para el ensemble).")
    if tr_idx is None:
        fallback_tr_mask = months_series < (test_start_month - EMBARGO_M)
        tr_idx = np.where(fallback_tr_mask)[0]

    estimator_fallback = clone(base_estimator)
    model_fallback = fit_with_rank_support(
        estimator_fallback, X.iloc[tr_idx], y.iloc[tr_idx], months_series.iloc[tr_idx]
    )
    return model_fallback.predict(X.iloc[te_idx])


In [ ]:
# =============================================================================
# BLOQUE 12 · Helpers de Ranking, CV externo mensual y fit con soporte de grupos
# =============================================================================
from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.base import clone as _sk_clone

# --------- Imports opcionales para detección segura de rankers ----------
try:
    from lightgbm import LGBMRanker
except Exception:
    class LGBMRanker:  # type: ignore
        pass

try:
    from xgboost import XGBRanker
except Exception:
    class XGBRanker:  # type: ignore
        pass

try:
    from catboost import CatBoostRanker  # Pool opcional, no es necesario para este fix
except Exception:
    CatBoostRanker = tuple()


# =============================================================================
# 12.1 · Normalización de meses y utilidades de queries/runs
# =============================================================================
def _to_series(obj) -> pd.Series:
    """Devuelve un pd.Series (copia liviana si no lo es)."""
    if isinstance(obj, pd.Series):
        return obj
    return pd.Series(obj)


def _months_to_period_m(months) -> pd.Series:
    """
    Normaliza cualquier months (Period/Datetime/str/array-like) a Series[Period[M]].
    """
    s = _to_series(months)
    if pd.api.types.is_period_dtype(s):
        return s.dt.asfreq("M")
    if pd.api.types.is_datetime64_any_dtype(s):
        return s.dt.to_period("M")
    return pd.to_datetime(s, errors="coerce").dt.to_period("M")


def _qids_from_months(months) -> np.ndarray:
    """
    Genera IDs de query por fila según el mes (en el ORDEN DE LAS FILAS).
    """
    m = _months_to_period_m(months)
    qids, _ = pd.factorize(m.astype(str), sort=False)
    return qids.astype(np.int32, copy=False)


def _run_lengths_from_qids(qids) -> np.ndarray:
    """
    Convierte qids (len==n) → tamaños de cada corrida (runs) consecutiva.
    Requiere que las filas estén agrupadas por query/mes (bloques contiguos).
    """
    q = np.asarray(qids, dtype=np.int32)
    if q.size == 0:
        return np.array([], dtype=np.int32)
    starts = np.flatnonzero(np.r_[True, q[1:] != q[:-1]])
    return np.diff(np.r_[starts, q.size]).astype(np.int32)


def _sizes_from_months(months) -> np.ndarray:
    """
    Deriva los tamaños por query/mes en el ORDEN DE LAS FILAS.
    (Usa run-lengths sobre qids para respetar el orden real).
    """
    return _run_lengths_from_qids(_qids_from_months(months))


# =============================================================================
# 12.2 · CV externo mensual (anclado a fronteras trimestrales) con embargo real
# =============================================================================
def cv_outer_monthly(months,
                     train_min: int | None = None,
                     gap: int | None = None,
                     hold: int = 1,
                     anchor_q: int | None = None,
                     limit: int | None = None):
    """
    Genera splits walk-forward con:
      • test en fronteras trimestrales (anchor_q ∈ {0,1,2} según (mes-1)%3)
      • train < (primer mes de test – gap)  [embargo en MESES reales]
      • hold = nº de fronteras trimestrales consecutivas en test (usualmente 1)

    Parámetros:
        months    : array-like de fechas/periodos (una por fila del dataset)
        train_min : meses mínimos en train (default: globals().TRAIN_MIN o 36)
        gap       : embargo en meses (default: globals().EMBARGO_M o 3)
        hold      : nº de fronteras trimestrales consecutivas para test
        anchor_q  : {0,1,2}; 2 ≈ meses 3/6/9/12 (fin de trimestre).
                     default: globals().ANCHOR_Q o 2
        limit     : máximo número de splits (default: globals().OUTER_SPLITS_LIMIT)

    Devuelve:
        lista de tuplas (tr_idx, te_idx) con índices POSICIONALES (np.ndarray)
    """
    # Defaults desde globals()
    if train_min is None: train_min = int(globals().get("TRAIN_MIN", 36))
    if gap       is None: gap       = int(globals().get("EMBARGO_M", 3))
    if anchor_q  is None: anchor_q  = int(globals().get("ANCHOR_Q", 2))
    if limit     is None: limit     = globals().get("OUTER_SPLITS_LIMIT", None)

    m = _months_to_period_m(months)
    # Unique meses en orden cronológico
    uniq = pd.PeriodIndex(m.unique()).sort_values()
    # Fronteras trimestrales según ancla
    boundaries = pd.PeriodIndex([u for u in uniq if ((u.month - 1) % 3) == anchor_q])

    # Convertir requisito mínimo de train en nº de trimestres
    min_quarters = max(1, int(np.ceil(train_min / 3)))

    splits = []
    # Avanza por fronteras trimestrales; el primer test posible requiere train_min
    for b in range(min_quarters - 1, len(boundaries) - hold):
        test_bounds = boundaries[b+1:b+1+hold]          # meses usados en test
        cutoff      = test_bounds.min() - gap           # Period[M] - int meses

        tr_idx = np.where(m < cutoff)[0]
        te_idx = np.where(m.isin(test_bounds))[0]

        if tr_idx.size and te_idx.size:
            splits.append((tr_idx, te_idx))
            if limit and len(splits) >= int(limit):
                break
    return splits


# =============================================================================
# 12.3 · Ajuste con soporte de grupos/meses para Rankers (versión robusta)
# =============================================================================
def _infer_last_step_name(est) -> str | None:
    """Nombre del último paso del Pipeline, o None si no es Pipeline."""
    if isinstance(est, Pipeline) and est.steps:
        return est.steps[-1][0]
    return None


def _prefix_fit_params(est, params: dict, step_name: str | None) -> dict:
    """
    Si hay Pipeline, prefija 'step__' para que los params lleguen al estimador final.
    """
    if isinstance(est, Pipeline) and step_name:
        return {f"{step_name}__{k}": v for k, v in params.items()}
    return params

def fit_with_rank_support(estimator_or_pipe, X, y, months, clone_first=True):
    """
    Ajusta 'estimator_or_pipe' respetando:
      - Rankers puros (LGBMRanker/XGBRanker/CatBoostRanker): y -> ranks por mes (LGBM/XGB) o y float (CatBoost) + grupos.
      - Wrappers (RFCAsRanker/CatBoostSk...): autoselección de kwargs (group_id / group / months).
      - Evita el 'prefix hell' de Pipeline.
    """
    import numpy as np, pandas as pd
    from sklearn.base import clone as _sk_clone
    from sklearn.pipeline import Pipeline

    def _split_pipeline(_est):
        if isinstance(_est, Pipeline):
            pre = _est[:-1]
            final_est = _est.steps[-1][1]
            last_name = _est.steps[-1][0]  # nombre del último paso
            return pre, final_est, last_name
        return None, _est, None

    est = _sk_clone(estimator_or_pipe) if clone_first else estimator_or_pipe
    y_for_fit = y
    fit_params = {}

    pre, final_est, last_step_name = _split_pipeline(est)
    clsname = type(final_est).__name__

    # CatBoost: distinguir nativo vs wrapper (Sk)
    is_catboost_native  = isinstance(final_est, CatBoostRanker)
    is_catboost_wrapper = (("CatBoost" in clsname) and not is_catboost_native) or getattr(final_est, "requires_months", False)

    # Usa las clases definidas arriba del bloque (LGBMRanker/XGBRanker pueden ser stubs)
    is_true_ranker    = isinstance(final_est, (LGBMRanker, XGBRanker)) or is_catboost_native
    is_wrapper_ranker = ("Ranker" in clsname and not is_true_ranker) or is_catboost_wrapper

    # ---------- rama ranker / wrapper ----------
    if is_true_ranker or is_wrapper_ranker:
        Xt = pre.fit_transform(X, y_for_fit) if pre is not None else X

        assert months is not None, "Para rankers se requiere 'months'."
        assert len(Xt) == len(y_for_fit) == len(months), "Dimensiones inconsistentes Xt/y/months."

        clean_params = {k: v for k, v in fit_params.items() if "__" not in k}

        if is_catboost_native:
            # CatBoost nativo: y float + group_id por fila
            y_fit_final = y_for_fit
            qids = _qids_from_months(months)
            assert len(qids) == len(Xt), "group_id (qids) debe igualar n filas."
            fit_kwargs = dict(group_id=qids)
            dbg = "[RANKER FIX] CatBoost nativo ← group_id"

        elif isinstance(final_est, (LGBMRanker, XGBRanker)):
            # LGBM/XGB Ranker: y en bins + tamaños por query
            y_rank = _to_series(make_rel_labels(pd.Series(y), months, n_bins=10)).fillna(0).astype("int32")
            y_fit_final = y_rank
            sizes = np.asarray(_sizes_from_months(months), dtype=np.int32)
            assert sizes.sum() == len(Xt), "Suma de tamaños de grupo != n filas."
            fit_kwargs = dict(group=sizes)
            dbg = "[RANKER FIX] LGBM/XGB ← group + y=bins"

        else:
            # ======= WRAPPERS (RFCAsRanker, CatBoostRankerSk, etc.) CON DETECCIÓN DINÁMICA =======
            import inspect
            y_fit_final = y_for_fit
            try:
                sig = inspect.signature(final_est.fit)
                params = sig.parameters
            except Exception:
                params = {}

            if "group_id" in params:
                # Wrappers estilo CatBoostSk: usa IDs por fila, y en float
                qids = _qids_from_months(months)
                assert len(qids) == len(Xt), "group_id (qids) debe igualar n filas."
                fit_kwargs = dict(group_id=qids)
                dbg = "[RANKER FIX] Wrapper ← group_id"

            elif "group" in params:
                # Wrappers que aceptan kwarg 'group'
                if "CatBoost" in clsname:
                    # CatBoost wrappers usan 'group' como group_id POR FILA
                    qids = _qids_from_months(months)
                    assert len(qids) == len(Xt), "group (qids) debe igualar n filas."
                    y_fit_final = y_for_fit  # CatBoost/YetiRank prefiere y float
                    fit_kwargs = dict(group=qids)
                    dbg = "[RANKER FIX] Wrapper CatBoost ← group(qids)"
                else:
                    # Otros wrappers: 'group' = tamaños por query y y=bins
                    y_rank = _to_series(make_rel_labels(pd.Series(y), months, n_bins=10)).fillna(0).astype("int32")
                    y_fit_final = y_rank
                    sizes = np.asarray(_sizes_from_months(months), dtype=np.int32)
                    assert sizes.sum() == len(Xt), "Suma de tamaños de grupo != n filas."
                    fit_kwargs = dict(group=sizes)
                    dbg = "[RANKER FIX] Wrapper ← group(sizes) + y=bins"

            elif "months" in params:
                # Wrappers propios tipo RFCAsRanker: aceptan months directamente
                fit_kwargs = dict(months=months)
                dbg = "[RANKER FIX] Wrapper ← months"

            else:
                # Último recurso: sin kwargs especiales
                fit_kwargs = {}
                dbg = "[RANKER FIX] Wrapper ← sin kwargs especiales"

        # ---- Ajuste final del estimador ----
        final_est.fit(Xt, y_fit_final, **fit_kwargs, **clean_params)
        try:
            y_mean = float(np.asarray(y_fit_final, dtype=np.float64).mean())
            print(f"{dbg}: y_mean={y_mean:.3f}, n_samples={len(Xt)}")
        except Exception:
            print(f"{dbg}: n_samples={len(Xt)}")

        est = Pipeline(list(pre.steps) + [(last_step_name, final_est)]) if pre is not None else final_est
        return est

    # ---------- rama No-ranker ----------
    if '_prefix_fit_params' in globals():
        fit_params = _prefix_fit_params(est, fit_params, last_step_name)
    elif isinstance(est, Pipeline) and last_step_name is not None:
        fit_params = {f"{last_step_name}__{k}": v for k, v in fit_params.items()}

    est.fit(X, y_for_fit, **fit_params)
    return est

# ======================== HALVING + GRIDS ========================
from sklearn.experimental import enable_halving_search_cv  # noqa: F401
from sklearn.model_selection import HalvingRandomSearchCV

LGBM_PARAM_GRID = {
    "num_leaves": [31, 63, 127],
    "min_data_in_leaf": [10, 20, 50],
    "feature_fraction": [0.70, 0.85, 0.95],
    "bagging_fraction": [0.70, 0.85],
    "reg_lambda": [0.0, 1.0, 5.0, 10.0],
}

def build_halving_lgbm(scorer=SPEARMAN_SAFE, seed=42):
    return HalvingRandomSearchCV(
        estimator=make_lgbm_cpu(seed=seed),
        param_distributions=LGBM_PARAM_GRID,
        factor=3,
        resource="n_estimators",
        max_resources=1200,
        min_resources=200,
        random_state=seed,
        n_jobs=1,
        scoring=scorer,
        error_score=0.0,
        verbose=0
    )

# RF "ranker-bins": grid válido SOLO con hiper de RF
RF_RANKER_PARAM_GRID = {
    "n_estimators": [400, 800],
    "max_depth": [None, 18, 28],
    "min_samples_leaf": [3, 5],
    "max_features": [0.5, 0.7],
}


In [ ]:
from inspect import signature  # Agrega este import si no lo tenés

def _tuner_fit_safe(tuner, X, y, months=None):
    try:
        # Si el estimator soporta 'months' en fit, pásalo; sino, no
        if months is not None:
            sig = signature(tuner.estimator.fit)
            if 'months' in sig.parameters:
                return tuner.fit(X, y, months=months)
        return tuner.fit(X, y)
    except Exception:
        # Si falla por cualquier razón, usa fit básico sin extras
        return tuner.fit(X, y)

In [ ]:
# =============================================================================
# BLOQUE 12: Backtest Walk-Forward con retuneo poco frecuente + OOF
# =============================================================================

def run_backtest_and_generate_oof(
    models_all: dict,
    search_cfg: dict,
    X: pd.DataFrame,
    y: pd.Series,
    months_all: pd.Series,
    dates_series: pd.Series,
    groups: pd.Series,
    save_oof: bool = True,
    oof_dir: str = "artifacts/oof_preds",
    hold: int = 1,
    expanding: bool = True,
):
    """
    Ejecuta walk-forward mensual con:
      • Retuneo cada N folds (poco frecuente)
      • Adopción condicional por mejora mínima (ADOPT_MIN_DELTA_IC)
      • Predicciones OOF + métricas (IC y Spread por mes y promedio)
      • Ensemble por lags (si DO_MONTHLY_ENSEMBLE=True)

    Requisitos: los helpers y flags ya definidos en celdas previas.
    """
    # ------- Flags/valores globales con fallback seguro -------
    DO_RETUNE           = bool(globals().get("DO_RETUNE", True))
    RETUNE_EVERY_FOLDS  = int(globals().get("RETUNE_EVERY_FOLDS", 4))
    N_ITER_RS           = int(globals().get("N_ITER_RS", 60))
    CV_SPLITS_INNER     = int(globals().get("CV_SPLITS_INNER", 5))
    TRAIN_MIN           = int(globals().get("TRAIN_MIN", 36))
    EMBARGO_M           = int(globals().get("EMBARGO_M", 3))
    OUTER_SPLITS_LIMIT  = globals().get("OUTER_SPLITS_LIMIT", None)

    DO_MONTHLY_ENSEMBLE = bool(globals().get("DO_MONTHLY_ENSEMBLE", False))
    ENSEMBLE_LAGS_SAFE  = list(globals().get("ENSEMBLE_LAGS_SAFE", [3,6,9]))
    ENSEMBLE_WEIGHTS    = np.asarray(globals().get("ENSEMBLE_WEIGHTS", [0.5,0.3,0.2]))
    MIN_TRAIN_MONTHS_ENSEMBLE = int(globals().get("MIN_TRAIN_MONTHS_ENSEMBLE", 36))

    ADOPT_MIN_DELTA_IC  = float(globals().get("ADOPT_MIN_DELTA_IC", 0.005))
    ADOPT_MIN_ABS_IC    = float(globals().get("ADOPT_MIN_ABS_IC", -1e9))
    ADOPT_COOLDOWN      = int(globals().get("ADOPT_COOLDOWN", 0))

    PRINT_TIMING        = bool(globals().get("PRINT_TIMING", True))
    WARN_FOLD_MINUTES   = int(globals().get("WARN_FOLD_MINUTES", 35))

    # ------- Generar folds outer -------
    folds = list(cv_outer_monthly(months_all, train_min=TRAIN_MIN, gap=EMBARGO_M, hold=hold))  # Fix: Quita expanding=expanding
    if OUTER_SPLITS_LIMIT is not None:
        folds = folds[:OUTER_SPLITS_LIMIT]
    n_folds = len(folds)
    if n_folds == 0:
        raise RuntimeError("No se generaron folds para el outer CV. Revisa TRAIN_MIN/EMBARGO_M/hold.")

    os.makedirs(oof_dir, exist_ok=True)

    results = {}
    total_start = time.time()

    # ===== Iterar por familia/modelo =====
    for fam, mdict in models_all.items():
        for name, base_pipe in mdict.items():
            model_key = f"{fam} :: {name}"
            if PRINT_TIMING:
                print(f"\n{('='*78)}\nModelo: {model_key}\nFolds: {n_folds}\n{('='*78)}")

            # Estado por modelo
            oof = np.full(len(X), np.nan, dtype="float32")
            current_params = None
            last_tuned_score = -np.inf
            cooldown = 0
            tune_count = 0

            fold_times = []
            for k, (tr_idx, te_idx) in enumerate(folds, start=1):
                t0 = time.time()

                X_tr = X.iloc[tr_idx]; y_tr = y.iloc[tr_idx]
                X_te = X.iloc[te_idx]
                m_tr = months_all.iloc[tr_idx]

                # -------- Retuneo poco frecuente --------
                do_tune = (k == 1) or (DO_RETUNE and (k % RETUNE_EVERY_FOLDS == 0) and cooldown == 0)
                if do_tune:
                    tune_count += 1
                    inner_cv = _make_inner_cv_dyn(m_tr, gap=EMBARGO_M, hold=1,
                                                  splits_desired=CV_SPLITS_INNER, min_train_inner=6)  # FIXED: Baja a 6 para más splits válidos

                    # FIXED: Filtrar splits válidos (test >=1, train >=10)
                    inner_cv = [(tr, te) for tr, te in inner_cv if len(te) >= 1 and len(tr) >= 10]

                    params_grid = search_cfg.get(name, {})
                    if inner_cv and params_grid:  # Solo si hay CV y grid
                        from sklearn.model_selection import RandomizedSearchCV
                        tuner = RandomizedSearchCV(
                            estimator=base_pipe,
                            param_distributions=params_grid,
                            n_iter=min(10, N_ITER_RS),
                            cv=inner_cv,
                            scoring=spearman_scorer,
                            n_jobs=4,
                            random_state=GLOBAL_SEED,
                            verbose=0,
                            refit=False,
                            return_train_score=False,
                            error_score=np.nan
                        )
                        try:
                            _tuner_fit_safe(tuner, X_tr, y_tr, months=m_tr)  # Fit seguro (no pasa months si no se soporta)
                            best_score = float(tuner.best_score_ if hasattr(tuner, 'best_score_') else np.nan)
                            best_params = tuner.best_params_ if hasattr(tuner, 'best_params_') else {}
                            print(f"[TUNING] Top score: {best_score:.4f}")
                        except ValueError as e:
                            if "fits failed" in str(e):
                                print("[TUNING] Todos candidatos fallaron (e.g., months no soportado) → skip.")
                                best_score, best_params = np.nan, {}
                            else:
                                raise
                        except Exception as e:
                            print(f"[TUNING] Error general: {e} → skip.")
                            best_score, best_params = np.nan, {}

                        # Adopción (igual que antes)
                        adopt = np.isfinite(best_score) and best_score > last_tuned_score + ADOPT_MIN_DELTA_IC
                        if adopt:
                            current_params = best_params
                            last_tuned_score = best_score
                            cooldown = ADOPT_COOLDOWN
                            print("Adoptado nuevo set.")
                        else:
                            print("No mejora → hold.")
                    else:
                        print("[TUNING] No CV o grid → skip.")
                else:
                    cooldown = max(0, cooldown - 1)

                # -------- Fit final del fold y predicción OOF --------
                est = _sk_clone(base_pipe)
                if current_params:
                    try:
                        est.set_params(**current_params)
                    except ValueError:
                        # Algún parámetro puede no aplicar a esta versión → ignora silenciosamente
                        pass

                if DO_MONTHLY_ENSEMBLE:
                    preds_te = ensemble_predict_on_indices(
                        base_estimator=est,
                        te_idx=np.asarray(te_idx),
                        tr_idx=np.asarray(tr_idx),
                        months_series=months_all,
                        lags=ENSEMBLE_LAGS_SAFE,
                        weights=ENSEMBLE_WEIGHTS,
                        min_months=MIN_TRAIN_MONTHS_ENSEMBLE
                    )
                else:
                    fitted = fit_with_rank_support(est, X_tr, y_tr, m_tr, clone_first=False)
                    preds_te = fitted.predict(X_te)

                # Guardar OOF en los índices de test del fold
                oof[te_idx] = preds_te.astype("float32")

                # -------- Timing del fold --------
                fold_minutes = (time.time() - t0) / 60.0
                fold_times.append(fold_minutes)
                if PRINT_TIMING:
                    warn = "  [LENTO]" if fold_minutes > WARN_FOLD_MINUTES else ""
                    print(f"[{k}/{n_folds}] fold listo en {fold_minutes:.1f} min{warn}")

            # ===== Métricas OOF por modelo =====
            ic_m, ic_mean = _ic_monthly_from_oof(dates_series, y, oof)
            spr_m, spr_mean = _spread_monthly_from_oof(dates_series, y, oof)

            summary = {
                "model": model_key,
                "n_folds": n_folds,
                "tunes": tune_count,
                "ic_mean": float(ic_mean) if np.isfinite(ic_mean) else None,
                "spread_mean": float(spr_mean) if np.isfinite(spr_mean) else None,
                "fold_time_min_mean": float(np.mean(fold_times)) if fold_times else None,
                "fold_time_min_p95": float(np.percentile(fold_times, 95)) if fold_times else None,
                "adopted_params": current_params or {},
            }

            # ===== Persistencia opcional =====
            if save_oof:
                base = os.path.join(oof_dir, model_key.replace(" ", "_").replace("/", "_"))
                np.save(f"{base}__oof.npy", oof)
                ic_m.to_csv(f"{base}__ic_mensual.csv", index=True)
                spr_m.to_csv(f"{base}__spread_mensual.csv", index=True)
                with open(f"{base}__summary.json", "w", encoding="utf-8") as f:
                    json.dump(summary, f, indent=2, ensure_ascii=False)

            results[model_key] = {
                "oof": oof, "ic_m": ic_m, "ic_mean": ic_mean,
                "spread_m": spr_m, "spread_mean": spr_mean,
                "summary": summary
            }

    total_minutes = (time.time() - total_start) / 60.0
    if PRINT_TIMING:
        print(f"\nBacktest completo ({len(results)} modelos). Tiempo total: {total_minutes:.1f} min.")
    return results


In [ ]:
# Ejecutar backtest + OOF para todos los modelos definidos
results = run_backtest_and_generate_oof(
    models_all=MODELS_ALL,
    search_cfg=SEARCH_CFG,
    X=X, y=y,
    months_all=MONTHS_ALL,
    dates_series=DATES_SERIES,
    groups=groups,
    save_oof=True,
    oof_dir="artifacts/oof_preds",
    hold=1,
    expanding=True
)

# Resumen compacto:
resumen = []
for k, v in results.items():
    resumen.append({
        "modelo": k,
        "IC_promedio": round(float(v["summary"]["ic_mean"]), 4) if v["summary"]["ic_mean"] is not None else None,
        "Spread_prom": round(float(v["summary"]["spread_mean"]), 4) if v["summary"]["spread_mean"] is not None else None,
        "tunes": v["summary"]["tunes"]
    })
pd.DataFrame(resumen).sort_values("IC_promedio", ascending=False)



Modelo: Lineales :: OLS (Linear Regression)
Folds: 2
[TUNING] No CV o grid → skip.
[1/2] fold listo en 0.1 min
[2/2] fold listo en 0.0 min

Modelo: Lineales :: ElasticNet
Folds: 2
[TUNING] No CV o grid → skip.
[1/2] fold listo en 0.2 min
[2/2] fold listo en 0.1 min

Modelo: Boosting :: Random Forest
Folds: 2
[TUNING] Top score: 0.2714
Adoptado nuevo set.
[1/2] fold listo en 5.6 min
[2/2] fold listo en 0.8 min

Modelo: Boosting :: Random Forest (Ranker-bins)
Folds: 2
[TUNING] Top score: 0.2727
Adoptado nuevo set.
[RANKER FIX] Wrapper ← months: y_mean=0.000, n_samples=6300
[RANKER FIX] Wrapper ← months: y_mean=0.000, n_samples=6300
[RANKER FIX] Wrapper ← months: y_mean=0.000, n_samples=6300
[RANKER FIX] Wrapper ← months: y_mean=0.000, n_samples=5760
[1/2] fold listo en 0.3 min
[RANKER FIX] Wrapper ← months: y_mean=-0.000, n_samples=6840
[2/2] fold listo en 0.0 min

Modelo: Boosting :: XGBoost
Folds: 2
[TUNING] Top score: nan
No mejora → hold.
[1/2] fold listo en 1.1 min
[2/2] fold listo

[1/2] fold listo en 1.2 min
[2/2] fold listo en 0.3 min

Backtest completo (9 modelos). Tiempo total: 19.3 min.


,modelo,IC_promedio,Spread_prom,tunes
0,Lineales :: OLS (Linear Regression),0.1210,0.0585,1
1,Lineales :: ElasticNet,0.1149,0.0425,1
8,Redes neuronales :: GKX NN (64-32-16),0.0615,-0.0330,1
7,Boosting :: CatBoost (Ranker),0.0587,0.0376,1
3,Boosting :: Random Forest (Ranker-bins),0.0479,0.0183,1
4,Boosting :: XGBoost,0.0189,0.0191,1
6,Boosting :: LightGBM (Ranker),0.0012,0.0069,1
2,Boosting :: Random Forest,-0.0167,-0.0159,1
5,Boosting :: LightGBM,-0.0587,-0.0504,1


In [ ]:
# =============================================================================
# BLOQUE 15: Leaderboard y selección SIN re-ejecutar backtest
# Requiere: results (dict) y MODELS_ALL ya construidos
# =============================================================================
import numpy as np
import pandas as pd

if 'results' not in globals():
    raise NameError("Falta 'results'. Ejecuta primero el bloque de backtest (run_backtest_and_generate_oof).")

rows = []
for name, r in results.items():
    s = r.get("summary", {}) if isinstance(r, dict) else {}
    # Intenta detectar una serie de retornos L/S mensuales si existe (opcional)
    ls_candidates = ["ls_monthly", "ls_by_month", "pnl_monthly", "ls"]
    ls_series = None
    for k in ls_candidates:
        v = r.get(k) if isinstance(r, dict) else None
        if v is not None:
            ls_series = pd.Series(v, dtype="float64")
            break

    sharpe_ann = np.nan
    ret_ann = np.nan
    if ls_series is not None and ls_series.dropna().shape[0] >= 2:
        ls_valid = ls_series.dropna().astype(float)
        mu_m = ls_valid.mean()
        sd_m = ls_valid.std(ddof=1)
        sharpe_ann = (np.sqrt(12.0) * mu_m / sd_m) if sd_m > 0 else np.nan
        # Retorno anualizado por composición
        ret_ann = (np.prod(1.0 + ls_valid) ** (12.0 / len(ls_valid))) - 1.0

    rows.append({
        "Modelo": name,
        "IC_Mean": s.get("ic_mean", np.nan),
        "Spread_Mean": s.get("spread_mean", np.nan),
        "Tunes": s.get("tunes", np.nan),
        "Sharpe_Anualizado": sharpe_ann,
        "Retorno_LS_Anual": ret_ann,
    })

leaderboard = pd.DataFrame(rows)

# Orden: si hay Sharpe, primero por Sharpe; si no, por IC
if leaderboard["Sharpe_Anualizado"].notna().any():
    leaderboard = leaderboard.sort_values(["Sharpe_Anualizado","IC_Mean"],
                                          ascending=[False, False],
                                          na_position="last")
else:
    leaderboard = leaderboard.sort_values("IC_Mean", ascending=False, na_position="last")

display(leaderboard)

# Selección de ganador
best_model_name = None
if not leaderboard.empty:
    best_model_name = str(leaderboard.iloc[0]["Modelo"])
    fam, pipe = None, None
    for fam_k, models in MODELS_ALL.items():
        if best_model_name in models:
            fam, pipe = fam_k, models[best_model_name]
            break
    print("\nSELECCIÓN FINAL")
    if fam is not None:
        print(f"Familia / Modelo: {fam} / {best_model_name}")
    else:
        print(f"Modelo: {best_model_name} (no se encontró en MODELS_ALL)")

# Persistencia
try:
    leaderboard.to_csv("leaderboard_final.csv", index=False)
    print("\nGuardado: 'leaderboard_final.csv'")
except Exception as e:
    print(f"\nAdvertencia al guardar leaderboard: {e}")


,Modelo,IC_Mean,Spread_Mean,Tunes,Sharpe_Anualizado,Retorno_LS_Anual
0,Lineales :: OLS (Linear Regression),0.121047,0.058512,1,NaN,NaN
1,Lineales :: ElasticNet,0.114874,0.042545,1,NaN,NaN
8,Redes neuronales :: GKX NN (64-32-16),0.061466,-0.033021,1,NaN,NaN
7,Boosting :: CatBoost (Ranker),0.058723,0.037603,1,NaN,NaN
3,Boosting :: Random Forest (Ranker-bins),0.047855,0.018338,1,NaN,NaN
4,Boosting :: XGBoost,0.018949,0.019083,1,NaN,NaN
6,Boosting :: LightGBM (Ranker),0.001238,0.006935,1,NaN,NaN
2,Boosting :: Random Forest,-0.016709,-0.015924,1,NaN,NaN
5,Boosting :: LightGBM,-0.058664,-0.050406,1,NaN,NaN



SELECCIÓN FINAL
Modelo: Lineales :: OLS (Linear Regression) (no se encontró en MODELS_ALL)

Guardado: 'leaderboard_final.csv'


In [ ]:
# ===== LEADERBOARD DESDE OOF (auto-descubrimiento + fallback memoria) =====
import os, glob
from pathlib import Path
from scipy.stats import spearmanr

# --- Helpers para normalizar "Familia :: Modelo" y encontrar el pipeline ---
def _split_family_model(model_str: str):
    if "::" in model_str:
        fam, name = [s.strip() for s in model_str.split("::", 1)]
        return fam, name
    return None, model_str.strip()

def find_pipeline_from_name(name_with_optional_fam: str):
    fam_hint, name = _split_family_model(name_with_optional_fam)
    # 1) Si vino con familia explícita y existe, úsala
    if fam_hint and fam_hint in MODELS_ALL and name in MODELS_ALL[fam_hint]:
        return fam_hint, name, MODELS_ALL[fam_hint][name]
    # 2) Buscar por nombre en todas las familias
    for fam, d in MODELS_ALL.items():
        if name in d:
            return fam, name, d[name]
    raise KeyError(f"No se encontró pipeline para '{name_with_optional_fam}'")


# --- Config y columnas estándar
COL_FECHA, COL_EMP, COL_Y, COL_P = "Fecha", "Empresa", "y_true", "y_pred"
COV_UMBRAL = 90.0

def _to_month_end(s):
    dt = pd.to_datetime(s, errors="coerce", utc=True)
    dt = dt.dt.tz_convert(None)
    return dt.dt.to_period("M").dt.to_timestamp("M")

def _ic_spearman_monthly(df_mes):
    g = df_mes[[COL_P, COL_Y]].dropna()
    if len(g) < 3 or g[COL_P].nunique() < 2:
        return np.nan
    return spearmanr(g[COL_Y], g[COL_P])[0]

def _decile_spread_monthly(df_mes, deciles=10):
    g = df_mes.dropna(subset=[COL_P, COL_Y]).copy()
    if len(g) < 2*deciles:
        return np.nan
    r = g[COL_P].rank(method="first")
    try:
        d = pd.qcut(r, deciles, labels=False, duplicates="drop")
    except ValueError:
        return np.nan
    return float(g.loc[d==d.max(), COL_Y].mean() - g.loc[d==d.min(), COL_Y].mean())

def _find_oof_files(candidate_dirs):
    pats = ("oof_*.csv", "oof_*.parquet", "oof_*.npy")
    out = []
    for d in candidate_dirs:
        if not d:
            continue
        for pat in pats:
            out += glob.glob(os.path.join(d, "**", pat), recursive=True)
    # unique + sorted
    return sorted(set(out))

# --- 1) Descubrir OOF en disco (varias rutas típicas)
CANDIDATE_DIRS = list(dict.fromkeys([
    globals().get("OOF_DIR", None),
    globals().get("oof_dir", None),
    "artifacts/oof_preds",
    "artifacts/oof_tuned_retune",
    "artifacts/oof",
    "."
]))
files = _find_oof_files(CANDIDATE_DIRS)
print(f"[INFO] OOF candidatos={len(files)}")
for f in files[:10]:
    print(" -", f)

oof_parts = []
if files:
    for f in files:
        if f.endswith(".csv"):
            df = pd.read_csv(f)
        elif f.endswith(".parquet"):
            df = pd.read_parquet(f)
        else:  # .npy
            arr = np.load(f, allow_pickle=True)

            if arr.ndim == 2 and arr.shape[1] >= 4:
                # Formato antiguo: [y_true, y_pred, Fecha, Empresa]
                df = pd.DataFrame(arr, columns=[COL_Y, COL_P, COL_FECHA, COL_EMP])
            else:
                # Vector 1D o (N,1) de predicciones → alinear por longitud mínima
                arr = np.asarray(arr).reshape(-1)

                dates_series = pd.to_datetime(globals().get("DATES_SERIES"), errors="coerce")
                y_series     = pd.Series(globals().get("y", globals().get("y_GLOBAL")))
                emp_series   = globals().get("EMP_SERIES", None)
                if emp_series is None:
                    # usa la longitud de y_series (no la de dates_series) para evitar descalces
                    emp_series = pd.Series(["NA"] * len(y_series), index=y_series.index)

                n = int(min(len(arr), len(y_series), len(dates_series), len(emp_series)))
                if n <= 0:
                    print(f"[WARN] Longitud nula reconstruyendo {f}; skip.")
                    continue

                df = pd.DataFrame({
                    COL_Y:     pd.to_numeric(y_series.iloc[:n], errors="coerce").to_numpy(),
                    COL_P:     pd.to_numeric(arr[:n], errors="coerce"),
                    COL_FECHA: dates_series.iloc[:n].to_numpy(),
                    COL_EMP:   pd.Series(emp_series, index=y_series.index).astype(str).iloc[:n].to_numpy(),
                })

        if "Model" not in df.columns:
            df["Model"] = Path(f).stem.replace("oof_", "").rsplit(".", 1)[0]
        df[COL_FECHA] = _to_month_end(df[COL_FECHA])
        if COL_EMP not in df.columns:
            df[COL_EMP] = "NA"
        df[COL_Y] = pd.to_numeric(df[COL_Y], errors="coerce")
        df[COL_P] = pd.to_numeric(df[COL_P], errors="coerce")
        oof_parts.append(df)

# --- 2) Fallback: reconstruir desde memoria si no hay nada en disco
if not oof_parts:
    print("[INFO] No se hallaron OOF en disco; intentando usar memoria…")
    oof_predictions = globals().get("oof_predictions", None)
    results         = globals().get("results", None)

    dates_series = pd.to_datetime(globals().get("DATES_SERIES"), errors="coerce")
    y_series     = pd.Series(globals().get("y", globals().get("y_GLOBAL")))
    emp_series   = globals().get("EMP_SERIES", None)
    if emp_series is None:
        emp_series = pd.Series(["NA"] * len(dates_series), index=y_series.index)

    if isinstance(oof_predictions, dict):
        for m, s in oof_predictions.items():
            s = pd.to_numeric(pd.Series(s), errors="coerce")
            s = s.reindex(y_series.index)
            df = pd.DataFrame({
                COL_Y:     pd.to_numeric(y_series, errors="coerce").to_numpy(),
                COL_P:     s.to_numpy(),
                COL_FECHA: dates_series.to_numpy(),
                COL_EMP:   pd.Series(emp_series, index=y_series.index).astype(str).to_numpy(),
                "Model":   str(m),
            })
            df[COL_FECHA] = _to_month_end(df[COL_FECHA])
            oof_parts.append(df)

    elif isinstance(results, dict):
        def _first_nonnull(d: dict, keys):
            for k in keys:
                if k in d and d[k] is not None:
                    return d[k]
            return None

        for m, obj in results.items():
            if not isinstance(obj, dict):
                continue

            s = _first_nonnull(obj, ["oof", "oof_predictions", "preds"])
            if s is None:
                continue

            # Soporta varios formatos: array/Serie o dict con 'y_pred'
            if isinstance(s, dict) and "y_pred" in s:
                s = s["y_pred"]

            s = pd.Series(s)  # no uses truthiness; convierte sin evaluar
            s = pd.to_numeric(s, errors="coerce").reindex(y_series.index)

            df = pd.DataFrame({
                COL_Y:     pd.to_numeric(y_series, errors="coerce").to_numpy(),
                COL_P:     s.to_numpy(),
                COL_FECHA: dates_series.to_numpy(),
                COL_EMP:   pd.Series(emp_series, index=y_series.index).astype(str).to_numpy(),
                "Model":   str(m),
            })
            df[COL_FECHA] = _to_month_end(df[COL_FECHA])
            oof_parts.append(df)


    if not oof_parts:
        raise RuntimeError(
            "No se encontraron OOF en disco ni en memoria. "
            "Soluciones: (a) re-ejecuta el backtest con save_oof=True y OOF_DIR fijo, "
            "o (b) expone 'oof_predictions' en memoria para poder reconstruir."
        )

# --- 3) Agregación y leaderboard
oof_all = pd.concat(oof_parts, ignore_index=True)
oof_all["Mes"] = oof_all[COL_FECHA]

ic_m = (oof_all.groupby(["Model", "Mes"]).apply(_ic_spearman_monthly)
        .rename("IC_mes").reset_index())
sp_m = (oof_all.groupby(["Model", "Mes"]).apply(_decile_spread_monthly)
        .rename("Spread_mes").reset_index())

# Meses esperados (si DATES_SERIES está disponible)
if "DATES_SERIES" in globals():
    meses_esp = _to_month_end(pd.to_datetime(DATES_SERIES, errors="coerce")).dropna().unique()
else:
    meses_esp = ic_m["Mes"].dropna().unique()
n_meses_total = len(pd.unique(meses_esp))

df_lb_oof = (
    ic_m.merge(sp_m, on=["Model", "Mes"], how="outer")
        .groupby("Model", as_index=False)
        .agg(IC_mensual=("IC_mes","mean"),
             Spread_mensual=("Spread_mes","mean"),
             n_meses=("Mes","nunique"))
)
df_lb_oof["cov_meses_%"] = 100.0 * df_lb_oof["n_meses"] / max(n_meses_total, 1)
df_lb_oof = df_lb_oof.sort_values(["IC_mensual", "Spread_mensual"], ascending=[False, False])

print("\n===== LEADERBOARD OOF =====")
display(df_lb_oof)

# --- 4) Cálculo de cobertura por filas válidas (NaN-aware) por modelo ---
def _coverage_rows_pct(df_one_model, col_pred="y_pred"):
    v = pd.to_numeric(df_one_model[col_pred], errors="coerce")
    return 100.0 * float(v.notna().sum()) / max(len(v), 1)

cov_rows = (
    oof_all.groupby("Model")["y_pred"]
           .apply(lambda s: 100.0 * s.notna().mean())
           .rename("cov_filas_%")
)

# Añádelo al leaderboard ya construido
df_lb_oof = df_lb_oof.merge(cov_rows, left_on="Model", right_index=True, how="left")

# Umbral adicional de filas válidas
COV_FILAS_UMBRAL = 60.0  # ajusta a gusto

# (La selección del ganador del parche 1 ya contempla este campo si existe)

# --- 5) Elección de ganador
winner_model = None
winner_family = None
best_pipeline = None

if not df_lb_oof.empty:
    # Suma la verificación de filas válidas si usas el parche 3 (ver abajo)
    df_valid = df_lb_oof.copy()
    if "cov_filas_%" in df_valid.columns:
        df_valid = df_valid.query("cov_meses_% >= @COV_UMBRAL and cov_filas_% >= @COV_FILAS_UMBRAL")
    else:
        df_valid = df_valid.query("cov_meses_% >= @COV_UMBRAL")

    if not df_valid.empty:
        _winner_str = df_valid.iloc[0]["Model"]
        try:
            winner_family, winner_model, best_pipeline = find_pipeline_from_name(_winner_str)
        except KeyError:
            pass  # caerá al fallback por folds
    else:
        print(f"[AVISO] Cobertura OOF insuficiente → usar fallback (folds).")

if best_pipeline is None:
    # Fallback por folds (si existe); si no, tomar top del leaderboard OOF
    if "best_by_folds" in globals() and isinstance(best_by_folds, dict):
        winner_model  = best_by_folds["Modelo"]
        winner_family = best_by_folds["Familia"]
        best_pipeline = MODELS_ALL[winner_family][winner_model]
    elif not df_lb_oof.empty:
        _winner_str = df_lb_oof.iloc[0]["Model"]
        winner_family, winner_model, best_pipeline = find_pipeline_from_name(_winner_str)
    else:
        raise RuntimeError("No hay forma de determinar el ganador: df_lb_oof vacío y sin best_by_folds.")


print(f"\n🏁 Modelo ganador (regla OOF): {winner_family} / {winner_model}")

# --- 6) Persistencia
try:
    df_lb_oof.to_csv("df_lb_oof.csv", index=False)
    print("Guardado: df_lb_oof.csv")
except Exception as e:
    print(f"[WARN] No se pudo guardar df_lb_oof.csv: {e}")


[INFO] OOF candidatos=0
[INFO] No se hallaron OOF en disco; intentando usar memoria…

===== LEADERBOARD OOF =====


,Model,IC_mensual,Spread_mensual,n_meses,cov_meses_%
7,Lineales :: OLS (Linear Regression),0.121047,0.058512,117,100.0
6,Lineales :: ElasticNet,0.114874,0.042545,117,100.0
8,Redes neuronales :: GKX NN (64-32-16),0.061466,-0.033021,117,100.0
0,Boosting :: CatBoost (Ranker),0.058723,0.037603,117,100.0
4,Boosting :: Random Forest (Ranker-bins),0.047855,0.018338,117,100.0
5,Boosting :: XGBoost,0.018949,0.019083,117,100.0
2,Boosting :: LightGBM (Ranker),0.001238,0.006935,117,100.0
3,Boosting :: Random Forest,-0.016709,-0.015924,117,100.0
1,Boosting :: LightGBM,-0.058664,-0.050406,117,100.0



🏁 Modelo ganador (regla OOF): None / Lineales :: OLS (Linear Regression)
Guardado: df_lb_oof.csv


In [ ]:
# ===== DIAGNÓSTICO OOF: tamaños y NaNs =====
import os, glob, numpy as np, pandas as pd

# Reutiliza la misma OOF_DIR unificada (si no existe, define el default)
OOF_DIR = globals().get("OOF_DIR", "artifacts/oof_preds")

oof_files = sorted(glob.glob(os.path.join(OOF_DIR, "oof_*.npy")))
print(f"[INFO] OOF encontrados: {len(oof_files)} en {OOF_DIR}")

def _base_name(p):
    b = os.path.basename(p)
    return b.replace("oof_", "").replace(".npy","")

rows = []
for fp in oof_files:
    arr = np.load(fp, allow_pickle=True)
    arr = np.asarray(arr, dtype="float64")
    n = arr.size
    nan_ratio = float(np.isnan(arr).sum()) / max(n, 1)
    rows.append({
        "Model": _base_name(fp),
        "len_pred": n,
        "%NaN": round(nan_ratio*100, 3),
        "%valid_filas": round(100.0*(1-nan_ratio), 3),
    })
diag = pd.DataFrame(rows).sort_values("Model")
print(diag if len(diag) else "[WARN] No hay OOF en disco.")



                                      Model   level_1        y_pred
0             Boosting :: CatBoost (Ranker)  len_pred  21060.000000
1             Boosting :: CatBoost (Ranker)      %NaN     98.290598
2                      Boosting :: LightGBM  len_pred  21060.000000
3                      Boosting :: LightGBM      %NaN     98.290598
4             Boosting :: LightGBM (Ranker)  len_pred  21060.000000
5             Boosting :: LightGBM (Ranker)      %NaN     98.290598
6                 Boosting :: Random Forest  len_pred  21060.000000
7                 Boosting :: Random Forest      %NaN     98.290598
9   Boosting :: Random Forest (Ranker-bins)      %NaN     98.290598
8   Boosting :: Random Forest (Ranker-bins)  len_pred  21060.000000
10                      Boosting :: XGBoost  len_pred  21060.000000
11                      Boosting :: XGBoost      %NaN     98.290598
12                   Lineales :: ElasticNet  len_pred  21060.000000
13                   Lineales :: ElasticNet     

In [ ]:
# ==============================================================================
# SECCIÓN 3.x — Tuning “segunda vuelta” sólo para el Top-K del leaderboard
# (usar DESPUÉS de tener df_leaderboard_tuned)
# ==============================================================================

from sklearn.base import clone as _sk_clone
from sklearn.metrics import make_scorer
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# -------------------- Parámetros de esta pasada focalizada --------------------
TOP_K            = 3           # en real podés subirlo
N_ITER_TOPK      = 10          # en real: 30–50
INNER_SPLITS     = 3           # en real: 4–5
RANDOM_SEED_TOPK = GLOBAL_SEED + 202
SEARCH_N_JOBS    = 1

# -------------------- Precondiciones mínimas ----------------------------------
assert 'df_leaderboard_tuned' in globals() and not df_leaderboard_tuned.empty, \
    "No hay leaderboard; corre la competencia primero."

for v in ['X', 'y', 'MODELS_ALL', 'SEARCH_CFG', 'DATES_SERIES', 'EMBARGO_M']:
    assert v in globals(), f"Falta variable global requerida: {v}"

ANCHOR_Q  = globals().get('ANCHOR_Q', 0)   # frontera trimestral
HOLDOUT_M = globals().get('HOLDOUT_M', 12) # meses de holdout
STEP_M    = globals().get('STEP_M', 3)     # paso en meses

# === Serie de meses UNA SOLA VEZ (Period[M]) ===
months_all = pd.to_datetime(DATES_SERIES).dt.to_period('M')

# -------------------- Scorer IC mensual ---------------------------------------
def _ic_monthly(y_true, y_pred):
    df = pd.DataFrame({'Mes': months_all, 'y_true': y_true, 'y_pred': y_pred})
    ic_m = df.groupby('Mes').apply(
        lambda g: spearmanr(g['y_true'], g['y_pred'])[0]
    ).dropna()
    return float(np.nanmean(ic_m)) if len(ic_m) else -1.0

spearman_scorer = make_scorer(_ic_monthly, greater_is_better=True)

# -------------------- Helper: inner-CV con embargo (firma flexible) ----------
def _make_inner_cv_safe(months, n_splits):
    fn = globals().get('make_inner_cv_with_embargo', None)
    if fn is None:
        raise RuntimeError("make_inner_cv_with_embargo no está en globals().")
    try:
        return fn(months, n_splits=n_splits)
    except TypeError:
        return fn(months, n_splits)

# -------------------- Helpers de ranking (CatBoost / LGBM) --------------------
def _has_nontrivial_groups(months_ser: pd.Series) -> bool:
    """True si TODOS los meses tienen al menos 2 filas (lo que CatBoost Ranker exige)."""
    g = months_ser.value_counts()
    return (g >= 2).all()

def _catboost_groups_from_months(months_ser: pd.Series) -> np.ndarray:
    """Mapea meses -> ids 0..k-1 en el MISMO orden que X."""
    codes, _ = months_ser.factorize()
    return codes.astype(np.int32)

def _lgbm_groups_from_months(months_ser: pd.Series) -> np.ndarray:
    """Devuelve group_sizes para LGBMRanker."""
    arr = months_ser.to_numpy()
    out, cur, cnt = [], arr[0], 1
    for v in arr[1:]:
        if v == cur:
            cnt += 1
        else:
            out.append(cnt)
            cur, cnt = v, 1
    out.append(cnt)
    return np.asarray(out, dtype=np.int32)

def _make_rank_labels_by_month(y_ser: pd.Series, m_ser: pd.Series, n_bins: int = 5) -> np.ndarray:
    """Convierte target continuo en labels 0..n_bins-1 dentro de cada mes."""
    y_ser = y_ser.reset_index(drop=True)
    m_ser = m_ser.reset_index(drop=True)
    ranks = np.zeros(len(y_ser), dtype=np.int32)
    for mm, idx in m_ser.groupby(m_ser).groups.items():
        this_y = y_ser.iloc[idx]
        q = min(n_bins, len(this_y))
        if q <= 1:
            ranks[idx] = 0
        else:
            ranks[idx] = pd.qcut(
                this_y, q=q, labels=False, duplicates="drop"
            ).astype(np.int32).to_numpy()
    return ranks


# -------------------- 1) Top-K sin NaN y sin duplicados ----------------------
df_no_nan = df_leaderboard_tuned[df_leaderboard_tuned['IC_mensual'].notna()].copy()
if df_no_nan.empty:
    df_no_nan = df_leaderboard_tuned.copy()

topk_rows = (
    df_no_nan
    .sort_values('IC_mensual', ascending=False)
    .drop_duplicates(subset=['Familia', 'Modelo'])
    .head(TOP_K)[['Familia', 'Modelo']]
)

print("\n🔁 Tuning focalizado del Top-K:")
print(topk_rows.to_string(index=False))

candidates = []   # (name, fam, tuned_est, best_cv)

# -------------------- 2) Tuning modelo a modelo ------------------------------
for _, row in topk_rows.iterrows():
    fam  = row['Familia']
    name = row['Modelo']

    base_pipe  = MODELS_ALL[fam][name]
    param_dist = SEARCH_CFG.get(name, None)

    print(f"\n—> {fam} / {name}")

    # si el modelo es un ranker de librería, lo ajustamos directo usando el helper
    if ("CatBoost (Ranker)" in name) or ("LightGBM (Ranker)" in name):
        print("   Modelo de ranking de librería → ajuste directo (con fallback a regressor si falla).")
        tuned_est = fit_with_rank_support(_sk_clone(base_pipe), X, y, months_all)
        best_cv   = np.nan
        candidates.append((name, fam, tuned_est, best_cv))
        continue

    # si NO hay hyper-grid, lo ajustamos tal cual pero con soporte de ranking
    if not param_dist:
        print("   (Sin hyper-grid definido: se usa el pipeline base tal cual)")
        tuned_est = fit_with_rank_support(_sk_clone(base_pipe), X, y, months_all)
        best_cv   = np.nan
    else:
        inner_cv = _make_inner_cv_safe(months_all, INNER_SPLITS)
        # chequeo embargo con ordinal
        for tr, te in inner_cv:
            mtr_ord = months_all[tr].max().ordinal
            mts_ord = months_all[te].min().ordinal
            assert (mts_ord - mtr_ord) >= int(EMBARGO_M), \
                f"Embargo roto: {months_all[tr].max()} -> {months_all[te].min()}"

        tuner = RandomizedSearchCV(
            estimator=_sk_clone(base_pipe),
            param_distributions=param_dist,
            n_iter=N_ITER_TOPK,
            cv=inner_cv,
            scoring=spearman_scorer,
            n_jobs=SEARCH_N_JOBS,
            random_state=RANDOM_SEED_TOPK,
            verbose=1,
        )
        tuner.fit(X, y)
        tuned_est = tuner.best_estimator_
        best_cv   = float(tuner.best_score_)
        print(f"   >> Mejor score CV (IC): {best_cv:.4f}")
        print(f"   >> Best params: {tuner.best_params_}")

    candidates.append((name, fam, tuned_est, best_cv))

# -------------------- 3) Mini holdout homogéneo ------------------------------
def _quick_holdout_ic(pipe, months_all_local=months_all):
    # convertimos Period[M] a ordinal para operar
    months_ord = months_all_local.astype('period[M]').map(lambda p: p.ordinal)
    uniq_ord   = np.array(sorted(pd.unique(months_ord)))
    # fronteras alineadas a STEP_M
    boundaries_ord = np.array([u for u in uniq_ord if ((u - uniq_ord[0]) % STEP_M) == 0])
    hold_quarters  = max(1, HOLDOUT_M // STEP_M)
    hold_bounds    = boundaries_ord[-hold_quarters:]

    tr_mask = months_ord < (hold_bounds.min() - int(EMBARGO_M))
    te_mask = np.isin(months_ord, hold_bounds)

    mdl = fit_with_rank_support(
        _sk_clone(pipe),
        X.loc[tr_mask], y.loc[tr_mask],
        months_all_local.loc[tr_mask]
    )
    preds = mdl.predict(X.loc[te_mask])
    df = pd.DataFrame({
        'Mes': months_all_local.loc[te_mask].values,
        'y_true': y.loc[te_mask].values,
        'y_pred': np.asarray(preds)
    })
    ic_m = df.groupby('Mes').apply(lambda g: spearmanr(g['y_true'], g['y_pred'])[0]).dropna()
    return float(ic_m.mean()), int(len(ic_m))

print("\n📏 Comparación rápida en holdout (IC medio):")
candidates_oos = []
for name, fam, pipe, cv in candidates:
    ic_oos, n_oos = _quick_holdout_ic(pipe)
    candidates_oos.append((name, fam, pipe, cv, ic_oos, n_oos))
    cv_show = cv if np.isfinite(cv) else float('nan')
    print(f"   {name:28s} | OOS_IC={ic_oos:.4f} (n={n_oos}) | CV_IC={cv_show:.4f}")

# -------------------- 4) Elegimos ganador ------------------------------------
candidates_oos.sort(
    key=lambda t: ((t[4] if np.isfinite(t[4]) else -1e9),
                   (t[3] if np.isfinite(t[3]) else -1e9)),
    reverse=True
)
best_model_name, best_model_family, best_model_pipeline, best_cv_ic, best_ic_oos, best_n_oos = candidates_oos[0]

print(f"\n🏁 Ganador tras tuning Top-{TOP_K}: {best_model_family} / {best_model_name}")
print(f"   OOS_IC={best_ic_oos:.4f} (n={best_n_oos}) | CV_IC={best_cv_ic if np.isfinite(best_cv_ic) else float('nan'):.4f}")

# Desde aquí tu Sección 4.1 puede usar `best_model_pipeline`


In [ ]:
# =============================================================================
# BLOQUE 13.x — GUARDA DE SEGURIDAD ANTES DEL HOLDOUT
# =============================================================================
assert 'best_model_pipeline' in globals() and best_model_pipeline is not None, \
    "No se encontró 'best_model_pipeline'. Revisa el leaderboard del tuning o reduce N_ITER_RS si la búsqueda falló."
# =============================================================================

In [ ]:
# =============================================================================
# BLOQUE 13.y — HELPER PARA SPLITS TRIMESTRALES (Holdout y Tests)
# =============================================================================

def quarter_masks(months_all,
                  holdout_m=HOLDOUT_M, step_m=STEP_M,
                  anchor_q=ANCHOR_Q, embargo_m=EMBARGO_M):
    """
    Devuelve (train_mask, test_mask, hold_bounds) usando fronteras trimestrales.
    - months_all: Serie/array Period['M'] para todo el dataset.
    """
    import numpy as np, pandas as pd
    m = pd.Series(months_all).astype('period[M]').to_numpy()
    uniq = np.array(sorted(pd.unique(m)))

    # Fronteras del trimestre según anchor_q (0/1/2)
    boundaries = np.array([u for u in uniq if ((u.month - 1) % 3) == anchor_q])

    hold_steps = max(1, holdout_m // step_m)
    hold_bounds = boundaries[-hold_steps:]

    # Embargo en MESES reales
    tr_mask = m < (hold_bounds.min() - embargo_m)
    te_mask = np.isin(m, hold_bounds)

    return tr_mask, te_mask, hold_bounds

In [ ]:
# =============================================================================
# BLOQUE 14 — Runner de Holdout (Versión Final con Ensemble Rolling)
# =============================================================================

from scipy.stats import spearmanr
import numpy as np
import pandas as pd
from sklearn.base import clone

# 1) Meses (usamos MONTHS_ALL que ya está alineada)
uniq_months = np.array(sorted(pd.unique(MONTHS_ALL)))

# 2) Conjuntos: train y holdout final usando fronteras trimestrales
train_mask, test_mask, hold_bounds = quarter_masks(MONTHS_ALL)

# 3) Entrenar y predecir en holdout (rolling por mes usando el helper)
final_est_holdout = clone(best_model_pipeline)
te_idx_holdout = np.where(test_mask)[0]

if DO_MONTHLY_ENSEMBLE:
    print("  · Generando predicciones de holdout con ENSEMBLE MENSUAL (rolling por mes)...")
    y_pred_full = np.full(X.shape[0], np.nan, dtype=np.float32)
    months_in_holdout = pd.unique(MONTHS_ALL.iloc[te_idx_holdout])

    for month in months_in_holdout:
        te_idx_for_month = te_idx_holdout[MONTHS_ALL.iloc[te_idx_holdout].values == month]
        if len(te_idx_for_month) == 0:
            continue

        y_pred_for_month = ensemble_predict_on_indices(final_est_holdout, te_idx_for_month, tr_idx=None)
        y_pred_full[te_idx_for_month] = y_pred_for_month

    y_pred = y_pred_full[te_idx_holdout]
else:
    print("  · Generando predicciones de holdout con SNAPSHOT ÚNICO...")
    mdl = fit_with_rank_support(
        final_est_holdout,
        X.loc[train_mask],
        y.loc[train_mask],
        MONTHS_ALL.loc[train_mask]
    )
    y_pred = mdl.predict(X.loc[test_mask])

# 4) Armar `oos` (perfectamente alineado)
oos = pd.DataFrame({
    'Fecha'   : pd.to_datetime(DATES_SERIES.loc[test_mask]).values,
    'Empresa' : groups.loc[test_mask].values,
    'y_true'  : y.loc[test_mask].values,
    'y_pred'  : np.asarray(y_pred)
}).reset_index(drop=True)
oos['Mes'] = oos['Fecha'].dt.to_period('M')

# Traer Sector_GICS y size_bucket desde df_sorted (ya 1:1 con df_model)
aux_align = df_sorted.loc[test_mask].reset_index(drop=True)
for c in ['Sector_GICS', 'size_bucket', 'P_Share']:
    if c in aux_align.columns:
        oos[c] = aux_align[c].values

print("✅ `oos` listo:", oos.shape, "| meses en holdout:", oos['Mes'].nunique())

In [ ]:
# =============================================================================
# BLOQUE 14.b — RESUMEN FINAL (Holdout): IC medio, t-stat y Spread
#   Requiere: oos con ['Fecha','Mes','Empresa','y_true','y_pred']
#             (opcionales): 'Sector_GICS', 'size_bucket'
# =============================================================================

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

# Asegurar 'Mes' como Period[M]
def _ensure_month(oos):
    o = oos.copy()
    if 'Mes' not in o.columns:
        o['Mes'] = pd.to_datetime(o['Fecha']).dt.to_period('M')
    elif not pd.api.types.is_period_dtype(o['Mes']):
        o['Mes'] = pd.to_datetime(o['Mes']).dt.to_period('M')
    return o

def _ic_series(df, score):
    def _ic(g):
        s = g[score]
        if isinstance(s, pd.DataFrame): s = s.iloc[:, 0]
        if len(g) < 3 or s.nunique() < 2: return np.nan
        return spearmanr(g['y_true'], s)[0]
    return df.groupby('Mes', sort=True).apply(_ic).dropna()

def _spread_series(df, score):
    def _spr(g):
        r = g[score].rank(method='first')
        try:
            if len(g) >= 10 and r.nunique() >= 10:
                d = pd.qcut(r, 10, labels=np.arange(1,11), duplicates='drop')
                m = g.assign(dec=d.astype(int)).groupby('dec')['y_true'].mean()
                return float(m.get(10, np.nan) - m.get(1, np.nan))
            elif len(g) >= 5 and r.nunique() >= 5:
                q = pd.qcut(r, 5, labels=np.arange(1,6), duplicates='drop')
                m = g.assign(q=q.astype(int)).groupby('q')['y_true'].mean()
                return float(m.get(5, np.nan) - m.get(1, np.nan))
            else:
                return np.nan
        except ValueError:
            return np.nan
    return df.groupby('Mes', sort=True).apply(_spr).dropna()

def _t_stat(series):
    if isinstance(series, pd.DataFrame):
        series = series.squeeze()  # Flatten to 1D if 2D/empty
    s = pd.Series(series).dropna()
    n = len(s)
    if n <= 1: return np.nan, n
    mean = float(s.mean())
    sd = float(s.std(ddof=1))
    t = mean / (sd / np.sqrt(n)) if sd > 0 else np.nan
    return t, n

def _neutralize(df, by_cols, base='y_pred', out='score_neut'):
    if isinstance(by_cols, str): by_cols = [by_cols]
    need = set(['Mes', base] + by_cols)
    if not need.issubset(df.columns): return None
    out_df = df.loc[:, ~df.columns.duplicated()].copy()
    def _z(g):
        s = g[base].astype(float)
        mu, sd = s.mean(), s.std(ddof=0)
        return (s - mu) / (sd + 1e-9)
    out_df[out] = out_df.groupby(['Mes'] + by_cols, group_keys=False).apply(_z).astype('float32')
    return out_df

# ---- RUN ----
oos = oos.loc[:, ~oos.columns.duplicated()].copy()
oos2 = _ensure_month(oos)

# Añadir columna de Trimestre para reporte
oos2['Q'] = oos2['Mes'].dt.asfreq('Q')

# --- Tabla de Resumen Principal (con IC y Spread sobre target residual) ---
rows = []
def _row(label, df, score_col):
    # --- Métricas Mensuales (ya las tenías) ---
    # Como ahora solo hay meses-frontera, n_ic será el número de trimestres
    ic_ser = _ic_series(df, score_col)
    t_ic, n_ic = _t_stat(ic_ser)

    spr_ser = _spread_series(df, score_col)
    spr_mean = float(spr_ser.mean()) if len(spr_ser) else np.nan
    t_spr, _ = _t_stat(spr_ser)

    # --- Métricas Trimestrales (NUEVO) ---
    # FIXED: Forzar Series 1D explícitamente para evitar 2D empty
    ic_q_values = []
    q_indices = []
    for q_name, g in df.groupby('Q', sort=True):
        if g[score_col].nunique() > 1:
            ic_val = spearmanr(g['y_true'], g[score_col])[0]
        else:
            ic_val = np.nan
        ic_q_values.append(ic_val)
        q_indices.append(q_name)
    ic_q_ser = pd.Series(ic_q_values, index=q_indices).dropna()
    ic_q_mean = float(ic_q_ser.mean()) if len(ic_q_ser) else np.nan
    t_ic_q, n_ic_q = _t_stat(ic_q_ser)

    # Añadimos las nuevas métricas a la fila
    rows.append([
        label,
        ic_q_mean,          # IC medio trimestral
        float(t_ic_q),      # t-stat del IC trimestral
        int(n_ic_q),        # n (número de trimestres)
        spr_mean,
        float(t_spr)
    ])

_row('Original', oos2, 'y_pred')
if 'size_bucket' in oos2.columns:
    tmp = _neutralize(oos2, 'size_bucket')
    if tmp is not None:
        tmp['Q'] = tmp['Mes'].asfreq('Q')  # FIXED: asfreq para Period
        _row('Size-neutral', tmp, 'score_neut')
if 'Sector_GICS' in oos2.columns:
    tmp = _neutralize(oos2, 'Sector_GICS')
    if tmp is not None:
        tmp['Q'] = tmp['Mes'].asfreq('Q')  # FIXED: asfreq para Period
        _row('Sector-neutral', tmp, 'score_neut')
if {'Sector_GICS','size_bucket'}.issubset(oos2.columns):
    tmp = _neutralize(oos2, ['Sector_GICS','size_bucket'])
    if tmp is not None:
        tmp['Q'] = tmp['Mes'].asfreq('Q')  # FIXED: asfreq para Period
        _row('Sector+Size', tmp, 'score_neut')

# Creamos el DataFrame con los nuevos nombres de columna
res = pd.DataFrame(rows, columns=['Score','IC_Q_mean','t_stat_IC_Q','n_quarters','Spread_mean','t_stat_Spread'])
res['Spread_bps'] = (res['Spread_mean'] * 10000).round(1)
safe_display(res.round({'IC_Q_mean':4, 't_stat_IC_Q':2, 'Spread_mean':4, 't_stat_Spread':2}))

# ==============================================================================
# === MÉTRICAS ADICIONALES OOS: FOCO EN COLAS Y P&L LONG/SHORT ===
# ==============================================================================
print("\n" + "="*60)
print("Métricas Adicionales del Holdout (Foco en Estrategia Long/Short)")
print("="*60)

def _t_stat_series(s):
    s = pd.Series(s).dropna()
    if len(s) <= 1: return np.nan, len(s)
    mean = s.mean(); std = s.std(ddof=1)
    if std == 0 or pd.isna(std): return np.nan, len(s)
    return mean / (std / np.sqrt(len(s))), len(s)

def _series_decile_spread(oos_df, score_col='y_pred', y_col='y_true'):
    monthly_spreads = {}
    for month, g in oos_df.groupby('Mes', sort=True):
        if len(g) < 20:
            continue
        r = g[score_col].rank(method='first')
        try:
            bins = pd.qcut(r, 10, labels=False, duplicates='drop') + 1
            g = g.copy()
            g['bin'] = bins
            top = g[g['bin'] == 10][y_col].mean()
            bot = g[g['bin'] == 1][y_col].mean()
            monthly_spreads[month] = float(top - bot)
        except ValueError:
            # Fallback 5-quantiles
            try:
                bins = pd.qcut(r, 5, labels=False, duplicates='drop') + 1
                g = g.copy()
                g['bin'] = bins
                top = g[g['bin'] == 5][y_col].mean()
                bot = g[g['bin'] == 1][y_col].mean()
                monthly_spreads[month] = float(top - bot)
            except ValueError:
                continue
    s = pd.Series(monthly_spreads).sort_index()
    t, n = _t_stat_series(s)
    return s, float(s.mean()) if len(s) else np.nan, t, n

def _ls_pnl(oos_df, k=20, score_col='y_pred', y_col='y_true', cost_bps=10):
    monthly_rets = []
    for month, g in oos_df.groupby('Mes', sort=True):
        if len(g) < 2 * k: continue
        gg = g.sort_values(score_col, ascending=False)
        long_ret = gg.head(k)[y_col].mean()
        short_ret = gg.tail(k)[y_col].mean()
        gross_spread = long_ret - short_ret
        net_spread = gross_spread - 2 * (cost_bps / 10000.0)
        monthly_rets.append(net_spread)
    s = pd.Series(monthly_rets)
    sharpe = (s.mean() / s.std(ddof=1)) * np.sqrt(12 / STEP_M) if s.std(ddof=1) > 0 else np.nan
    return s, float(s.mean()), float(sharpe)

# --- Ejecución y Reporte ---
# NOTA: oos2['y_true'] es el retorno RESIDUAL. Para P&L, usamos el retorno BRUTO (y_raw)
oos2['y_true_raw'] = y_raw.loc[oos2.index]

# 1. Spread de Deciles (Bruto, usando y_raw)
dec_ser, dec_mean, dec_t, dec_n = _series_decile_spread(
    oos2, score_col='y_pred', y_col='y_true_raw'
)
print(f"[OOS Bruto] Spread D10-D1: mean={dec_mean:.4f} | t-stat={dec_t:.2f} | n={dec_n} meses")

# 2. P&L Long/Short Top-K (Bruto, usando y_raw)
K = 20
ls_ser, ls_mean, ls_sharpe = _ls_pnl(
    oos2, k=K, score_col='y_pred', y_col='y_true_raw', cost_bps=10
)
t_ls, n_ls = _t_stat_series(ls_ser)
print(f"[OOS Bruto] L/S K={K}: mean={ls_mean:.4f} | t-stat={t_ls:.2f} | n={n_ls} meses | Sharpe≈{ls_sharpe:.2f}")

# 3. Métricas Neutralizadas (Alfa Puro)
if {'Sector_GICS', 'size_bucket'}.issubset(oos2.columns):
    print("-" * 20)
    oos_neut = _neutralize(oos2, ['Sector_GICS', 'size_bucket'], base='y_pred', out='score_neut')
    if oos_neut is not None:
        oos_neut['Q'] = oos_neut['Mes'].dt.asfreq('Q')

        dec_ser_n, dec_mean_n, dec_t_n, dec_n_n = _series_decile_spread(
            oos_neut, score_col='score_neut', y_col='y_true'
        )
        print(f"[OOS Neutralizado] Spread D10-D1 (Alfa): mean={dec_mean_n:.4f} | t-stat={dec_t_n:.2f} | n={dec_n_n} meses")

        ls_ser_n, ls_mean_n, ls_sharpe_n = _ls_pnl(
            oos_neut, k=K, score_col='score_neut', y_col='y_true', cost_bps=10
        )
        t_ls_n, n_ls_n = _t_stat_series(ls_ser_n)
        print(f"[OOS Neutralizado] P&L L/S (Alfa): mean={ls_mean_n:.4f} | t-stat={t_ls_n:.2f} | n={n_ls_n} meses | Sharpe≈{ls_sharpe_n:.2f}")

In [ ]:
# Sólo si NO viene del Top-3 tuning, tomamos el del leaderboard base
if 'best_model_pipeline' not in locals():
    if 'df_leaderboard_tuned' in locals() and not df_leaderboard_tuned.empty:
        best_model_info   = df_leaderboard_tuned.iloc[0]
        best_model_name   = best_model_info['Modelo']
        best_model_family = best_model_info['Familia']
        best_model_pipeline = MODELS_ALL[best_model_family][best_model_name]
        print(f"🏆 Modelo Ganador (base): {best_model_family} / {best_model_name}")
    else:
        best_model_pipeline = MODELS_ALL['Boosting']['LightGBM (Ranker)']
        print("⚠️ Leaderboard no disponible. Usando fallback: LightGBM (Ranker)")
else:
    print("✅ Manteniendo best_model_pipeline ya tuneado (Top-3).")


In [ ]:
# =========================
# BLOQUE "PERIOD-SAFE" + TESTS DE LEAKAGE (A–D) — v3 (silencioso)
# =========================

import numpy as np
import pandas as pd
import io, contextlib, warnings
from scipy.stats import spearmanr
from sklearn.base import clone
from pandas.api.types import is_period_dtype

# ---- Guardas mínimos (auto-resolución de X/y/fechas/modelo) ----
if 'df_model' in globals():
    if 'X' not in globals():
        X = df_model[[c for c in df_model.columns
                      if c not in ('Empresa','Fecha') and not c.startswith('retorno_futuro')]]
    if 'y' not in globals():
        tgt_guess = [c for c in df_model.columns if c.startswith('retorno_futuro')]
        if not tgt_guess:
            raise RuntimeError("No encuentro la columna target en df_model.")
        y = df_model[tgt_guess[0]]
    if 'DATES_SERIES' not in globals():
        DATES_SERIES = pd.to_datetime(df_model['Fecha'])
else:
    raise RuntimeError("No encuentro df_model en el entorno.")

EMPRESAS_SERIES = df_model['Empresa'] if 'Empresa' in df_model.columns else pd.Series(['X']*len(df_model))

# Modelo base
if 'best_model_pipeline' in globals() and best_model_pipeline is not None:
    est_base = best_model_pipeline
else:
    try:
        est_base = MODELS_ALL['Boosting']['LightGBM (Ranker)']
        print("[Info] Usando fallback: LightGBM (Ranker) de MODELS_ALL")
    except Exception:
        from sklearn.linear_model import LinearRegression
        est_base = LinearRegression()
        print("[Info] Usando fallback: LinearRegression")

# Silenciar modelo si expone el parámetro scikit
try:
    est_base.set_params(model__verbosity=-1, model__verbose=-1)
    print("[Info] Modelo silenciado (verbosity=-1). La salida será limpia.")
except Exception:
    pass

# ------------------------
# Helpers PERIOD-SAFE
# ------------------------
def _to_period_m(dates_like):
    """Convierte a Period[M] conservando orden/longitud."""
    if isinstance(dates_like, pd.Series) and is_period_dtype(dates_like.dtype):
        return dates_like.dt.asfreq('M')
    if isinstance(dates_like, pd.PeriodIndex):
        return dates_like.asfreq('M')
    return pd.to_datetime(pd.Series(dates_like).values).to_period('M')

def ic_monthly_mean(dates_like, y_true, y_pred):
    m = _to_period_m(dates_like)
    df = pd.DataFrame({'m': m, 'y': np.asarray(y_true), 'p': np.asarray(y_pred)})
    df.dropna(subset=['y', 'p'], inplace=True)
    if df.empty:
        return np.nan, pd.Series(dtype=float)
    ic_m = df.groupby('m', sort=True).apply(lambda g: spearmanr(g['y'], g['p'])[0]).dropna()
    return (float(ic_m.mean()) if len(ic_m) else np.nan), ic_m

def _iter_purged_splits(months_period, train_min=None, gap=None, hold=None):
    if train_min is None: train_min = TRAIN_MIN
    if gap is None:       gap       = GAP_STEPS
    if hold is None:      hold      = HOLD

    mp = _to_period_m(months_period)
    uniq = np.array(sorted(pd.unique(mp)))
    for j in range(train_min + gap, len(uniq) - hold + 1):
        m_ts = uniq[j]
        m_te = uniq[j + hold - 1]
        m_cut = m_ts - gap
        tr_idx = np.where(mp <  m_cut)[0]
        te_idx = np.where((mp >= m_ts) & (mp <= m_te))[0]
        if tr_idx.size and te_idx.size:
            yield tr_idx, te_idx

# ------------------------
# OOF purged (silencioso durante fit/predict) — **reemplazo clave**
# ------------------------
def purged_oof_ic(estimator, Xs, ys, months_s,
                  train_min=None, gap=None, hold=None,
                  permute_train_y=False, rng=None):
    if train_min is None: train_min = TRAIN_MIN
    if gap is None:       gap       = GAP_STEPS
    if hold is None:      hold      = HOLD

    mp = _to_period_m(months_s)
    preds = np.full(len(ys), np.nan, dtype=float)
    rng = np.random.RandomState(42) if (permute_train_y and rng is None) else rng
    months_series = months_s if isinstance(months_s, pd.Series) else pd.Series(months_s)

    for tr, te in _iter_purged_splits(mp, train_min, gap, hold):
        est = clone(estimator)
        y_tr = ys.iloc[tr]
        if permute_train_y:
            y_tr = pd.Series(rng.permutation(y_tr.values), index=y_tr.index)

        # silenciar/verbatim y alias LightGBM/XGB (igual que tenías) …
        try:
            est.set_params(model__verbosity=-1, model__verbose=-1)
        except Exception:
            pass
        try:
            params = est.get_params()
            if 'model__min_data_in_leaf' in params:
                est.set_params(model__min_child_samples=params['model__min_data_in_leaf'])
            if 'model__bagging_fraction' in params:
                est.set_params(model__subsample=params['model__bagging_fraction'])
            if 'model__feature_fraction' in params:
                est.set_params(model__colsample_bytree=params['model__feature_fraction'])
        except Exception:
            pass

        with warnings.catch_warnings(), io.StringIO() as _out, io.StringIO() as _err, \
             contextlib.redirect_stdout(_out), contextlib.redirect_stderr(_err):
            warnings.simplefilter("ignore")
            est_f = fit_with_rank_support(est, Xs.iloc[tr], y_tr, months_series.iloc[tr])
            preds[te] = est_f.predict(Xs.iloc[te])

    ic_mean, ic_m = ic_monthly_mean(mp, ys, preds)
    return ic_mean, ic_m, preds


# ------------------------
# Serie de meses (Period[M]) alineada con X/y
# ------------------------
months_all = pd.Series(_to_period_m(DATES_SERIES))

# =========================
# TEST A — Target shift (6m, 12m) → debería ≈ 0
# =========================
def target_shift_ic(k, train_min=None, gap=None, hold=None):
    if train_min is None: train_min = TRAIN_MIN
    if gap is None:       gap       = GAP_STEPS
    if hold is None:      hold      = HOLD

    y_shift = y.groupby(EMPRESAS_SERIES).shift(-k)
    mask = y_shift.notna()
    Xs = X.loc[mask].reset_index(drop=True)
    ys = y_shift.loc[mask].reset_index(drop=True)
    ms = months_all[mask].reset_index(drop=True)
    icm, _, _ = purged_oof_ic(est_base, Xs, ys, ms, train_min=TRAIN_MIN, gap=EMBARGO_M, hold=hold, permute_train_y=False)
    return icm

# =========================
# TEST B — Extra lag en features (0..3) → no debería mejorar
# =========================
def extra_feature_lag_ic(k, train_min=None, gap=None, hold=None):
    if train_min is None: train_min = TRAIN_MIN
    if gap is None:       gap       = GAP_STEPS
    if hold is None:      hold      = HOLD

    if k == 0:
        Xs, ys, ms = X, y, months_all
    else:
        emp_series = (EMPRESAS_SERIES if len(EMPRESAS_SERIES) == len(X) else groups).reset_index(drop=True)
        Xdf = pd.DataFrame(X).copy()
        Xdf["__emp__"] = emp_series.values if hasattr(emp_series, "values") else emp_series
        Xs = (Xdf.groupby("__emp__", group_keys=False).shift(k).drop(columns="__emp__", errors="ignore"))
        mask = ~Xs.isna().any(axis=1)
        Xs = Xs.loc[mask].reset_index(drop=True)
        ys = y.loc[mask].reset_index(drop=True)
        ms = months_all.loc[mask].reset_index(drop=True)

    icm, _, _ = purged_oof_ic(est_base, Xs, ys, ms, train_min=train_min, gap=EMBARGO_M, hold=hold, permute_train_y=False)
    return icm

# =========================
# Tolerancias y helpers de salida
# =========================
TOL_IC_CERO  = 0.02   # |IC| ≤ 0.02 se considera ≈ 0
TOL_LAG_INCR = 0.01   # tolerancia al aumento con lags
TOL_GAP_INCR = 0.01   # tolerancia al aumento con GAP
_EPS         = 1e-12

_is_finite = lambda x: (x is not None) and np.isfinite(x)
_fmt = lambda x: f"{float(x):.4f}" if _is_finite(x) else str(x)

# =========================
# EJECUCIÓN DE TESTS A–D (PASA/NO PASA)
# =========================

# --- TEST A: Target shift ---
print("\n— TEST A: Target shift —")
icsA = {k: target_shift_ic(k, train_min=TRAIN_MIN, gap=EMBARGO_M, hold=HOLD) for k in (6, 12)}
if all(_is_finite(v) and abs(v) <= TOL_IC_CERO for v in icsA.values()):
    print(f"✅ TEST A PASA — |IC|≤{TOL_IC_CERO:.3f} | " +
          ", ".join([f"+{k}m={_fmt(v)}" for k, v in icsA.items()]))
else:
    print("❌ TEST A NO PASA — hay señal con target desplazado (sospecha de leakage)")
    print("   Detalle:", ", ".join([f"+{k}m={_fmt(v)}" for k, v in icsA.items()]))

# --- TEST B (NAIVE): Extra lag en features ---
print("\n— TEST B: Lags extra en features —")
ic0 = extra_feature_lag_ic(0, train_min=TRAIN_MIN, gap=EMBARGO_M, hold=HOLD)
icsB = {k: extra_feature_lag_ic(k, train_min=TRAIN_MIN, gap=EMBARGO_M, hold=HOLD) for k in (1, 2, 3)}

# Regla: agregar más lag NO debería mejorar el IC (salvo ruido ≤ TOL_LAG_INCR)
okB = all(_is_finite(v) and (v <= (ic0 + TOL_LAG_INCR)) for v in icsB.values())

if _is_finite(ic0) and okB:
    det = ", ".join([f"+{k}m={_fmt(v)}" for k, v in icsB.items()])
    print(f"✅ TEST B PASA — base={_fmt(ic0)} | {det} (TOL_LAG_INCR={TOL_LAG_INCR:.2f})")
else:
    det = ", ".join([f"+{k}m={_fmt(v)}" for k, v in icsB.items()])
    print(f"❌ TEST B NO PASA — base={_fmt(ic0)} | {det} (mejora anómala con lag extra)")

# --- TEST C: Permutación de y (solo en TRAIN) ---
print("\n— TEST C: Permutación sólo en TRAIN —")
ic_perm, _, _ = purged_oof_ic(
    est_base, X, y, months_all,
    train_min=TRAIN_MIN,
    gap=EMBARGO_M,
    hold=HOLD,
    permute_train_y=True,
    rng=np.random.RandomState(123)
)
if _is_finite(ic_perm) and abs(ic_perm) <= TOL_IC_CERO:
    print(f"✅ TEST C PASA — IC_perm={_fmt(ic_perm)} (|IC|≤{TOL_IC_CERO:.2f} esperado≈0)")
else:
    print(f"❌ TEST C NO PASA — IC_perm={_fmt(ic_perm)} (> {TOL_IC_CERO:.2f})")

# --- TEST D: Sensibilidad al GAP (embargo temporal) ---
print("\n— TEST D: Sensibilidad al GAP —")
# Compara IC con menos/más embargo. Al aumentar GAP no debería subir el IC (salvo ruido).
gap_candidates = sorted(set([max(0, GAP_STEPS-1), GAP_STEPS, GAP_STEPS+1]))
ic_by_gap = {}
for g in gap_candidates:
    ic_g, _, _ = purged_oof_ic(est_base, X, y, months_all,
                               train_min=TRAIN_MIN, gap=g, hold=HOLD,
                               permute_train_y=False)
    ic_by_gap[g] = ic_g

# Reglas: IC(g+1) ≤ IC(g) + TOL_GAP_INCR
gaps_sorted = sorted(ic_by_gap.keys())
okD = True
for i in range(len(gaps_sorted)-1):
    g, gp1 = gaps_sorted[i], gaps_sorted[i+1]
    v, vp1 = ic_by_gap[g], ic_by_gap[gp1]
    if _is_finite(v) and _is_finite(vp1):
        if vp1 > (v + TOL_GAP_INCR + _EPS):
            okD = False

det = " | ".join([f"GAP={g}: IC={_fmt(v)}" for g, v in ic_by_gap.items()])
if okD:
    print(f"✅ TEST D PASA — {det} (TOL_GAP_INCR={TOL_GAP_INCR:.2f})")
else:
    print(f"❌ TEST D NO PASA — {det} (IC mejora al aumentar embargo: sospecha de leakage)")

print("\n— FIN TESTS A–D —")

In [ ]:
# ====================================================================
# TEST B (FAIR SAMPLE) — Extra lag en features — SILENCIOSO
#   · Requisitos ya definidos en tu notebook: X, y, months_all (Period[M]),
#     EMPRESAS_SERIES, est_base, purged_oof_ic, fit_with_rank_support
# ====================================================================

import io, os, warnings, contextlib
import numpy as np
import pandas as pd

# --- Context manager para silenciar stdout/stderr y warnings ---
try:
    no_log
except NameError:
    @contextlib.contextmanager
    def no_log():
        old_lvl = os.environ.get("TF_CPP_MIN_LOG_LEVEL")
        os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
        with warnings.catch_warnings(), io.StringIO() as _out, io.StringIO() as _err, \
             contextlib.redirect_stdout(_out), contextlib.redirect_stderr(_err):
            warnings.simplefilter("ignore")
            yield
        if old_lvl is None: os.environ.pop("TF_CPP_MIN_LOG_LEVEL", None)
        else:               os.environ["TF_CPP_MIN_LOG_LEVEL"] = old_lvl

# --- Bajar verbosidad del modelo (si es Pipeline con paso 'model') ---
try:
    est_base.set_params(model__verbose=-1)
except Exception:
    pass
try:
    est_base.set_params(model__verbosity=0)
except Exception:
    pass

# --- Tolerancia y helpers de formato (si no existen) ---
try:
    TOL_LAG_INCR
except NameError:
    TOL_LAG_INCR = 0.01  # max mejora aceptable de IC al agregar lags

_is_finite = (lambda x: (x is not None) and np.isfinite(x)) if '_is_finite' not in globals() else _is_finite
_fmt       = (lambda x: f"{float(x):.4f}" if _is_finite(x) else str(x))     if '_fmt'       not in globals() else _fmt

# --- Función FAIR SAMPLE (muestra común para todos los k) ---
def extra_feature_lag_ic_fair(ks=(0,1,2,3), train_min=12, gap=1, hold=HOLD):
    emp = EMPRESAS_SERIES.loc[X.index]
    Xk, Mk = {}, {}
    for k in ks:
        Xlag = X.copy()
        if k > 0:
            for c in Xlag.columns:
                Xlag[c] = Xlag.groupby(emp)[c].shift(k)
        mask = ~Xlag.isna().any(axis=1) & y.notna()
        Xk[k] = Xlag
        Mk[k] = mask

    if len(ks) == 0:
        return {}
    common_mask = np.logical_and.reduce([Mk[k].to_numpy() for k in ks])
    if common_mask.sum() == 0:
        return {k: np.nan for k in ks}

    Xc = {k: Xk[k].loc[common_mask].reset_index(drop=True) for k in ks}
    yc = y.loc[common_mask].reset_index(drop=True)
    mc = months_all[common_mask].reset_index(drop=True)

    out = {}
    for k in ks:
        icm, _, _ = purged_oof_ic(
            est_base, Xc[k], yc, mc,
            train_min=train_min, gap=gap, hold=hold,
            permute_train_y=False
        )
        out[k] = icm
    return out

# --- Ejecución SILENCIOSA ---
with no_log():
    fair_lag_results = extra_feature_lag_ic_fair(ks=(0,1,2,3), train_min=12, gap=1, hold=HOLD)


# --- ÚNICA salida: veredicto y números de IC por lag ---
vals_ok = all(_is_finite(fair_lag_results.get(k)) for k in (0,1,2,3))
if not vals_ok:
    verdict_fair = False
    nums = ", ".join([f"L{k}={_fmt(fair_lag_results.get(k))}" for k in (0,1,2,3)])
    print(f"❌ TEST B (FAIR) NO PASA — valores no finitos | {nums}")
else:
    ic0    = fair_lag_results[0]
    max_k  = max(fair_lag_results[k] for k in (1,2,3))
    margin = max_k - ic0
    nums   = ", ".join([f"L{k}={_fmt(fair_lag_results[k])}" for k, v in fair_lag_results.items()])
    if margin <= TOL_LAG_INCR + 1e-12:
        verdict_fair = True
        print(f"✅ TEST B (FAIR) PASA — max(IC_lag≥1)−IC_0={_fmt(margin)} ≤ {TOL_LAG_INCR:.3f} | {nums}")
    else:
        verdict_fair = False
        print(f"❌ TEST B (FAIR) NO PASA — max(IC_lag≥1)−IC_0={_fmt(margin)} > {TOL_LAG_INCR:.3f} | {nums}")


# — Comparador NAIVE vs FAIR —
try:
    if 'verdict_naive' in globals() and verdict_naive != verdict_fair:
        print("⚠️  DISCREPANCIA B: NAIVE vs FAIR — posible efecto por cambio de muestra. Toma el FAIR como veredicto principal.")
except Exception:
    pass


In [ ]:
# ====================================================================
# TEST MACRO-LAG1 — PASA / NO PASA (auditoría robusta de *_lag1)
# ====================================================================
import numpy as np
import pandas as pd

# Umbrales (puedes ajustar)
try:
    T_MATCH_TM1_MIN
except NameError:
    T_MATCH_TM1_MIN = 0.95   # proporción mínima que debe igualar a t-1
try:
    T_MATCH_T_MAX
except NameError:
    T_MATCH_T_MAX = 0.80     # proporción máxima que puede igualar a t (para evitar fuga)

# Helper robusto (si no existe)
if '_frac_equal_robust' not in globals():
    def _frac_equal_robust(a: pd.Series, b: pd.Series, mask: pd.Series) -> float:
        if not mask.any():
            return np.nan
        a1, b1 = a[mask], b[mask]
        a_num = pd.to_numeric(a1, errors='coerce').to_numpy()
        b_num = pd.to_numeric(b1, errors='coerce').to_numpy()
        mnum  = np.isfinite(a_num) & np.isfinite(b_num)
        if mnum.any():
            return float(np.isclose(a_num[mnum], b_num[mnum], rtol=1e-6, atol=1e-8).mean())
        return float((a1.astype('string') == b1.astype('string')).mean())

def audit_macro_lag1_passfail(macro_cols):
    """
    Verifica que cada columna {base}_lag1 en X coincida (≈1) con base en t-1
    y NO coincida en exceso (≤ T_MATCH_T_MAX) con base en t.
    Imprime solo PASA/NO PASA y, si falla, cuáles y por qué.
    """
    fmt = lambda x: "nan" if (x is None or not np.isfinite(x)) else f"{x:.3f}"
    if not isinstance(macro_cols, (list, tuple)) or len(macro_cols) == 0:
        print("⚠️ TEST MACRO-LAG1: no hay columnas macro para auditar.")
        return pd.DataFrame(columns=['lag_col','base','match_t-1','match_t','veredicto'])

    # Índice maestro en el orden de df_model
    mi = df_model.set_index(['Empresa','Fecha']).index

    rows = []
    for base in macro_cols:
        lagcol = f"{base}_lag1"
        if lagcol not in X.columns or base not in df_sorted.columns:
            continue

        # Series alineadas 1:1 con df_model
        df_idx = df_sorted.set_index(['Empresa','Fecha'])
        base_t   = df_idx[base].reindex(mi).reset_index(drop=True)
        base_tm1 = df_idx.groupby(level=0)[base].shift(1).reindex(mi).reset_index(drop=True)
        lag_ser  = X[lagcol].reset_index(drop=True)

        # Máscaras de comparación válidas
        m_t   = lag_ser.notna() & base_t.notna()
        m_tm1 = lag_ser.notna() & base_tm1.notna()

        # Métricas de coincidencia
        f_tm1 = _frac_equal_robust(lag_ser, base_tm1, m_tm1)  # esperamos ≈1
        f_t   = _frac_equal_robust(lag_ser, base_t,   m_t)    # esperamos ≈0

        ok_tm1 = (np.isnan(f_tm1) or f_tm1 >= T_MATCH_TM1_MIN)
        ok_t   = (np.isnan(f_t)   or f_t   <= T_MATCH_T_MAX)
        verdict = "OK" if (ok_tm1 and ok_t) else "Sospechosa"
        rows.append([lagcol, base, f_tm1, f_t, verdict])

    out = pd.DataFrame(rows, columns=['lag_col','base','match_t-1','match_t','veredicto'])

    if out.empty:
        print("⚠️ TEST MACRO-LAG1: no se encontraron pares base/_lag1 presentes en los datos.")
        return out

    bad = out[out['veredicto'] != 'OK'].copy()
    if bad.empty:
        print(f"✅ TEST MACRO-LAG1 PASA — {len(out)}/{len(out)} columnas OK "
              f"(umbral t-1 ≥ {T_MATCH_TM1_MIN:.2f}, umbral t ≤ {T_MATCH_T_MAX:.2f}).")
    else:
        print(f"❌ TEST MACRO-LAG1 NO PASA — {len(bad)}/{len(out)} columnas sospechosas "
              f"(t-1 ≥ {T_MATCH_TM1_MIN:.2f} y t ≤ {T_MATCH_T_MAX:.2f}).")
        # Detalle breve de las que fallan
        for _, r in bad.iterrows():
            print(f"   · {r['lag_col']}: match_t-1={fmt(r['match_t-1'])}, match_t={fmt(r['match_t'])}")

    return out

# Correr
_ = audit_macro_lag1_passfail(features_macro)


In [ ]:
# ================== BLOQUE · Fama–French Attribution (con PASA/NO PASA) =========================
# Requisitos: que exista `oos` con columnas ['Fecha','Mes','Empresa','y_true','y_pred']
# y conexión a internet para pandas_datareader (o reemplazar por df local de factores).

# Umbrales del test (puedes sobreescribirlos antes de correr este bloque)
ALPHA_T_MIN    = globals().get('ALPHA_T_MIN', 1.96)   # t-α mínimo (≈ 5% bilateral)
ALPHA_ANN_MIN  = globals().get('ALPHA_ANN_MIN', 0.00) # α anualizado debe ser > 0
N_MIN_MONTHS   = globals().get('N_MIN_MONTHS', 24)    # meses mínimos
USE_HAC        = globals().get('USE_HAC', True)       # usar HAC/Newey–West

if 'DO_ATTRIB' in globals() and DO_ATTRIB:
    import numpy as np, pandas as pd
    import statsmodels.api as sm
    import pandas_datareader.data as web
    from pandas.api.types import is_period_dtype

    if 'oos' not in globals() or oos is None or oos.empty:
        print("❌ FF Attribution — NO PASA (no hay OOS para construir el spread)")
    else:
        # 1) Q5–Q1 mensual a partir de OOS (deciles si se puede; si no, quintiles)
        def _q_spread_monthly(oos_df, n_bins_try=(10, 5)):
            df = oos_df.copy()
            # Normalizar 'Mes' a Period[M] sin romper si ya lo es
            if 'Mes' in df.columns:
                if is_period_dtype(df['Mes']):
                    df['Mes'] = df['Mes'].astype('period[M]')
                else:
                    df['Mes'] = pd.to_datetime(df['Mes']).dt.to_period('M')
            elif 'Fecha' in df.columns:
                df['Mes'] = pd.to_datetime(df['Fecha']).dt.to_period('M')
            else:
                raise ValueError("OOS debe tener 'Mes' o 'Fecha'.")

            out = []
            for m, g in df.groupby('Mes', sort=True):
                r = g['y_pred'].rank(method='first')
                spread = np.nan
                for q in n_bins_try:
                    try:
                        bins = pd.qcut(r, q, labels=np.arange(1, q+1))
                        mret = g.assign(bin=bins.astype(int)).groupby('bin')['y_true'].mean()
                        spread = float(mret.iloc[-1] - mret.iloc[0])
                        break
                    except ValueError:
                        continue
                out.append((m.to_timestamp('M'), spread))

            s = pd.Series({d: v for d, v in out}, name='spread').dropna()
            return s

        spread_m = _q_spread_monthly(oos)  # en decimales
        if spread_m.empty:
            print("❌ FF Attribution — NO PASA (spread OOS vacío)")
        else:
            # 2) Descargar factores (FF3/FF5/MOM). Columnas en decimales y MonthEnd.
            def _load_ff():
                factors = {}
                try:
                    ff3 = web.DataReader('F-F_Research_Data_Factors', 'famafrench')[0]
                    ff3.index = ff3.index.to_timestamp('M')
                    ff3 = ff3 / 100.0
                    ff3 = ff3.rename(columns={'Mkt-RF': 'MKT_RF'})
                    factors['ff3'] = ff3[['MKT_RF', 'SMB', 'HML', 'RF']]
                except Exception:
                    pass

                try:
                    ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench')[0]
                    ff5.index = ff5.index.to_timestamp('M')
                    ff5 = ff5 / 100.0
                    ff5 = ff5.rename(columns={'Mkt-RF': 'MKT_RF'})
                    factors['ff5'] = ff5[['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']]
                except Exception:
                    pass

                try:
                    mom = web.DataReader('F-F_Momentum_Factor', 'famafrench')[0]
                    mom.index = mom.index.to_timestamp('M')
                    mom = mom / 100.0
                    mom_col = mom.columns[0]
                    mom = mom.rename(columns={mom_col: 'MOM'})
                    factors['mom'] = mom[['MOM']]
                except Exception:
                    pass

                base = None
                if 'ff5' in factors:
                    base = factors['ff5']
                elif 'ff3' in factors:
                    base = factors['ff3']
                if base is None:
                    return None

                if 'mom' in factors:
                    base = base.join(factors['mom'], how='left')
                return base

            df_ff = _load_ff()
            if df_ff is None:
                mu = float(spread_m.mean())
                print(f"Media mensual spread (Q5–Q1): {mu:.6f}  |  anualizado: {(1+mu)**12 - 1:.4f}")
                print("❌ FF Attribution — NO PASA (sin factores no se puede testear α)")
            else:
                # 3) Dataset de regresión (excess return del spread)
                idx = spread_m.index.intersection(df_ff.index)
                if len(idx) < max(12, N_MIN_MONTHS // 2):
                    print(f"❌ FF Attribution — NO PASA (muy poca intersección: {len(idx)} meses)")
                else:
                    y_ex = (spread_m.loc[idx] - df_ff.loc[idx, 'RF']).astype(float)

                    def _run_reg(name, xcols):
                        X = df_ff.loc[idx, xcols].astype(float)
                        X = sm.add_constant(X)
                        if USE_HAC:
                            res = sm.OLS(y_ex.values, X.values, missing='drop').fit(
                                cov_type='HAC', cov_kwds={'maxlags': 6}
                            )
                        else:
                            res = sm.OLS(y_ex.values, X.values, missing='drop').fit()
                        alpha_m = float(res.params[0])
                        alpha_t = float(res.tvalues[0])
                        r2 = float(res.rsquared)
                        return dict(Model=name,
                                    Alpha_m=alpha_m,
                                    Alpha_ann=alpha_m * 12.0,
                                    t_alpha=alpha_t,
                                    R2=r2,
                                    n=int(len(y_ex)))

                    results = []
                    if 'MKT_RF' in df_ff.columns:
                        results.append(_run_reg('CAPM', ['MKT_RF']))
                    if set(['MKT_RF', 'SMB', 'HML']).issubset(df_ff.columns):
                        results.append(_run_reg('FF3', ['MKT_RF', 'SMB', 'HML']))
                    if set(['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA']).issubset(df_ff.columns):
                        results.append(_run_reg('FF5', ['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA']))
                    if set(['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']).issubset(df_ff.columns):
                        results.append(_run_reg('FF5+MOM', ['MKT_RF', 'SMB', 'HML', 'RMW', 'CMA', 'MOM']))

                    if not results:
                        print("❌ FF Attribution — NO PASA (no hay especificaciones disponibles)")
                    else:
                        results_ff = (pd.DataFrame(results)
                                        .sort_values('Alpha_ann', ascending=False)
                                        .reset_index(drop=True))
                        # 4) Veredicto PASA/NO PASA usando el modelo más rico disponible
                        order_pref = ['FF5+MOM', 'FF5', 'FF3', 'CAPM']
                        chosen = None
                        for mname in order_pref:
                            hit = results_ff[results_ff['Model'] == mname]
                            if len(hit):
                                chosen = hit.iloc[0].to_dict()
                                break
                        if chosen is None:
                            chosen = results_ff.iloc[0].to_dict()

                        n_ok   = chosen['n'] >= N_MIN_MONTHS
                        t_ok   = chosen['t_alpha'] >= ALPHA_T_MIN
                        alp_ok = chosen['Alpha_ann'] > ALPHA_ANN_MIN
                        passed = bool(n_ok and t_ok and alp_ok)

                        print(
                            f"Test FF ({chosen['Model']}) → "
                            + ("✅ PASA" if passed else "❌ NO PASA")
                            + f" | n={int(chosen['n'])} (≥{N_MIN_MONTHS}), "
                              f"α_m={chosen['Alpha_m']:.6f}, α_ann={chosen['Alpha_ann']:.4%} (> {ALPHA_ANN_MIN:.2%}), "
                              f"t-α={chosen['t_alpha']:.2f} (≥ {ALPHA_T_MIN:.2f})"
                        )
else:
    print("ℹ️  DO_ATTRIB=False — omito atribución Fama–French (ponlo en True para correr).")
# ================================================================================================


In [ ]:
# ================= PASA / NO PASA – Overlays y umbrales =================
# (Pega esto al final del bloque, o justo después de cada sección relevante)

# ---- Umbrales (puedes ajustarlos arriba si querés) ----
TH_SECTOR_UNKNOWN_MAX      = globals().get('TH_SECTOR_UNKNOWN_MAX', 0.20)   # ≤20% filas Unknown
TH_RF_MIN_NONNAN_MONTHS    = globals().get('TH_RF_MIN_NONNAN_MONTHS', 12)   # ≥12 meses con RF válido
TH_ANCLAJE_TOL             = globals().get('TH_ANCLAJE_TOL', 1e-6)          # tolerancia numérica
TH_BASELINE_MIN_MONTHS     = globals().get('TH_BASELINE_MIN_MONTHS', 12)    # meses min para baseline
TH_BASELINE_MOM12_MIN      = globals().get('TH_BASELINE_MOM12_MIN', 0.00)   # IC mom12 ≥ 0
TH_BASELINE_SEC_MIN        = globals().get('TH_BASELINE_SEC_MIN', 0.00)     # IC sector ≥ 0
TH_HOLDOUT_T_MIN           = globals().get('TH_HOLDOUT_T_MIN', 1.96)        # t(IC) holdout ≥ 1.96
TH_HOLDOUT_SPREAD_T_MIN    = globals().get('TH_HOLDOUT_SPREAD_T_MIN', 1.65) # t(spread) ≥ 1.65

def _pf(ok, msg_ok, msg_fail):
    print(("✅ " + msg_ok) if ok else ("❌ " + msg_fail))

def _t_stat(s):
    try:
        s = s.dropna()
        n = int(s.shape[0])
        if n <= 1: return float('nan')
        std = float(s.std(ddof=1))
        if std <= 0: return float('nan')
        return float(s.mean() / (std / np.sqrt(n)))
    except Exception:
        return float('nan')

print("\n================= RESUMEN PASA/NO PASA =================")

# ---- 1) Cobertura del mapeo Industry → GICS ----
if 'unknown_rows_pct' in globals():
    _pf(unknown_rows_pct <= TH_SECTOR_UNKNOWN_MAX,
        f"GICS mapping PASA (Unknown={unknown_rows_pct:.1%} ≤ {TH_SECTOR_UNKNOWN_MAX:.0%})",
        f"GICS mapping NO PASA (Unknown={unknown_rows_pct:.1%} > {TH_SECTOR_UNKNOWN_MAX:.0%})")

# ---- 2) Risk-free (FRED) descargado y usable ----
if 'df_risk_free' in globals():
    ok_rf = (df_risk_free is not None) and df_risk_free['risk_free_monthly_return'].notna().sum() >= TH_RF_MIN_NONNAN_MONTHS
    _pf(ok_rf,
        f"Risk-free (FRED) PASA (meses válidos ≥ {TH_RF_MIN_NONNAN_MONTHS})",
        "Risk-free (FRED) NO PASA (sin datos o muy pocos meses válidos)")

# ---- 3) Validaciones de anclaje y target ----
# 3.1 Chequeo 1: constancia intra-trimestre + cambios sólo en límites de Q
if 'viol_intra_all' in globals() and 'viol_boundary_all' in globals():
    ok_chk1 = (not viol_intra_all) and (not viol_boundary_all)
    _pf(ok_chk1, "Chequeo 1 PASA (anclaje Q-2 consistente)",
               "Chequeo 1 NO PASA (variaciones intra-trimestre o cambios fuera de límite)")

# 3.2 Chequeo 3: contables sólo cambian cuando toca
if 'violaciones' in globals():
    ok_chk3 = (not violaciones)
    _pf(ok_chk3, "Chequeo 3 PASA (sin look-ahead en contables)",
               "Chequeo 3 NO PASA (cambios prematuros en contables)")

# 3.3 Chequeo 4: reconstrucción exacta del target
if 'err' in globals():
    try:
        ok_chk4 = (err.dropna().max() < TH_ANCLAJE_TOL)
    except Exception:
        ok_chk4 = False
    _pf(ok_chk4, f"Chequeo 4 PASA (reconstrucción target < {TH_ANCLAJE_TOL:g})",
               "Chequeo 4 NO PASA (discrepancias en reconstrucción del target)")

# ---- 4) Baselines de señal (IC mensual) ----
# 4.1 Momentum 12m (lag1)
if 'ic_mom' in globals():
    icm = float(ic_mom.mean()) if len(ic_mom) else float('nan')
    ok_mom = (len(ic_mom) >= TH_BASELINE_MIN_MONTHS) and (icm >= TH_BASELINE_MOM12_MIN)
    _pf(ok_mom,
        f"Baseline mom12 PASA (IC={icm:.4f}, n={len(ic_mom)})",
        f"Baseline mom12 NO PASA (IC={icm:.4f}, n={len(ic_mom)} < {TH_BASELINE_MIN_MONTHS})")

# 4.2 Señal sectorial segura
if 'ic_sec' in globals():
    ics = float(ic_sec.mean()) if len(ic_sec) else float('nan')
    ok_sec = (len(ic_sec) >= TH_BASELINE_MIN_MONTHS) and (ics >= TH_BASELINE_SEC_MIN)
    _pf(ok_sec,
        f"Baseline sectorial PASA (IC={ics:.4f}, n={len(ic_sec)})",
        f"Baseline sectorial NO PASA (IC={ics:.4f}, n={len(ic_sec)} < {TH_BASELINE_MIN_MONTHS})")

# ---- 5) Holdout OOS (si se construyó en este bloque) ----
if 'oos' in globals() and isinstance(oos, pd.DataFrame) and not oos.empty:
    try:
        # En tu bloque se calcularon ic_m y spr_m para oos
        t_ic   = _t_stat(ic_m)  if 'ic_m'  in globals() else float('nan')
        t_spr  = _t_stat(spr_m) if 'spr_m' in globals() else float('nan')
        ok_oos = (np.isfinite(t_ic) and t_ic >= TH_HOLDOUT_T_MIN) and \
                 (np.isfinite(t_spr) and t_spr >= TH_HOLDOUT_SPREAD_T_MIN)
        _pf(ok_oos,
            f"Holdout OOS PASA (t_IC={t_ic:.2f} ≥ {TH_HOLDOUT_T_MIN:.2f} y t_spread={t_spr:.2f} ≥ {TH_HOLDOUT_SPREAD_T_MIN:.2f})",
            f"Holdout OOS NO PASA (t_IC={t_ic:.2f}, t_spread={t_spr:.2f})")
    except Exception as e:
        print(f"⚠️  No pude evaluar PASA/NO PASA de holdout OOS: {e}")

print("================= FIN RESUMEN PASA/NO PASA =================\n")


In [ ]:
# ================================================================================
# BLOQUE: ANÁLISIS DE FEATURE IMPORTANCE DE LOS MEJORES MODELOS
# ================================================================================

print("\n" + "="*60)
print("Análisis de Importancia de Features de los Mejores Modelos")
print("="*60)

# --- 1. Aplanar el diccionario de modelos si aún no existe ---
if 'models_to_compare_flat' not in locals():
    models_to_compare_flat = {}
    for family, model_dict in MODELS_ALL.items():
        for model_name, pipeline in model_dict.items():
            models_to_compare_flat[model_name] = pipeline

# --- 2. Definir la función auxiliar "inteligente" ---
def plot_feature_importances(model, feature_names, title="Importancia de Características", top_n=20):
    """
    Grafica la importancia de features para diferentes tipos de modelos,
    incluyendo wrappers personalizados y modelos lineales.
    """
    final_estimator = model.steps[-1][1] if hasattr(model, 'steps') else model
    estimator_name = final_estimator.__class__.__name__

    importances = None

    try:
        # --- Lógica para modelos de árboles estándar (XGB, LGBM, RF Regressor) ---
        if hasattr(final_estimator, 'feature_importances_'):
            importances = final_estimator.feature_importances_

        # --- Lógica para wrappers personalizados ---
        elif estimator_name == 'RFCAsRanker' and hasattr(final_estimator, '_clf'):
            importances = final_estimator._clf.feature_importances_

        elif estimator_name == 'CatBoostRankerSk' and hasattr(final_estimator, '_mdl'):
            importances = final_estimator._mdl.get_feature_importance(type='PredictionValuesChange')

        # --- Lógica para modelos lineales ---
        elif hasattr(final_estimator, 'coef_'):
            # Para modelos lineales, usamos el valor absoluto de los coeficientes
            importances = np.abs(final_estimator.coef_.flatten())

        else:
            print(f"Advertencia: El estimador {estimator_name} no tiene un método conocido para obtener la importancia de features.")
            return None

        # --- Creación del DataFrame y gráfico ---
        fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
        fi_df = fi_df.sort_values('importance', ascending=False).head(top_n)

        plt.figure(figsize=(10, 8))
        plt.barh(fi_df['feature'], fi_df['importance'])
        plt.xlabel('Importancia')
        plt.ylabel('Característica')
        plt.title(title)
        plt.gca().invert_yaxis()
        plt.tight_layout()
        plt.show()

        return fi_df

    except Exception as e:
        print(f"Error al graficar importancia para {title}: {e}")
        return None

# --- 3. Seleccionar los 3 Mejores Modelos del Leaderboard ---
try:
    # Usamos la variable correcta: df_leaderboard_tuned
    if 'df_leaderboard_tuned' not in locals() or df_leaderboard_tuned.empty:
        raise NameError("'df_leaderboard_tuned' no está definido o está vacío.")

    # Ordenamos por la columna correcta: IC_mensual
    top_3_configs = df_leaderboard_tuned.sort_values(by='IC_mensual', ascending=False).head(3)

    print("\n--- Top 3 Modelos Seleccionados para Análisis (por IC_mensual) ---")
    # Mostramos las columnas correctas
    safe_display(top_3_configs[['Familia', 'Modelo', 'IC_mensual', 'DecileSpread']])

except (NameError, KeyError) as e:
    print(f"Error al seleccionar los mejores modelos: {e}")
    top_3_configs = pd.DataFrame()

# --- 4. Re-entrenar y Graficar Importancia para Cada Modelo ---
if not top_3_configs.empty:
    # Verificar que las variables necesarias existan
    if 'X' not in locals() or 'y' not in locals():
        print("ERROR: Los DataFrames 'X' y/o 'y' no están definidos.")
    elif 'predictor_cols_final' not in locals():
        print("ERROR: La lista 'predictor_cols_final' no está definida.")
    else:
        for index, config_row in top_3_configs.iterrows():
            model_name = config_row['Modelo']

# --- 4. Re-entrenar y Graficar Importancia para Cada Modelo ---
if not top_3_configs.empty:
    if 'X' not in locals() or 'y' not in locals():
        print("ERROR: Los DataFrames 'X' y/o 'y' no están definidos.")
    elif 'predictor_cols_final' not in locals():
        print("ERROR: La lista 'predictor_cols_final' no está definida.")
    else:
        # Preparamos la serie de meses una sola vez
        months_full = pd.to_datetime(DATES_SERIES).dt.to_period('M')

        for index, config_row in top_3_configs.iterrows():
            model_name = config_row['Modelo']
            pipeline_base = models_to_compare_flat.get(model_name)

            if pipeline_base is None:
                print(f"\nNo se encontró el pipeline para '{model_name}'. Saltando.")
                continue

            print("\n" + "-"*50)
            print(f"Re-entrenando y analizando: {model_name}")
            print("-"*50)

            try:
                start_fit_time = time.time()

                # Usamos fit_with_rank_support para entrenar correctamente
                pipeline_fitted = fit_with_rank_support(
                    clone(pipeline_base),
                    X, y,
                    months_full
                )

                end_fit_time = time.time()
                print(f"  Modelo re-entrenado en el dataset completo en {end_fit_time - start_fit_time:.2f} seg.")

                # Graficar importancia de features
                plot_feature_importances(
                    model=pipeline_fitted,
                    feature_names=predictor_cols_final,
                    title=f"Importancia de Features - {model_name}"
                )

            except Exception as e:
                print(f"  ERROR al re-entrenar o graficar para {model_name}: {e}")
else:
    print("\nNo se seleccionaron modelos para analizar (leaderboard vacío).")

In [ ]:
# ================================================================================
# BLOQUE: ANÁLISIS DE INTERPRETABILIDAD CON SHAP (VERSIÓN FINAL ROBUSTA)
# ================================================================================
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.pipeline import Pipeline as _SkPipe
from lightgbm import LGBMRanker
from xgboost import XGBRanker

print("\n" + "="*80)
print("### INICIO: Análisis de Interpretabilidad con SHAP (Versión Robusta) ###")
print("="*80)

# --- Helper robusto para colapsar valores SHAP a 1D ---
def _to_shap_global_1d(shap_values):
    # Lista por clase (clasificación): promedio de |SHAP| por clase y luego sobre clases
    if isinstance(shap_values, list):
        per_class = [np.abs(np.asarray(sv)).mean(axis=0) for sv in shap_values]
        return np.mean(np.vstack(per_class), axis=0)

    # Explanation o array -> tomar el .values si existe
    sv = getattr(shap_values, "values", shap_values)
    sv = np.asarray(sv)

    if sv.ndim == 2:              # (n_samples, n_features)
        return np.abs(sv).mean(axis=0)
    if sv.ndim == 3:              # (n_samples, n_features, n_outputs)
        return np.abs(sv).mean(axis=(0, 2)) # Promedia sobre muestras y outputs
    if sv.ndim == 1:              # (n_features,)
        return np.abs(sv)

    raise ValueError(f"Forma inesperada de SHAP: {sv.shape}")

# --- 1) Preparar modelo y datos transformados ---
try:
    assert 'best_model_pipeline' in globals() and best_model_pipeline is not None, \
        "La variable 'best_model_pipeline' no está definida."

    mdl_name = (best_model_name if 'best_model_name' in globals()
                else type(best_model_pipeline.named_steps['model']).__name__)
    print(f"Analizando el modelo ganador: {mdl_name}")

    print("   Re-entrenando el modelo en TODOS los datos…")
    months_full = pd.to_datetime(DATES_SERIES).dt.to_period('M')
    pipeline_fitted = fit_with_rank_support(clone(best_model_pipeline), X, y, months_full)
    print("   Modelo re-entrenado.")

    preprocessor_fitted = _SkPipe(pipeline_fitted.steps[:-1])
    model_fitted = pipeline_fitted.steps[-1][1]

    print("   Aplicando transformaciones (winsor/zscore/…) para SHAP…")
    X_transformed = preprocessor_fitted.transform(X)
    X_transformed_df = pd.DataFrame(X_transformed, columns=predictor_cols_final)
    print("   Transformación completada.")

except Exception as e:
    print(f"Error preparando el modelo/transformaciones para SHAP: {e}")
    model_fitted = None

# --- 2) Resolver el estimador compatible con TreeExplainer ---
def _resolve_tree_model(m):
    if hasattr(m, "_clf") and m._clf is not None:
        return m._clf, "RFCAsRanker(RandomForestClassifier)"
    name = m.__class__.__name__
    if any(k in name for k in ["LGBM", "XGB", "RandomForest", "DecisionTree"]):
        return m, name
    return None, name

# --- 3) Submuestreo y cálculo SHAP ---
if model_fitted is not None:
    model_for_shap, model_tag = _resolve_tree_model(model_fitted)

    # --- Guard-rail para rankers puros (LGBMRanker/XGBRanker) ---
    if isinstance(model_fitted, (LGBMRanker, XGBRanker)):
        print("⚠️  SHAP no soporta bien objetivos de ranking (LGBMRanker/XGBRanker). "
              "Saltaré interpretabilidad para este modelo.")
        model_for_shap = None

    if model_for_shap is None:
        print(f"No se puede calcular SHAP para este tipo de modelo: {model_tag}")
    else:
        if len(X_transformed_df) > N_SHAP_SAMPLES:
            print(f"   Usando un subconjunto de {N_SHAP_SAMPLES} filas para SHAP…")
            X_shap_subset = shap.sample(X_transformed_df, N_SHAP_SAMPLES, random_state=GLOBAL_SEED)
        else:
            X_shap_subset = X_transformed_df

        print("   Calculando valores SHAP…")
        explainer = shap.TreeExplainer(model_for_shap, feature_perturbation="tree_path_dependent")
        shap_values = explainer.shap_values(X_shap_subset)
        print("   Cálculo SHAP completado.")

        # --- 4) Visualizaciones ---
        shap_global = _to_shap_global_1d(shap_values)

        # Chequeo defensivo de dimensiones
        if shap_global.shape[0] != X_shap_subset.shape[1]:
            raise ValueError(
                f"Desacople SHAP/features: {shap_global.shape[0]} vs {X_shap_subset.shape[1]}"
            )

        fi_df = (pd.DataFrame({
                    "feature": X_shap_subset.columns,
                    "shap_importance": shap_global.astype(float)
                })
                .sort_values("shap_importance", ascending=False)
                .reset_index(drop=True))

        print("\n--- Top 15 features por importancia SHAP global ---")
        print(fi_df.head(15).to_string(index=False))

        # (B) Bar plot
        plt.figure()
        plt.bar(fi_df["feature"].values[:20], fi_df["shap_importance"].values[:20])
        plt.xticks(rotation=90)
        plt.title(f"Importancia media |SHAP| (Top 20) – {mdl_name}")
        plt.tight_layout()
        plt.show()

        # (C) Summary plot
        if isinstance(shap_values, list): # Multiclase
            classes = getattr(model_for_shap, "classes_", list(range(len(shap_values))))
            idx_top = int(np.argmax(classes))
            print(f"\n--- Summary plot (clase/top bin = {classes[idx_top]}) ---")
            shap.summary_plot(shap_values[idx_top], X_shap_subset, show=False)
            plt.title(f"SHAP summary – clase {classes[idx_top]} – {mdl_name}")
            plt.tight_layout()
            plt.show()
        else: # Regresión/binaria
            print("\n--- Summary plot (regresión/binaria) ---")
            shap.summary_plot(shap_values, X_shap_subset, show=False)
            plt.title(f"SHAP summary – {mdl_name}")
            plt.tight_layout()
            plt.show()
else:
    print("No se pudo calcular SHAP. El modelo final no es compatible o no se entrenó correctamente.")

In [ ]:
# ================================================================================
# BLOQUE 15 (CORREGIDO): CONSTRUCCIÓN DE CARTERA FINAL DESDE EL HOLDOUT
# ================================================================================
import pandas as pd
import numpy as np
from pandas.api.types import is_period_dtype

print("\n" + "="*80)
print("🚀 Construcción de la Cartera Final para el Trimestre Más Reciente")
print("="*80)

def build_final_portfolio(
    oos_df: pd.DataFrame,
    ticker_col: str = "Empresa",
    quarter_col: str = "Q",
    rank_col: str = "rank_q",
    signal_col: str = "y_pred",
    top_n: int = 50
):
    if oos_df is None or oos_df.empty:
        print("⚠️ El DataFrame 'oos' está vacío. No se puede construir la cartera.")
        return pd.DataFrame(), None

    df = oos_df.copy()

    # 1) asegurar Mes
    if 'Mes' not in df.columns:
        df['Mes'] = pd.to_datetime(df['Fecha']).dt.to_period('M')
    elif not is_period_dtype(df['Mes']):
        df['Mes'] = pd.to_datetime(df['Mes']).dt.to_period('M')

    # 2) trimestre period-safe
    df[quarter_col] = df['Mes'].dt.asfreq('Q')

    # 3) rankeo mensual
    df['rank_monthly'] = df.groupby('Mes')[signal_col].rank(ascending=False, method='first')

    # 4) último trimestre
    latest_quarter = df[quarter_col].max()
    if pd.isna(latest_quarter):
        print("⚠️ No se pudo determinar el trimestre más reciente.")
        return pd.DataFrame(), None

    print(f"  -> Seleccionando activos para el trimestre: {latest_quarter}")

    # 5) filtrar trimestre
    df_latest_q = df[df[quarter_col] == latest_quarter].copy()
    if df_latest_q.empty:
        print("⚠️ No hay datos en el trimestre más reciente.")
        return pd.DataFrame(), latest_quarter

    # 6) promedio por empresa
    df_final_signal = (
        df_latest_q
        .groupby(ticker_col)['rank_monthly']
        .mean()
        .reset_index()
        .rename(columns={'rank_monthly': 'avg_monthly_rank'})
    )

    # 7) rank final
    df_final_signal[rank_col] = df_final_signal['avg_monthly_rank'].rank(method='first')

    portfolio = (
        df_final_signal
        .sort_values(rank_col, ascending=True)
        .head(top_n)
        .reset_index(drop=True)
    )
    return portfolio, latest_quarter


# --- EJECUCIÓN ---
if 'oos' in globals() and 'best_model_name' in globals():
    print(f"Usando las predicciones del holdout del modelo ganador: '{best_model_name}'")

    cartera_final, last_q = build_final_portfolio(oos_df=oos, top_n=50)

    if cartera_final is not None and not cartera_final.empty:
        print(f"\n--- ✅ Cartera Top 50 para el trimestre {last_q} ---")
        safe_display(cartera_final)
    else:
        print("\nNo se pudo generar la cartera final.")
else:
    print("\n⚠️ Faltan 'oos' o 'best_model_name'. No se puede construir la cartera final.")
